In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import zipfile
import sys
import torch

# Connect Google Drive
drive.mount("/content/drive")

ROOT = Path("/content/UA_FloodNet")
CODE = ROOT / "ablations_code"
DATA = ROOT / "dataset"

BACKUP = Path(
    "/content/drive/MyDrive/UA_FloodNet"
)

CODE.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

print("RESTORING UA-FLOODNET RESEARCH ENVIRONMENT")
print("=" * 65)

# 1. Restore Python source files
for filename in [
    "ablation_model.py",
    "data.py",
    "train_v4.py"
]:
    source = BACKUP / "v4/code" / filename
    assert source.exists(), f"Missing backup: {source}"

    shutil.copy2(source, CODE / filename)
    print("Restored:", filename)

# 2. Restore NPZ dataset
archive = (
    BACKUP /
    "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
)

assert archive.exists(), "Dataset backup missing."

with zipfile.ZipFile(archive) as z:
    root = DATA.resolve()

    for item in z.infolist():
        target = (DATA / item.filename).resolve()
        if not target.is_relative_to(root):
            raise ValueError(
                f"Unsafe ZIP entry: {item.filename}"
            )

    z.extractall(DATA)

# 3. Verify dataset
print("\nDATASET VERIFICATION")

for split, expected in [
    ("train", 499),
    ("val", 125),
    ("test", 70)
]:
    count = len(list((DATA / split).glob("*.npz")))
    print(f"{split}: {count} samples")
    assert count == expected

# 4. Restore import paths
sys.path.insert(0, str(CODE))

from ablation_model import UAFloodNet
from data import NPZFloodDataset

print("\nMODEL IMPORTS: SUCCESS")

# 5. Verify all three checkpoints
checkpoints = {
    "V1 control":
        BACKUP / "v4/full/control_seed42_50ep.pt",
    "V4 strong":
        BACKUP / "v4/full/augmented_seed42_50ep.pt",
    "V4 light":
        BACKUP / "v4/light/light_seed42_50ep.pt"
}

print("\nCHECKPOINT VERIFICATION")

for name, path in checkpoints.items():
    assert path.exists(), f"Missing checkpoint: {path}"
    print(name, "FOUND")

print("\nGPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

print("\nUA-FLOODNET ENVIRONMENT READY")

In [ ]:
import torch
from ablation_model import UAFloodNet

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

models = {}

for name, path in checkpoints.items():

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    model = UAFloodNet(
        width=ckpt["width"],
        use_alignment=ckpt["use_alignment"],
        use_reliability=ckpt["use_reliability"]
    ).to(device)

    model.load_state_dict(ckpt["model"])
    model.eval()

    models[name] = model

    print(
        name,
        "| best epoch:", ckpt["epoch"],
        "| validation IoU:",
        round(ckpt["metrics"]["flood_iou"], 5)
    )

print("\nALL THREE MODELS LOADED SUCCESSFULLY")

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from data import NPZFloodDataset

# Models are already loaded from Step 63.
assert set(models.keys()) == {
    "V1 control", "V4 strong", "V4 light"
}

device = next(iter(models.values())).parameters().__next__().device

dataset = NPZFloodDataset(
    "/content/UA_FloodNet/dataset/val"
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

assert len(dataset) == 125

def shift_image(x, dx, dy):
    _, _, h, w = x.shape
    result = torch.zeros_like(x)

    y0 = max(0, dy)
    y1 = min(h, h + dy)
    x0 = max(0, dx)
    x1 = min(w, w + dx)

    if y1 > y0 and x1 > x0:
        result[:, :, y0:y1, x0:x1] = x[
            :, :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]

    return result

# Include zero shift and translations up to 8 pixels.
SHIFTS = [
    (0, 0),
    (-1, 0), (1, 0),
    (-2, 0), (2, 0),
    (-4, 0), (4, 0),
    (-8, 0), (8, 0),
    (0, -2), (0, 2),
    (0, -4), (0, 4)
]

THRESHOLD = 0.50
MARGIN = 20
records = []

for model in models.values():
    model.eval()

with torch.inference_mode():

    for dx, dy in SHIFTS:

        totals = {
            name: {"tp": 0, "fp": 0, "fn": 0}
            for name in models
        }

        for batch in loader:

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            # SAR remains unchanged.
            sar_pre = batch["sar_pre"]
            sar_post = batch["sar_post"]

            # Apply identical translations to optical dates.
            ms_pre = shift_image(batch["ms_pre"], dx, dy)
            ms_post = shift_image(batch["ms_post"], dx, dy)

            gt = batch["mask"] >= 0.5

            # Evaluate every model over the same interior.
            valid = torch.zeros_like(gt, dtype=torch.bool)
            valid[
                :,
                :,
                MARGIN:-MARGIN,
                MARGIN:-MARGIN
            ] = True

            for name, model in models.items():

                output = model(
                    sar_pre,
                    sar_post,
                    ms_pre,
                    ms_post
                )

                pred = (
                    torch.sigmoid(output["logits"])
                    >= THRESHOLD
                )

                totals[name]["tp"] += (
                    pred & gt & valid
                ).sum().item()

                totals[name]["fp"] += (
                    pred & ~gt & valid
                ).sum().item()

                totals[name]["fn"] += (
                    ~pred & gt & valid
                ).sum().item()

        for name, counts in totals.items():

            tp = counts["tp"]
            fp = counts["fp"]
            fn = counts["fn"]

            iou = tp / max(tp + fp + fn, 1)
            f1 = 2 * tp / max(2 * tp + fp + fn, 1)

            records.append({
                "model": name,
                "dx": dx,
                "dy": dy,
                "shift_pixels": float(np.hypot(dx, dy)),
                "IoU": iou,
                "F1": f1
            })

        print(f"Completed shift ({dx:+d}, {dy:+d})", flush=True)

results64 = pd.DataFrame(records)

print("\nSTEP 64: THREE-MODEL ROBUSTNESS EVALUATION")
print("=" * 72)

print("\nIoU BY TRANSLATION")
print(
    results64.pivot_table(
        index=["dx", "dy"],
        columns="model",
        values="IoU"
    ).round(5).to_string()
)

summary = []

for name, group in results64.groupby("model"):

    clean = group[
        (group.dx == 0) & (group.dy == 0)
    ].iloc[0]

    shifted = group[
        ~((group.dx == 0) & (group.dy == 0))
    ]

    summary.append({
        "model": name,
        "clean_iou": clean.IoU,
        "mean_shifted_iou": shifted.IoU.mean(),
        "mean_iou_loss": clean.IoU - shifted.IoU.mean(),
        "worst_shifted_iou": shifted.IoU.min(),
        "clean_f1": clean.F1,
        "mean_shifted_f1": shifted.F1.mean()
    })

summary64 = pd.DataFrame(summary).set_index("model")

print("\nROBUSTNESS SUMMARY")
print(summary64.round(5).to_string())

print("\nDIFFERENCE FROM V1 CONTROL")

control = summary64.loc["V1 control"]

for name in ["V4 strong", "V4 light"]:
    row = summary64.loc[name]

    print(f"\n{name}")
    print(
        "Clean IoU difference:",
        round(row.clean_iou - control.clean_iou, 5)
    )
    print(
        "Mean shifted IoU difference:",
        round(
            row.mean_shifted_iou - control.mean_shifted_iou,
            5
        )
    )
    print(
        "Mean degradation difference:",
        round(row.mean_iou_loss - control.mean_iou_loss, 5)
    )

# Save results permanently
save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/robustness"
)
save_dir.mkdir(parents=True, exist_ok=True)

results64.to_csv(
    save_dir / "step64_three_model_results.csv",
    index=False
)

summary64.to_csv(
    save_dir / "step64_three_model_summary.csv"
)

print("\nSTEP 64 COMPLETED")
print("Results saved to:", save_dir)

In [ ]:
%%writefile /content/UA_FloodNet/ablations_code/train_multiseed.py
import argparse
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from ablation_model import UAFloodNet, training_loss
from data import NPZFloodDataset
from train_v4 import augment_misregistration


def evaluate(model, loader, device):
    model.eval()
    tp = fp = fn = 0

    with torch.inference_mode():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            logits = model(
                batch["sar_pre"], batch["sar_post"],
                batch["ms_pre"], batch["ms_post"]
            )["logits"]

            pred = logits.sigmoid() >= 0.5
            gt = batch["mask"] >= 0.5

            tp += (pred & gt).sum().item()
            fp += (pred & ~gt).sum().item()
            fn += (~pred & gt).sum().item()

    return {
        "flood_iou": tp / max(tp + fp + fn, 1),
        "flood_f1": 2 * tp / max(2 * tp + fp + fn, 1)
    }


def atomic_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(obj, temporary)
    temporary.replace(path)


def main():
    p = argparse.ArgumentParser()

    p.add_argument("--train", required=True)
    p.add_argument("--val", required=True)
    p.add_argument("--out-dir", required=True)
    p.add_argument("--seed", type=int, required=True)
    p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--batch-size", type=int, default=4)
    p.add_argument("--width", type=int, default=24)
    p.add_argument("--lr", type=float, default=0.001)
    p.add_argument("--shift-prob", type=float, default=0.0)
    p.add_argument("--max-shift", type=int, default=2)

    args = p.parse_args()

    if not 0 <= args.shift_prob <= 1:
        raise ValueError("Invalid shift probability")

    if args.epochs < 1 or args.max_shift < 0:
        raise ValueError("Invalid epoch count or max shift")

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    last_path = out / "last.pt"
    best_path = out / "best.pt"
    history_path = out / "history.json"

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # Separate generator makes shuffle state restorable.
    generator = torch.Generator()
    generator.manual_seed(args.seed)

    train_loader = DataLoader(
        NPZFloodDataset(args.train),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator
    )

    val_loader = DataLoader(
        NPZFloodDataset(args.val),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=0
    )

    model = UAFloodNet(
        width=args.width,
        use_alignment=True,
        use_reliability=True
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=args.lr
    )

    start_epoch = 1
    best_iou = -1.0
    history = []

    config = {
        "seed": args.seed,
        "width": args.width,
        "batch_size": args.batch_size,
        "lr": args.lr,
        "shift_prob": args.shift_prob,
        "max_shift": args.max_shift,
        "train": str(Path(args.train).resolve()),
        "val": str(Path(args.val).resolve()),
    }

    if last_path.exists():
        checkpoint = torch.load(
            last_path,
            map_location="cpu",
            weights_only=True
        )

        if checkpoint["config"] != config:
            raise RuntimeError(
                "Saved checkpoint configuration differs "
                "from the requested experiment."
            )

        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])

        # Transfer optimizer state tensors to training device.
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(device)

        generator.set_state(checkpoint["shuffle_rng"])
        torch.set_rng_state(checkpoint["torch_rng"])
        if device.type == "cuda":
            torch.cuda.set_rng_state_all(
                checkpoint["cuda_rng"]
            )

        start_epoch = checkpoint["epoch"] + 1
        best_iou = checkpoint["best_iou"]
        history = checkpoint["history"]

        print(
            f"Resuming after epoch {start_epoch-1}",
            flush=True
        )

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            batch = augment_misregistration(
                batch,
                args.shift_prob,
                args.max_shift
            )

            optimizer.zero_grad(set_to_none=True)

            output = model(
                batch["sar_pre"], batch["sar_post"],
                batch["ms_pre"], batch["ms_post"]
            )

            loss, _ = training_loss(
                model, output, batch, 0.1, 0.1, 0.01
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 5.0
            )

            optimizer.step()
            total_loss += loss.item()

        metrics = evaluate(model, val_loader, device)
        mean_loss = total_loss / len(train_loader)

        record = {
            "epoch": epoch,
            "train_loss": mean_loss,
            **metrics
        }
        history.append(record)

        improved = metrics["flood_iou"] > best_iou

        if improved:
            best_iou = metrics["flood_iou"]

            atomic_save({
                "model": model.state_dict(),
                "epoch": epoch,
                "width": args.width,
                "metrics": metrics,
                "seed": args.seed,
                "shift_prob": args.shift_prob,
                "max_shift": args.max_shift,
                "variant": "full",
                "use_alignment": True,
                "use_reliability": True
            }, best_path)

        # Save all state needed to continue training.
        atomic_save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "best_iou": best_iou,
            "history": history,
            "config": config,
            "shuffle_rng": generator.get_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": (
                torch.cuda.get_rng_state_all()
                if device.type == "cuda" else []
            )
        }, last_path)

        history_path.write_text(
            json.dumps(history, indent=2)
        )

        print(
            f"epoch={epoch} "
            f"loss={mean_loss:.4f} "
            f"val_iou={metrics['flood_iou']:.4f} "
            f"val_f1={metrics['flood_f1']:.4f} "
            f"best_iou={best_iou:.4f}",
            flush=True
        )

    print(
        f"FINISHED epoch={args.epochs} "
        f"best_iou={best_iou:.4f}",
        flush=True
    )


if __name__ == "__main__":
    main()

In [ ]:
import py_compile
from pathlib import Path
import shutil

script = Path(
    "/content/UA_FloodNet/ablations_code/train_multiseed.py"
)

py_compile.compile(str(script), doraise=True)

backup = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/code"
)
backup.mkdir(parents=True, exist_ok=True)

shutil.copy2(script, backup / script.name)

print("Syntax verification: PASSED")
print("Training script backed up:", backup / script.name)

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_multiseed.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 2 \
    --batch-size 4 \
    --width 24 \
    --lr 0.001 \
    --seed 7 \
    --shift-prob 0.0 \
    --max-shift 2 \
    --out-dir /content/drive/MyDrive/UA_FloodNet/v4/multiseed/control_seed7

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_multiseed.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --lr 0.001 \
    --seed 7 \
    --shift-prob 0.0 \
    --max-shift 2 \
    --out-dir /content/drive/MyDrive/UA_FloodNet/v4/multiseed/control_seed7

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python -u train_multiseed.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --lr 0.001 \
    --seed 7 \
    --shift-prob 0.25 \
    --max-shift 2 \
    --out-dir /content/drive/MyDrive/UA_FloodNet/v4/multiseed/light_seed7

In [ ]:
%cd /content/UA_FloodNet/ablations_code

import subprocess
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/UA_FloodNet/v4/multiseed")

experiments = [
    ("control_seed17", 0.0, 2),
    ("light_seed17", 0.25, 2),
]

for name, shift_prob, max_shift in experiments:
    out_dir = ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)

    command = [
        "python", "-u", "train_multiseed.py",
        "--train", "/content/UA_FloodNet/dataset/train",
        "--val", "/content/UA_FloodNet/dataset/val",
        "--epochs", "50",
        "--batch-size", "4",
        "--width", "24",
        "--lr", "0.001",
        "--seed", "17",
        "--shift-prob", str(shift_prob),
        "--max-shift", str(max_shift),
        "--out-dir", str(out_dir)
    ]

    print(f"\nSTARTING: {name}", flush=True)

    log_path = out_dir / "console.log"

    with log_path.open("a") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

    print(f"{name} exit code: {return_code}", flush=True)

    if return_code != 0:
        raise RuntimeError(
            f"{name} did not finish successfully; "
            f"inspect {log_path}"
        )

print("\nSTEP 68: BOTH SEED-17 RUNS FINISHED")

In [ ]:
%cd /content/UA_FloodNet/ablations_code

import subprocess
from pathlib import Path

root = Path("/content/drive/MyDrive/UA_FloodNet/v4/multiseed")

experiments = [
    ("control_seed37", 0.0, 2),
    ("light_seed37", 0.25, 2),
]

for name, shift_prob, max_shift in experiments:
    out_dir = root / name
    out_dir.mkdir(parents=True, exist_ok=True)

    command = [
        "python", "-u", "train_multiseed.py",
        "--train", "/content/UA_FloodNet/dataset/train",
        "--val", "/content/UA_FloodNet/dataset/val",
        "--epochs", "50",
        "--batch-size", "4",
        "--width", "24",
        "--lr", "0.001",
        "--seed", "37",
        "--shift-prob", str(shift_prob),
        "--max-shift", str(max_shift),
        "--out-dir", str(out_dir),
    ]

    print(f"\nSTARTING: {name}", flush=True)

    log_path = out_dir / "console.log"

    with log_path.open("a") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        exit_code = process.wait()

    print(f"{name} exit code: {exit_code}", flush=True)

    if exit_code != 0:
        raise RuntimeError(f"Training failed: {name}")

print("\nSTEP 69 COMPLETED: BOTH SEED-37 RUNS FINISHED")

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

# Mount Google Drive
drive.mount("/content/drive")

print("=" * 65)
print("UA-FLOODNET FILE RECOVERY")
print("=" * 65)

DRIVE = Path("/content/drive/MyDrive")
ROOT = DRIVE / "UA_FloodNet"
CODE = Path("/content/UA_FloodNet/ablations_code")

print("\n1. DRIVE STATUS")
print("Google Drive mounted:", DRIVE.exists())
print("Project folder exists:", ROOT.exists())
print("Working code directory exists:", CODE.exists())

print("\n2. CHECK GOOGLE DRIVE PROJECT FOLDERS")

if ROOT.exists():
    for folder in sorted(ROOT.iterdir()):
        print(folder.name)
else:
    print("UA_FloodNet not found at expected location")

print("\n3. SEARCH SOURCE CODE AND CHECKPOINTS")

filenames = [
    "ablation_model.py",
    "data.py",
    "train_v4.py",
    "train_multiseed.py",
    "control_seed42_50ep.pt",
    "light_seed42_50ep.pt",
    "best.pt",
    "last.pt"
]

for filename in filenames:
    matches = list(ROOT.rglob(filename)) if ROOT.exists() else []

    # Include any files still available in /content.
    local_root = Path("/content/UA_FloodNet")
    if local_root.exists():
        matches += list(local_root.rglob(filename))

    # Remove duplicate paths.
    matches = sorted(set(matches))

    print(f"\n{filename}: {len(matches)} matches")

    for path in matches[:15]:
        print("  ", path)

print("\n4. RESTORE AVAILABLE SOURCE FILES")

BACKUP = ROOT / "v4/code"
CODE.mkdir(parents=True, exist_ok=True)

for filename in [
    "ablation_model.py",
    "data.py",
    "train_v4.py",
    "train_multiseed.py"
]:
    source = BACKUP / filename
    target = CODE / filename

    if source.exists():
        shutil.copy2(source, target)
        print("Restored:", filename)
    elif target.exists():
        print("Already available:", filename)
    else:
        print("NOT FOUND:", filename)

print("\nSTEP 70A DIAGNOSTIC COMPLETED")

In [ ]:
from pathlib import Path
import torch
import ast
import hashlib
import json

CODE = Path("/content/UA_FloodNet/ablations_code")
DRIVE = Path("/content/drive/MyDrive/UA_FloodNet")

print("=" * 72)
print("STEP 70B: TRAINING CONFIGURATION AUDIT")
print("=" * 72)

# 1. Verify original and new source-code files.
for filename in ["train_v4.py", "train_multiseed.py"]:
    path = CODE / filename
    assert path.exists(), f"Missing source file: {path}"

    print(f"\n{'='*20} {filename} {'='*20}")

    tree = ast.parse(path.read_text())

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print("Function:", node.name)

    terms = (
        "Adam", "optimizer", "training_loss",
        "criterion", "contrast_weight", "recon_weight",
        "flow_weight", "weight_decay", "clip_grad",
        "augment_misregistration", "seed",
        "DataLoader", "scheduler"
    )

    lines = path.read_text().splitlines()

    for i, line in enumerate(lines):
        if any(term.lower() in line.lower() for term in terms):
            print(f"{i+1:04d}: {line.strip()}")

# 2. Verify checkpoint metadata.
checkpoints = {
    "control_42": DRIVE / "v4/full/control_seed42_50ep.pt",
    "light_42": DRIVE / "v4/light/light_seed42_50ep.pt"
}

for seed in (7, 17, 37):
    for variant in ("control", "light"):
        checkpoints[f"{variant}_{seed}"] = (
            DRIVE / "v4/multiseed"
            / f"{variant}_seed{seed}" / "best.pt"
        )

print("\nCHECKPOINT METADATA")
print("=" * 72)

records = []

for name, path in checkpoints.items():
    assert path.exists(), f"Missing checkpoint: {path}"

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    metric = ckpt.get("metrics", {})

    record = {
        "model": name,
        "seed": ckpt.get("seed"),
        "best_epoch": ckpt.get("epoch"),
        "width": ckpt.get("width"),
        "shift_probability": ckpt.get("shift_prob"),
        "max_shift": ckpt.get("max_shift"),
        "validation_iou": metric.get("flood_iou")
    }

    records.append(record)
    print(record)

# 3. Verify completion of newer experiments.
print("\nTRAINING COMPLETION")
print("=" * 72)

for seed in (7, 17, 37):
    for variant in ("control", "light"):
        path = (
            DRIVE / "v4/multiseed"
            / f"{variant}_seed{seed}" / "last.pt"
        )

        state = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        history = state["history"]

        assert state["epoch"] == 50
        assert len(history) == 50
        assert history[-1]["epoch"] == 50

        print(f"{variant}_seed{seed}: 50/50 VERIFIED")

# 4. Archive audit metadata.
audit_dir = DRIVE / "v4/audit"
audit_dir.mkdir(parents=True, exist_ok=True)

report = {
    "checkpoints": records,
    "source_sha256": {}
}

for filename in (
    "ablation_model.py",
    "data.py",
    "train_v4.py",
    "train_multiseed.py"
):
    path = CODE / filename

    report["source_sha256"][filename] = (
        hashlib.sha256(path.read_bytes()).hexdigest()
    )

(audit_dir / "step70_checkpoint_inventory.json").write_text(
    json.dumps(report, indent=2)
)

print("\nAUDIT INVENTORY SAVED")
print(audit_dir / "step70_checkpoint_inventory.json")

In [ ]:
from pathlib import Path

base = Path("/content/UA_FloodNet/ablations_code")

sections = {
    "train_v4.py": [
        (75, 120)
    ],
    "train_multiseed.py": [
        (108, 120),
        (175, 215)
    ]
}

for filename, ranges in sections.items():
    lines = (base / filename).read_text().splitlines()

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    for start, end in ranges:
        for i in range(start - 1, min(end, len(lines))):
            print(f"{i+1:03d}: {lines[i]}")

In [ ]:
import ast
from pathlib import Path

code_dir = Path("/content/UA_FloodNet/ablations_code")

functions = {
    "train_v4.py": "scores",
    "train_multiseed.py": "evaluate"
}

for filename, function_name in functions.items():
    source = (code_dir / filename).read_text()
    tree = ast.parse(source)
    lines = source.splitlines()

    matches = [
        node for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        and node.name == function_name
    ]

    assert len(matches) == 1, (
        f"Could not locate {function_name} in {filename}"
    )

    node = matches[0]

    print("\n" + "=" * 70)
    print(f"{filename} → {function_name}()")
    print("=" * 70)

    for index in range(node.lineno - 1, node.end_lineno):
        print(f"{index+1:03d}: {lines[index]}")

print("\nSTEP 70D SOURCE INSPECTION COMPLETED")

In [ ]:
from pathlib import Path
from google.colab import drive
import zipfile

# Mount Drive if required
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ARCHIVE = Path(
    "/content/drive/MyDrive/UA_FloodNet/"
    "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
)

DEST = Path("/content/UA_FloodNet/dataset")

print("STEP 71A: RESTORING FROZEN OMBRIA DATASET")
print("=" * 65)

assert ARCHIVE.exists(), f"Dataset backup not found: {ARCHIVE}"

print("Archive found:", ARCHIVE.name)
print("Archive size (MB):", round(ARCHIVE.stat().st_size / 1024**2, 2))

DEST.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ARCHIVE, "r") as z:
    bad = z.testzip()
    assert bad is None, f"Corrupt ZIP member: {bad}"

    # Check archive structure and reject unsafe paths.
    names = z.namelist()
    print("First archive entries:", names[:5])

    root = DEST.resolve()
    for member in z.infolist():
        target = (DEST / member.filename).resolve()
        if not target.is_relative_to(root):
            raise ValueError(f"Unsafe archive path: {member.filename}")

    z.extractall(DEST)

print("\nDATASET VERIFICATION")
print("=" * 65)

expected = {"train": 499, "val": 125, "test": 70}

for split, required in expected.items():
    folder = DEST / split
    files = list(folder.glob("*.npz"))

    print(f"{split}: {len(files)} / {required} files")

    assert len(files) == required, (
        f"Incorrect {split} count: {len(files)}"
    )

print("\nSUCCESS: DATASET RESTORED")
print("Validation path:", DEST / "val")

In [ ]:
import sys
from pathlib import Path
from torch.utils.data import DataLoader

CODE = Path("/content/UA_FloodNet/ablations_code")
sys.path.insert(0, str(CODE))

from data import NPZFloodDataset

dataset = NPZFloodDataset(
    "/content/UA_FloodNet/dataset/val"
)

loader = DataLoader(dataset, batch_size=4, shuffle=False)

batch = next(iter(loader))

print("Validation samples:", len(dataset))
print("SAR pre shape:", tuple(batch["sar_pre"].shape))
print("MS pre shape:", tuple(batch["ms_pre"].shape))
print("Mask shape:", tuple(batch["mask"].shape))

assert len(dataset) == 125
print("\nSTEP 71B PASSED — READY FOR ROBUSTNESS EVALUATION")

In [ ]:
import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

DRIVE = Path("/content/drive/MyDrive/UA_FloodNet")
CODE = Path("/content/UA_FloodNet/ablations_code")
VAL = Path("/content/UA_FloodNet/dataset/val")
SAVE = DRIVE / "v4/robustness/multiseed"

SAVE.mkdir(parents=True, exist_ok=True)
CODE.mkdir(parents=True, exist_ok=True)

# Restore code if this is a fresh runtime.
for filename in ("ablation_model.py", "data.py"):
    target = CODE / filename
    if not target.exists():
        source = DRIVE / "v4/code" / filename
        assert source.exists(), f"Missing source: {source}"
        shutil.copy2(source, target)

sys.path.insert(0, str(CODE))

from ablation_model import UAFloodNet
from data import NPZFloodDataset

# Do not silently evaluate the wrong dataset.
assert VAL.exists(), (
    "Validation dataset is missing. Restore the frozen "
    "candidate_seed42 NPZ archive before continuing."
)

dataset = NPZFloodDataset(str(VAL))
assert len(dataset) == 125, (
    f"Expected 125 validation patches, found {len(dataset)}"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Validation patches:", len(dataset))

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

# Exact checkpoint inventory
checkpoints = {}

for seed in (42, 7, 17, 37):
    for variant in ("control", "light"):

        if seed == 42:
            if variant == "control":
                path = DRIVE / "v4/full/control_seed42_50ep.pt"
            else:
                path = DRIVE / "v4/light/light_seed42_50ep.pt"
        else:
            path = (
                DRIVE / "v4/multiseed"
                / f"{variant}_seed{seed}"
                / "best.pt"
            )

        assert path.exists(), f"Missing checkpoint: {path}"
        checkpoints[(seed, variant)] = path

SHIFTS = [
    (0, 0),
    (-1, 0), (1, 0),
    (-2, 0), (2, 0),
    (-4, 0), (4, 0),
    (-8, 0), (8, 0),
    (0, -2), (0, 2),
    (0, -4), (0, 4)
]

MARGIN = 20
THRESHOLD = 0.5


def translate_zero(x, dx, dy):
    _, _, h, w = x.shape
    result = torch.zeros_like(x)

    y0, y1 = max(0, dy), min(h, h + dy)
    x0, x1 = max(0, dx), min(w, w + dx)

    if y1 > y0 and x1 > x0:
        result[:, :, y0:y1, x0:x1] = x[
            :, :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]

    return result


records = []

for (seed, variant), checkpoint_path in checkpoints.items():

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    assert checkpoint["seed"] == seed
    assert checkpoint["use_alignment"] is True
    assert checkpoint["use_reliability"] is True

    model = UAFloodNet(
        width=checkpoint["width"],
        use_alignment=True,
        use_reliability=True
    ).to(device)

    model.load_state_dict(checkpoint["model"])
    model.eval()

    print(
        f"\nEvaluating {variant}, seed {seed}, "
        f"best epoch {checkpoint['epoch']}",
        flush=True
    )

    for dx, dy in SHIFTS:

        tp = fp = fn = 0

        with torch.inference_mode():

            for batch in loader:

                sar_pre = batch["sar_pre"].to(device)
                sar_post = batch["sar_post"].to(device)

                ms_pre = translate_zero(
                    batch["ms_pre"].to(device), dx, dy
                )
                ms_post = translate_zero(
                    batch["ms_post"].to(device), dx, dy
                )

                gt = batch["mask"].to(device) >= 0.5

                logits = model(
                    sar_pre, sar_post, ms_pre, ms_post
                )["logits"]

                pred = logits.sigmoid() >= THRESHOLD

                valid = torch.zeros_like(gt, dtype=torch.bool)
                valid[
                    :, :, MARGIN:-MARGIN, MARGIN:-MARGIN
                ] = True

                tp += (pred & gt & valid).sum().item()
                fp += (pred & ~gt & valid).sum().item()
                fn += (~pred & gt & valid).sum().item()

        iou = tp / max(tp + fp + fn, 1)
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)

        records.append({
            "seed": seed,
            "variant": variant,
            "dx": dx,
            "dy": dy,
            "shift_pixels": float(np.hypot(dx, dy)),
            "IoU": iou,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn
        })

        print(
            f"  shift=({dx:+d},{dy:+d}) "
            f"IoU={iou:.5f}",
            flush=True
        )

    # Save incremental results after each completed model.
    pd.DataFrame(records).to_csv(
        SAVE / "step71_all_shift_results.csv",
        index=False
    )

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()


# Summarise each model and training seed.
results71 = pd.DataFrame(records)
summaries = []

for (seed, variant), group in results71.groupby(
    ["seed", "variant"]
):
    clean = group[
        (group.dx == 0) & (group.dy == 0)
    ].iloc[0]

    shifted = group[
        ~((group.dx == 0) & (group.dy == 0))
    ]

    summaries.append({
        "seed": seed,
        "variant": variant,
        "clean_iou": clean.IoU,
        "mean_shifted_iou": shifted.IoU.mean(),
        "mean_iou_loss": clean.IoU - shifted.IoU.mean(),
        "worst_shifted_iou": shifted.IoU.min(),
        "clean_f1": clean.F1,
        "mean_shifted_f1": shifted.F1.mean()
    })

summary71 = pd.DataFrame(summaries).sort_values(
    ["seed", "variant"]
)

print("\n" + "=" * 75)
print("STEP 71: EIGHT-CHECKPOINT ROBUSTNESS SUMMARY")
print("=" * 75)
print(summary71.round(5).to_string(index=False))

# Paired light-minus-control comparisons.
wide = summary71.pivot(
    index="seed",
    columns="variant",
    values=[
        "clean_iou",
        "mean_shifted_iou",
        "mean_iou_loss",
        "worst_shifted_iou"
    ]
)

paired = pd.DataFrame(index=wide.index)

for metric in (
    "clean_iou",
    "mean_shifted_iou",
    "mean_iou_loss",
    "worst_shifted_iou"
):
    paired[metric + "_difference"] = (
        wide[(metric, "light")]
        - wide[(metric, "control")]
    )

print("\nPAIRED DIFFERENCES: LIGHT MINUS CONTROL")
print(paired.round(5).to_string())

print("\nMEAN PAIRED DIFFERENCES")
print(paired.mean().round(5).to_string())

summary71.to_csv(
    SAVE / "step71_seed_summary.csv",
    index=False
)

paired.to_csv(
    SAVE / "step71_paired_differences.csv"
)

print("\nSTEP 71 COMPLETED")
print("Results saved to:", SAVE)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/robustness/multiseed"
)

paired = pd.read_csv(
    ROOT / "step71_paired_differences.csv",
    index_col=0
)

summary = pd.read_csv(
    ROOT / "step71_seed_summary.csv"
)

metrics = {
    "clean_iou_difference": "Clean IoU",
    "mean_shifted_iou_difference": "Mean shifted IoU",
    "mean_iou_loss_difference": "IoU degradation",
    "worst_shifted_iou_difference": "Worst shifted IoU"
}

records = []

for column, label in metrics.items():
    d = paired[column].dropna().to_numpy(dtype=float)
    n = len(d)

    assert n == 4, f"Expected four seeds for {label}"

    mean = d.mean()
    sd = d.std(ddof=1)
    se = sd / np.sqrt(n)

    critical = stats.t.ppf(0.975, df=n-1)

    ci_low = mean - critical * se
    ci_high = mean + critical * se

    t_stat, p_value = stats.ttest_1samp(d, 0)

    records.append({
        "metric": label,
        "n_seeds": n,
        "mean_difference": mean,
        "sd_difference": sd,
        "ci95_lower": ci_low,
        "ci95_upper": ci_high,
        "exploratory_p_value": p_value
    })

report = pd.DataFrame(records)

print("=" * 78)
print("STEP 72: FOUR-SEED PAIRED UNCERTAINTY ANALYSIS")
print("=" * 78)

print(report.round(6).to_string(index=False))

print("\nMEAN PERFORMANCE BY CONFIGURATION")
print(
    summary.groupby("variant")[
        [
            "clean_iou",
            "mean_shifted_iou",
            "mean_iou_loss",
            "worst_shifted_iou"
        ]
    ].agg(["mean", "std"]).round(6).to_string()
)

report.to_csv(
    ROOT / "step72_statistical_summary.csv",
    index=False
)

print("\nSTATISTICAL SUMMARY SAVED")
print(ROOT / "step72_statistical_summary.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile

ROOT = Path("/content/drive/MyDrive/UA_FloodNet")
BACKUP = ROOT / "dataset_backup"

print("=" * 70)
print("STEP 73A: OMBRIA EVENT-INDEPENDENCE AUDIT")
print("=" * 70)

print("\n1. DATASET BACKUP FILES")

for path in sorted(BACKUP.iterdir()):
    if path.is_file():
        size = path.stat().st_size / (1024**2)
        print(f"{path.name} | {size:.2f} MB")

print("\n2. SPLIT MANIFEST")

manifest = BACKUP / "candidate_split_seed42.csv"

if manifest.exists():
    df = pd.read_csv(manifest)

    print("Manifest shape:", df.shape)
    print("Columns:", list(df.columns))
    print("\nFirst 10 records:")
    print(df.head(10).to_string(index=False))

    for column in df.columns:
        print(
            f"\n{column}: "
            f"{df[column].nunique(dropna=False)} unique values"
        )
else:
    print("Manifest not found.")

print("\n3. METADATA FILES")

for path in sorted(BACKUP.glob("*.json")):
    print("\nMetadata:", path.name)
    print(path.read_text()[:4000])

print("\n4. AVAILABLE SOURCE ARCHIVES")

archives = list(ROOT.rglob("*.zip"))

for path in archives:
    print("\nArchive:", path)

    with zipfile.ZipFile(path) as z:
        print("Total entries:", len(z.namelist()))
        print("First 15 entries:")
        for name in z.namelist()[:15]:
            print(" ", name)

print("\nSTEP 73A COMPLETE")

In [ ]:
import requests

url = "https://api.github.com/repos/geodrak/OMBRIA/git/trees/HEAD?recursive=1"

response = requests.get(url, timeout=30)
response.raise_for_status()

tree = response.json().get("tree", [])

print("OMBRIA REPOSITORY INVENTORY")
print("=" * 65)

keywords = (
    ".csv", ".json", ".geojson", ".shp",
    "metadata", "event", "location",
    "readme", "test", "train"
)

matches = [
    item["path"]
    for item in tree
    if any(k in item["path"].lower() for k in keywords)
]

print("Matching repository entries:", len(matches))

for path in matches[:100]:
    print(path)

In [ ]:
import requests
from pathlib import Path

url = "https://api.github.com/repos/geodrak/OMBRIA/git/trees/HEAD?recursive=1"

response = requests.get(url, timeout=60)
response.raise_for_status()

payload = response.json()
assert not payload.get("truncated", False), (
    "Repository inventory is incomplete."
)

tree = payload["tree"]

# Exclude all imagery.
image_extensions = {
    ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"
}

# Inspect documentation, metadata and small source files.
relevant_extensions = {
    ".csv", ".json", ".geojson", ".shp",
    ".kml", ".gpkg", ".xlsx", ".xml",
    ".txt", ".md", ".ipynb"
}

selected = []

for item in tree:
    if item.get("type") != "blob":
        continue

    path = item["path"]
    suffix = Path(path).suffix.lower()

    if suffix in image_extensions:
        continue

    if suffix in relevant_extensions:
        selected.append(path)

print("STEP 73C: METADATA AND DOCUMENTATION INVENTORY")
print("=" * 70)
print("Relevant non-image files:", len(selected))

for path in selected:
    print(path)

print("\nPOTENTIAL EVENT/GEOGRAPHIC METADATA")
print("=" * 70)

metadata_terms = (
    "event", "metadata", "coordinate",
    "location", "geograph", "footprint",
    "source", "mapping", "manifest"
)

candidates = [
    path for path in selected
    if any(term in path.lower() for term in metadata_terms)
]

if candidates:
    for path in candidates:
        print(path)
else:
    print("No explicitly named event/geographic metadata files.")

print("\nSTEP 73C COMPLETED")

In [ ]:
import requests
import json
import re
from pathlib import Path

OWNER = "geodrak"
REPO = "OMBRIA"
BRANCH = "master"

files = [
    "patches.ipynb",
    "Reconstruct.ipynb",
    "Create 3D Images.ipynb",
    "README.md"
]

terms = re.compile(
    r"geotransform|georeferenc|coordinate|latitude|longitude|"
    r"latitude|longitude|utm|epsg|crs|transform|bounds|"
    r"event|activation|ems|location|region|city|country|"
    r"patch|tile|filename|save|export|rasterio",
    re.IGNORECASE
)

out_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/audit"
)
out_dir.mkdir(parents=True, exist_ok=True)

report = []

for filename in files:
    url = (
        f"https://raw.githubusercontent.com/"
        f"{OWNER}/{REPO}/{BRANCH}/{filename}"
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    if filename.endswith(".ipynb"):
        notebook = response.json()

        cells = [
            (i, cell)
            for i, cell in enumerate(notebook.get("cells", []))
            if cell.get("cell_type") in ("code", "markdown")
        ]
    else:
        cells = [
            (0, {"cell_type": "markdown",
                 "source": response.text.splitlines()})
        ]

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    for index, cell in cells:
        source = cell.get("source", [])
        if isinstance(source, list):
            source = "".join(source)

        lines = source.splitlines()
        matches = [
            (line_no, line.strip())
            for line_no, line in enumerate(lines, start=1)
            if terms.search(line)
        ]

        if not matches:
            continue

        print(f"\nCELL {index} ({cell['cell_type']})")

        for line_no, line in matches[:15]:
            print(f"  L{line_no}: {line[:250]}")

        report.append({
            "file": filename,
            "cell": index,
            "cell_type": cell["cell_type"],
            "matches": matches
        })

report_path = out_dir / "step73d_source_notebook_audit.json"
report_path.write_text(json.dumps(report, indent=2))

print("\nSTEP 73D COMPLETED")
print("Audit saved to:", report_path)

In [ ]:
import sys
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

DRIVE = Path("/content/drive/MyDrive/UA_FloodNet")
CODE = Path("/content/UA_FloodNet/ablations_code")
TEST = Path("/content/UA_FloodNet/dataset/test")
SAVE = DRIVE / "v4/official_test"

SAVE.mkdir(parents=True, exist_ok=True)
CODE.mkdir(parents=True, exist_ok=True)

for filename in ("ablation_model.py", "data.py"):
    target = CODE / filename
    if not target.exists():
        shutil.copy2(DRIVE / "v4/code" / filename, target)

sys.path.insert(0, str(CODE))

from ablation_model import UAFloodNet
from data import NPZFloodDataset

assert TEST.exists(), "Official test dataset not found"

dataset = NPZFloodDataset(str(TEST))
assert len(dataset) == 70, (
    f"Expected 70 test samples, found {len(dataset)}"
)

loader = DataLoader(
    dataset, batch_size=4, shuffle=False, num_workers=0
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SHIFTS = [
    (0, 0),
    (-1, 0), (1, 0),
    (-2, 0), (2, 0),
    (-4, 0), (4, 0),
    (-8, 0), (8, 0),
    (0, -2), (0, 2),
    (0, -4), (0, 4)
]

THRESHOLD = 0.5
MARGIN = 20

# Prevent accidental repetition or replacement of the final test.
final_results_path = SAVE / "step74_frozen_test_results.csv"
assert not final_results_path.exists(), (
    "Official test results already exist. "
    "Do not rerun or overwrite the locked evaluation."
)

def shift_zero(x, dx, dy):
    _, _, h, w = x.shape
    result = torch.zeros_like(x)

    y0, y1 = max(0, dy), min(h, h + dy)
    x0, x1 = max(0, dx), min(w, w + dx)

    if x1 > x0 and y1 > y0:
        result[:, :, y0:y1, x0:x1] = x[
            :, :, y0-dy:y1-dy, x0-dx:x1-dx
        ]
    return result

checkpoints = {}

for seed in (7, 17, 37, 42):
    for variant in ("control", "light"):
        if seed == 42:
            path = (
                DRIVE / "v4/full/control_seed42_50ep.pt"
                if variant == "control"
                else DRIVE / "v4/light/light_seed42_50ep.pt"
            )
        else:
            path = (
                DRIVE / "v4/multiseed"
                / f"{variant}_seed{seed}/best.pt"
            )

        assert path.exists(), f"Missing checkpoint: {path}"
        checkpoints[(seed, variant)] = path

protocol = {
    "dataset": "OMBRIA official source test partition",
    "samples": 70,
    "seeds": [7, 17, 37, 42],
    "variants": ["control", "light"],
    "threshold": THRESHOLD,
    "margin": MARGIN,
    "shifts_dx_dy": SHIFTS,
    "metric": "pooled pixel-level flood IoU and F1",
    "checkpoint_rule": "best validation IoU",
    "event_independence": "not verified",
    "model_selection_using_test": False
}

(SAVE / "step74_protocol.json").write_text(
    json.dumps(protocol, indent=2)
)

records = []

print("STEP 74: OFFICIAL HELD-OUT TEST")
print("Device:", device)
print("Test images:", len(dataset))

for (seed, variant), path in checkpoints.items():
    ckpt = torch.load(
        path, map_location="cpu", weights_only=True
    )

    assert ckpt["seed"] == seed
    assert bool(ckpt["use_alignment"])
    assert bool(ckpt["use_reliability"])

    model = UAFloodNet(
        width=ckpt["width"],
        use_alignment=True,
        use_reliability=True
    ).to(device)

    model.load_state_dict(ckpt["model"])
    model.eval()

    for dx, dy in SHIFTS:
        tp = fp = fn = 0

        with torch.inference_mode():
            for batch in loader:
                sar_pre = batch["sar_pre"].to(device)
                sar_post = batch["sar_post"].to(device)

                ms_pre = shift_zero(
                    batch["ms_pre"].to(device), dx, dy
                )
                ms_post = shift_zero(
                    batch["ms_post"].to(device), dx, dy
                )

                gt = batch["mask"].to(device) >= 0.5

                logits = model(
                    sar_pre, sar_post, ms_pre, ms_post
                )["logits"]

                pred = logits.sigmoid() >= THRESHOLD

                valid = torch.zeros_like(gt, dtype=torch.bool)
                valid[:, :, MARGIN:-MARGIN, MARGIN:-MARGIN] = True

                tp += (pred & gt & valid).sum().item()
                fp += (pred & ~gt & valid).sum().item()
                fn += (~pred & gt & valid).sum().item()

        records.append({
            "seed": seed,
            "variant": variant,
            "dx": dx,
            "dy": dy,
            "IoU": tp / max(tp + fp + fn, 1),
            "F1": 2 * tp / max(2 * tp + fp + fn, 1),
            "TP": tp,
            "FP": fp,
            "FN": fn
        })

    print(
        f"Finished {variant}, seed {seed}",
        flush=True
    )

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(records)

assert len(results) == 104

summary = []

for (seed, variant), group in results.groupby(
    ["seed", "variant"]
):
    clean = group[
        (group.dx == 0) & (group.dy == 0)
    ].iloc[0]

    shifted = group[
        ~((group.dx == 0) & (group.dy == 0))
    ]

    summary.append({
        "seed": seed,
        "variant": variant,
        "clean_iou": clean.IoU,
        "mean_shifted_iou": shifted.IoU.mean(),
        "mean_iou_loss": clean.IoU - shifted.IoU.mean(),
        "worst_shifted_iou": shifted.IoU.min(),
        "clean_f1": clean.F1
    })

summary = pd.DataFrame(summary).sort_values(
    ["seed", "variant"]
)

results.to_csv(final_results_path, index=False)
summary.to_csv(
    SAVE / "step74_frozen_test_summary.csv",
    index=False
)

print("\nOFFICIAL TEST SUMMARY")
print(summary.round(5).to_string(index=False))

print("\nGROUP MEANS")
print(
    summary.groupby("variant")[
        [
            "clean_iou",
            "mean_shifted_iou",
            "mean_iou_loss",
            "worst_shifted_iou"
        ]
    ].mean().round(5).to_string()
)

print("\nSTEP 74 COMPLETED")
print("Saved to:", SAVE)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import json
import hashlib

ROOT = Path("/content/drive/MyDrive/UA_FloodNet/v4")
VAL = ROOT / "robustness/multiseed/step71_seed_summary.csv"
TEST = ROOT / "official_test/step74_frozen_test_summary.csv"
OUT = ROOT / "final_analysis"

OUT.mkdir(parents=True, exist_ok=True)

metrics = [
    "clean_iou",
    "mean_shifted_iou",
    "mean_iou_loss",
    "worst_shifted_iou"
]

def paired_analysis(path, partition):
    df = pd.read_csv(path)

    assert set(df["seed"]) == {7, 17, 37, 42}
    assert len(df) == 8
    assert not df.duplicated(["seed", "variant"]).any()

    wide = df.pivot(
        index="seed",
        columns="variant",
        values=metrics
    )

    records = []
    differences = pd.DataFrame(index=wide.index)

    for metric in metrics:
        d = (
            wide[(metric, "light")]
            - wide[(metric, "control")]
        ).to_numpy(dtype=float)

        differences[metric] = d

        mean = d.mean()
        sd = d.std(ddof=1)
        critical = stats.t.ppf(0.975, df=3)
        half_width = critical * sd / np.sqrt(4)

        records.append({
            "partition": partition,
            "metric": metric,
            "control_mean": wide[
                (metric, "control")
            ].mean(),
            "light_mean": wide[
                (metric, "light")
            ].mean(),
            "mean_difference": mean,
            "paired_sd": sd,
            "ci95_lower": mean - half_width,
            "ci95_upper": mean + half_width,
            "exploratory_p": stats.ttest_1samp(
                d, 0
            ).pvalue,
            "seeds_favoring_light": int(
                (d < 0).sum()
                if metric == "mean_iou_loss"
                else (d > 0).sum()
            )
        })

    differences.index.name = "seed"

    differences.to_csv(
        OUT / f"{partition}_paired_seed_differences.csv"
    )

    return records

records = (
    paired_analysis(VAL, "validation")
    + paired_analysis(TEST, "official_test")
)

report = pd.DataFrame(records)

print("=" * 85)
print("STEP 75: FINAL FOUR-SEED STATISTICAL SYNTHESIS")
print("=" * 85)

print(
    report.round(6).to_string(index=False)
)

report.to_csv(
    OUT / "step75_validation_test_statistics.csv",
    index=False
)

# Preserve source-file fingerprints.
sources = {
    "validation_summary_sha256":
        hashlib.sha256(VAL.read_bytes()).hexdigest(),
    "official_test_summary_sha256":
        hashlib.sha256(TEST.read_bytes()).hexdigest()
}

(OUT / "step75_source_hashes.json").write_text(
    json.dumps(sources, indent=2)
)

print("\nSTEP 75 COMPLETED")
print("Final analysis directory:", OUT)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/UA_FloodNet/v4")
OUT = ROOT / "final_analysis/figures"
OUT.mkdir(parents=True, exist_ok=True)

val = pd.read_csv(
    ROOT / "robustness/multiseed/step71_seed_summary.csv"
)
test = pd.read_csv(
    ROOT / "official_test/step74_frozen_test_summary.csv"
)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "savefig.dpi": 600
})

# Figure 1: Degradation by training seed
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, (name, df) in zip(
    axes,
    [("Candidate validation", val), ("Official held-out test", test)]
):
    data = df.pivot(
        index="seed",
        columns="variant",
        values="mean_iou_loss"
    ).sort_index()

    x = np.arange(len(data))
    w = 0.36

    ax.bar(x - w/2, data["control"], w,
           label="Control", color="#355C7D")
    ax.bar(x + w/2, data["light"], w,
           label="Light augmentation", color="#2A9D8F")

    ax.set_xticks(x, data.index.astype(str))
    ax.set_xlabel("Training seed")
    ax.set_title(name, fontweight="bold")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

axes[0].set_ylabel("IoU degradation (clean − mean shifted)")
axes[0].legend(frameon=False)

fig.tight_layout()
fig.savefig(OUT / "Figure_1_Seed_Robustness.png",
            dpi=600, bbox_inches="tight")
fig.savefig(OUT / "Figure_1_Seed_Robustness.pdf",
            bbox_inches="tight")
plt.show()

# Figure 2: Paired differences and 95% CI
stats = pd.read_csv(
    ROOT / "final_analysis/step75_validation_test_statistics.csv"
)

subset = stats[
    stats["metric"] == "mean_iou_loss"
].copy()

fig, ax = plt.subplots(figsize=(8, 4.8))

x = np.arange(len(subset))
means = subset["mean_difference"].to_numpy()
lower = subset["ci95_lower"].to_numpy()
upper = subset["ci95_upper"].to_numpy()

ax.errorbar(
    x, means,
    yerr=[means - lower, upper - means],
    fmt="o", markersize=9, capsize=7,
    color="#355C7D", linewidth=2
)

ax.axhline(0, color="#777777", linestyle="--", linewidth=1)
ax.set_xticks(
    x,
    ["Candidate validation", "Official held-out test"]
)
ax.set_ylabel("Paired difference in IoU degradation\n(light − control)")
ax.set_title("Effect of Light Augmentation on Shift Sensitivity",
             fontweight="bold")
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()
fig.savefig(OUT / "Figure_2_Paired_Robustness_Uncertainty.png",
            dpi=600, bbox_inches="tight")
fig.savefig(OUT / "Figure_2_Paired_Robustness_Uncertainty.pdf",
            bbox_inches="tight")
plt.show()

print("STEP 76 COMPLETED")
print("Figures saved to:", OUT)

In [ ]:
import sys
import shutil
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

DRIVE = Path("/content/drive/MyDrive/UA_FloodNet")
CODE = Path("/content/UA_FloodNet/ablations_code")
TEST = Path("/content/UA_FloodNet/dataset/test")
OUT = DRIVE / "v4/final_analysis/figures/qualitative"

OUT.mkdir(parents=True, exist_ok=True)

# Restore model code if required.
CODE.mkdir(parents=True, exist_ok=True)

for filename in ("ablation_model.py", "data.py"):
    target = CODE / filename

    if not target.exists():
        shutil.copy2(
            DRIVE / "v4/code" / filename,
            target
        )

sys.path.insert(0, str(CODE))

from ablation_model import UAFloodNet

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Use fixed, previously trained seed-42 models.
paths = {
    "Control": DRIVE / "v4/full/control_seed42_50ep.pt",
    "Light": DRIVE / "v4/light/light_seed42_50ep.pt"
}

models = {}

for name, path in paths.items():

    assert path.exists(), f"Missing checkpoint: {path}"

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    model = UAFloodNet(
        width=ckpt["width"],
        use_alignment=ckpt["use_alignment"],
        use_reliability=ckpt["use_reliability"]
    ).to(device)

    model.load_state_dict(ckpt["model"])
    model.eval()

    models[name] = model

print("Both models loaded successfully.")

# Deterministic sample selection, independent of performance.
test_files = sorted(TEST.glob("*.npz"))

assert len(test_files) == 70, (
    f"Expected 70 test images; found {len(test_files)}"
)

indices = [0, 34, 69]
selected_files = [test_files[i] for i in indices]

print("\nSelected official test samples:")

for file in selected_files:
    print(file.name)


def translate_zero(x, dx=0, dy=0):
    _, _, h, w = x.shape

    result = torch.zeros_like(x)

    y0, y1 = max(0, dy), min(h, h + dy)
    x0, x1 = max(0, dx), min(w, w + dx)

    if y1 > y0 and x1 > x0:
        result[:, :, y0:y1, x0:x1] = x[
            :, :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]

    return result


def to_display_rgb(tensor):
    # OMBRIA's saved optical input is a 3-channel composite.
    # No unverified spectral-band names are assumed.
    image = tensor.detach().cpu().numpy()

    if image.ndim == 3:
        image = np.transpose(image, (1, 2, 0))

    return np.clip(image, 0, 1)


def make_error_map(pred, gt):
    # 0 correct background
    # 1 true positive flood
    # 2 false positive
    # 3 false negative

    error = np.zeros(gt.shape, dtype=np.uint8)

    error[pred & gt] = 1
    error[pred & ~gt] = 2
    error[~pred & gt] = 3

    return error


error_cmap = ListedColormap([
    "#F0F0F0",  # Correct background
    "#287D8E",  # True positive
    "#E69F00",  # False positive
    "#C73E5A"   # False negative
])

error_norm = BoundaryNorm(
    [-0.5, 0.5, 1.5, 2.5, 3.5],
    error_cmap.N
)

mask_cmap = ListedColormap([
    "#F4F4F4",
    "#2474A6"
])

for file in selected_files:

    with np.load(file) as data:
        sample = {
            key: torch.tensor(data[key]).float().unsqueeze(0).to(device)
            for key in (
                "sar_pre",
                "sar_post",
                "ms_pre",
                "ms_post",
                "mask"
            )
        }

    gt = (
        sample["mask"][0, 0].cpu().numpy() >= 0.5
    )

    sar_after = sample["sar_post"][0, 0].cpu().numpy()

    for dx, dy, condition in [
        (0, 0, "clean"),
        (4, 0, "shift4")
    ]:

        ms_pre = translate_zero(sample["ms_pre"], dx, dy)
        ms_post = translate_zero(sample["ms_post"], dx, dy)

        preds = {}

        with torch.inference_mode():

            for name, model in models.items():

                logits = model(
                    sample["sar_pre"],
                    sample["sar_post"],
                    ms_pre,
                    ms_post
                )["logits"]

                preds[name] = (
                    logits.sigmoid()[0, 0].cpu().numpy()
                    >= 0.5
                )

        # Error map for light model.
        errors = make_error_map(preds["Light"], gt)

        panels = [
            (sar_after, "a  |  SAR after flood", "gray"),
            (
                to_display_rgb(ms_post[0]),
                "b  |  Optical after flood",
                None
            ),
            (gt.astype(int), "c  |  Reference mask", mask_cmap),
            (
                preds["Control"].astype(int),
                "d  |  Control prediction",
                mask_cmap
            ),
            (
                preds["Light"].astype(int),
                "e  |  Light prediction",
                mask_cmap
            ),
            (errors, "f  |  Light prediction errors", error_cmap)
        ]

        fig, axes = plt.subplots(
            2, 3,
            figsize=(12.5, 8.5),
            constrained_layout=True
        )

        for ax, (image, title, cmap) in zip(
            axes.flat, panels
        ):
            if title.startswith("f"):
                ax.imshow(
                    image,
                    cmap=error_cmap,
                    norm=error_norm,
                    interpolation="nearest"
                )
            else:
                ax.imshow(
                    image,
                    cmap=cmap,
                    vmin=0 if cmap is not None else None,
                    vmax=1 if cmap is not None else None,
                    interpolation="nearest"
                )

            ax.set_title(title, fontsize=11)
            ax.set_xticks([])
            ax.set_yticks([])

        filename = (
            f"Figure_Qualitative_{file.stem}_{condition}"
        )

        fig.savefig(
            OUT / f"{filename}.png",
            dpi=600,
            bbox_inches="tight"
        )

        fig.savefig(
            OUT / f"{filename}.pdf",
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)

        print("Saved:", filename)

print("\nSTEP 77A COMPLETED")
print("Figure directory:", OUT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from pathlib import Path
import torch

OUT = Path(
    "/content/drive/MyDrive/UA_FloodNet/"
    "v4/final_analysis/figures/paired_errors"
)
OUT.mkdir(parents=True, exist_ok=True)

assert all(k in models for k in ("Control", "Light"))
assert len(test_files) == 70

selected = [test_files[i] for i in (0, 34, 69)]

error_cmap = ListedColormap([
    "#f1f1f1",  # True negative
    "#23867c",  # True positive
    "#e89d13",  # False positive
    "#c83e63"   # False negative
])
error_norm = BoundaryNorm(
    [-0.5, 0.5, 1.5, 2.5, 3.5], 4
)

mask_cmap = ListedColormap(["#f1f1f1", "#2474a6"])

# 0 = agreement, 1 = control only, 2 = light only
difference_cmap = ListedColormap([
    "#f1f1f1", "#d95f02", "#7570b3"
])

legend = [
    Patch(facecolor="#f1f1f1", edgecolor="#888",
          label="Correct non-flood"),
    Patch(facecolor="#23867c", label="True positive flood"),
    Patch(facecolor="#e89d13", label="False positive"),
    Patch(facecolor="#c83e63", label="False negative")
]

for path in selected:
    with np.load(path) as data:
        sample = {
            key: torch.tensor(data[key]).float()
            .unsqueeze(0).to(device)
            for key in (
                "sar_pre", "sar_post",
                "ms_pre", "ms_post", "mask"
            )
        }

    gt = sample["mask"][0, 0].cpu().numpy() >= 0.5

    for dx, label in ((0, "clean"), (4, "shift4")):
        optical_pre = translate_zero(sample["ms_pre"], dx, 0)
        optical_post = translate_zero(sample["ms_post"], dx, 0)

        predictions = {}

        with torch.inference_mode():
            for name in ("Control", "Light"):
                logits = models[name](
                    sample["sar_pre"],
                    sample["sar_post"],
                    optical_pre,
                    optical_post
                )["logits"]

                predictions[name] = (
                    logits.sigmoid()[0, 0].cpu().numpy() >= 0.5
                )

        control_error = make_error_map(predictions["Control"], gt)
        light_error = make_error_map(predictions["Light"], gt)

        disagreement = np.zeros(gt.shape, dtype=np.uint8)
        disagreement[
            predictions["Control"] & ~predictions["Light"]
        ] = 1
        disagreement[
            predictions["Light"] & ~predictions["Control"]
        ] = 2

        panels = [
            (gt.astype(int), "a  Reference flood mask",
             mask_cmap, None),
            (predictions["Control"].astype(int),
             "b  Control prediction", mask_cmap, None),
            (predictions["Light"].astype(int),
             "c  Light prediction", mask_cmap, None),
            (control_error, "d  Control errors",
             error_cmap, error_norm),
            (light_error, "e  Light errors",
             error_cmap, error_norm),
            (disagreement, "f  Prediction disagreement",
             difference_cmap, None)
        ]

        fig, axes = plt.subplots(
            2, 3, figsize=(12, 8),
            constrained_layout=True
        )

        for ax, (image, title, cmap, norm) in zip(
            axes.flat, panels
        ):
            kwargs = {"interpolation": "nearest", "cmap": cmap}
            if norm is not None:
                kwargs["norm"] = norm
            else:
                kwargs["vmin"] = 0
                kwargs["vmax"] = 2 if "disagreement" in title else 1

            ax.imshow(image, **kwargs)
            ax.set_title(title, fontsize=11)
            ax.axis("off")

        fig.legend(
            handles=legend,
            loc="outside lower center",
            ncol=4,
            frameon=False
        )

        filename = f"Paired_Errors_{path.stem}_{label}"

        fig.savefig(
            OUT / f"{filename}.png",
            dpi=600,
            bbox_inches="tight"
        )
        fig.savefig(
            OUT / f"{filename}.pdf",
            bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)

        print("Saved:", filename)

print("\nSTEP 78 COMPLETED")
print("Output directory:", OUT)

In [ ]:
from pathlib import Path
import os

print("Drive mount:", os.path.ismount("/content/drive"))
print("MyDrive exists:", Path("/content/drive/MyDrive").exists())

drive_dir = Path("/content/drive")

if drive_dir.exists():
    print("\nDirectory contents:")
    for p in list(drive_dir.iterdir())[:15]:
        print(p.name)

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import sys

print("=" * 65)
print("UA-FLOODNET SAFE DRIVE RECOVERY")
print("=" * 65)

PRIMARY = Path("/content/drive")
ALTERNATIVE = Path("/content/ua_drive_mount")

if os.path.ismount(str(PRIMARY)) and (PRIMARY / "MyDrive").exists():
    MOUNT = PRIMARY
    print("Using existing Google Drive mount.")

elif os.path.ismount(str(ALTERNATIVE)) and (ALTERNATIVE / "MyDrive").exists():
    MOUNT = ALTERNATIVE
    print("Using existing alternative mount.")

else:
    # Use a separate, empty directory instead of disturbing
    # anything already present in /content/drive.
    ALTERNATIVE.mkdir(parents=True, exist_ok=True)

    if any(ALTERNATIVE.iterdir()):
        raise RuntimeError(
            "Alternative mount directory is not empty. "
            "Inspect it before continuing."
        )

    drive.mount(str(ALTERNATIVE))
    MOUNT = ALTERNATIVE

ROOT = MOUNT / "MyDrive/UA_FloodNet"
BACKUP = ROOT / "v4/code"
CODE = Path("/content/UA_FloodNet/ablations_code")

print("\nProject found:", ROOT.exists())
print("Code backup found:", BACKUP.exists())

assert ROOT.exists(), (
    "UA_FloodNet project not found. Verify the connected "
    "Google account and Drive location."
)

CODE.mkdir(parents=True, exist_ok=True)

for filename in ["ablation_model.py", "data.py"]:
    source = BACKUP / filename
    target = CODE / filename

    if source.exists():
        shutil.copy2(source, target)
        print("Restored:", filename)
    elif target.exists():
        print("Already available:", filename)
    else:
        print("MISSING:", filename)

sys.path.insert(0, str(CODE))

if all((CODE / name).exists() for name in
       ["ablation_model.py", "data.py"]):
    from ablation_model import UAFloodNet
    from data import NPZFloodDataset
    print("\nModel imports: SUCCESS")

print("\nCHECKPOINT STATUS")

for name, path in {
    "Control": ROOT / "v4/full/control_seed42_50ep.pt",
    "Light": ROOT / "v4/light/light_seed42_50ep.pt"
}.items():
    print(name, "FOUND" if path.exists() else "MISSING")

print("\nTEST DATA STATUS")
test_dir = Path("/content/UA_FloodNet/dataset/test")
print("Test samples:", len(list(test_dir.glob("*.npz"))))

print("\nActive project root:", ROOT)

In [ ]:
# ============================================================
# STEP 79B — RESTORE OMBRIA DATASET FROM GOOGLE DRIVE
# ============================================================

from pathlib import Path
import zipfile
import numpy as np

# Use the successful alternative Google Drive mount
ROOT = Path("/content/ua_drive_mount/MyDrive/UA_FloodNet")

ARCHIVE = (
    ROOT / "dataset_backup"
    / "OMBRIA_NPZ_candidate_seed42.zip"
)

DATA = Path("/content/UA_FloodNet/dataset")
DATA.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("STEP 79B: RESTORING OMBRIA DATASET")
print("=" * 70)

assert ARCHIVE.exists(), f"Dataset archive missing: {ARCHIVE}"

print("Archive:", ARCHIVE.name)
print("Archive size:", round(ARCHIVE.stat().st_size / 1024**2, 2), "MB")

# Verify and safely extract the frozen dataset.
with zipfile.ZipFile(ARCHIVE, "r") as z:

    bad_file = z.testzip()
    assert bad_file is None, f"Corrupt ZIP member: {bad_file}"

    root = DATA.resolve()

    for member in z.infolist():
        target = (DATA / member.filename).resolve()

        if not target.is_relative_to(root):
            raise ValueError(
                f"Unsafe ZIP entry: {member.filename}"
            )

    z.extractall(DATA)

print("\nDATASET VERIFICATION")
print("-" * 70)

expected = {
    "train": 499,
    "val": 125,
    "test": 70
}

for split, required in expected.items():

    files = sorted((DATA / split).glob("*.npz"))
    count = len(files)

    print(f"{split}: {count}/{required} samples")

    assert count == required, (
        f"Incorrect {split} sample count: {count}"
    )

# Check one official test sample.
sample_file = sorted((DATA / "test").glob("*.npz"))[0]

with np.load(sample_file) as sample:
    print("\nSAMPLE:", sample_file.name)

    for key in [
        "sar_pre", "sar_post",
        "ms_pre", "ms_post", "mask"
    ]:
        print(key, sample[key].shape)

print("\nSUCCESS: OMBRIA DATASET RESTORED")
print("Official test samples: 70")
print("Test directory:", DATA / "test")

In [ ]:
# ============================================================
# STEP 79 — FINAL QUALITATIVE FLOOD PREDICTION FIGURES
# UA-FloodNet | OMBRIA Official Held-Out Test
#
# Outputs:
#   6 high-resolution PNG figures (600 dpi)
#   6 vector PDF figures
#   1 CSV of selected-sample metrics
#
# Models: Control and V4-light, seed 42
# Samples: Fixed official test indices [0, 34, 69]
# Conditions: Clean (0,0) and Shifted (+4,0)
# ============================================================

from pathlib import Path
import sys
import shutil
import json

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# ============================================================
# 1. PROJECT DIRECTORIES — CORRECTED DRIVE PATH
# ============================================================

DRIVE = Path(
    "/content/ua_drive_mount/MyDrive/UA_FloodNet"
)

CODE = Path(
    "/content/UA_FloodNet/ablations_code"
)

TEST = Path(
    "/content/UA_FloodNet/dataset/test"
)

OUT = (
    DRIVE
    / "v4"
    / "final_analysis"
    / "figures"
    / "paired_errors"
)

OUT.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("STEP 79: FINAL UA-FLOODNET PAIRED ERROR MAPS")
print("=" * 72)

assert DRIVE.exists(), (
    f"Google Drive project not found: {DRIVE}"
)

print("Project root:", DRIVE)
print("Output directory:", OUT)

# ============================================================
# 2. RESTORE MODEL SOURCE CODE IF NEEDED
# ============================================================

CODE.mkdir(parents=True, exist_ok=True)

for filename in [
    "ablation_model.py",
    "data.py"
]:

    source = DRIVE / "v4/code" / filename
    destination = CODE / filename

    if not destination.exists():

        assert source.exists(), (
            f"Missing model source file: {source}"
        )

        shutil.copy2(source, destination)

    print("Source available:", filename)

sys.path.insert(0, str(CODE))

from ablation_model import UAFloodNet

# ============================================================
# 3. VERIFY COMPUTING DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nComputing device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 4. LOAD THE TWO FROZEN SEED-42 MODELS
# ============================================================

checkpoint_paths = {
    "Control": (
        DRIVE
        / "v4/full/control_seed42_50ep.pt"
    ),

    "Light": (
        DRIVE
        / "v4/light/light_seed42_50ep.pt"
    )
}

models = {}

for name, checkpoint_path in checkpoint_paths.items():

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: {checkpoint_path}"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    assert checkpoint["seed"] == 42, (
        f"Unexpected seed in {name} checkpoint"
    )

    model = UAFloodNet(
        width=checkpoint["width"],
        use_alignment=checkpoint["use_alignment"],
        use_reliability=checkpoint["use_reliability"]
    ).to(device)

    model.load_state_dict(checkpoint["model"])

    model.eval()

    models[name] = model

    print(
        f"{name} model loaded successfully | "
        f"Seed: {checkpoint['seed']} | "
        f"Best epoch: {checkpoint['epoch']}"
    )

# ============================================================
# 5. VERIFY ORIGINAL OFFICIAL TEST DATA
# ============================================================

assert TEST.exists(), (
    "Official test directory missing. "
    "Restore the frozen OMBRIA dataset archive."
)

test_files = sorted(TEST.glob("*.npz"))

assert len(test_files) == 70, (
    "Expected 70 official test samples, "
    f"but found {len(test_files)}."
)

# Selection is fixed and independent of model performance.
selected_indices = [0, 34, 69]

selected_files = [
    test_files[i] for i in selected_indices
]

print("\nOFFICIAL TEST SAMPLE VERIFICATION")
print("-" * 72)

print("Total test samples:", len(test_files))

for path in selected_files:
    print("Selected:", path.name)

# ============================================================
# 6. HELPER FUNCTIONS
# ============================================================

def translate_zero(x, dx=0, dy=0):
    """
    Apply integer spatial translation with zero padding.

    Input tensor:
        [batch, channels, height, width]

    Positive dx shifts imagery to the right.
    Positive dy shifts imagery downward.
    """

    _, _, height, width = x.shape

    shifted = torch.zeros_like(x)

    y0 = max(0, dy)
    y1 = min(height, height + dy)

    x0 = max(0, dx)
    x1 = min(width, width + dx)

    if y1 > y0 and x1 > x0:

        shifted[
            :, :, y0:y1, x0:x1
        ] = x[
            :,
            :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]

    return shifted


def make_error_map(prediction, reference):
    """
    Pixel classification:
        0 = True negative
        1 = True positive
        2 = False positive
        3 = False negative
    """

    result = np.zeros(
        reference.shape,
        dtype=np.uint8
    )

    result[
        prediction & reference
    ] = 1

    result[
        prediction & ~reference
    ] = 2

    result[
        ~prediction & reference
    ] = 3

    return result


def make_disagreement_map(control, light):
    """
    0 = Both predictions agree
    1 = Control predicts flood; light does not
    2 = Light predicts flood; control does not
    """

    result = np.zeros(
        control.shape,
        dtype=np.uint8
    )

    result[
        control & ~light
    ] = 1

    result[
        light & ~control
    ] = 2

    return result


def calculate_metrics(prediction, reference):
    """
    Full-patch IoU, F1, precision and recall.
    These are illustrative patch metrics, not
    the official pooled interior-crop test scores.
    """

    tp = int(np.logical_and(
        prediction, reference
    ).sum())

    fp = int(np.logical_and(
        prediction, ~reference
    ).sum())

    fn = int(np.logical_and(
        ~prediction, reference
    ).sum())

    iou = tp / max(tp + fp + fn, 1)

    f1 = (
        2 * tp
        / max(2 * tp + fp + fn, 1)
    )

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    return {
        "IoU": iou,
        "F1": f1,
        "precision": precision,
        "recall": recall,
        "TP": tp,
        "FP": fp,
        "FN": fn
    }

# ============================================================
# 7. SCIENTIFIC COLOUR PALETTES
# ============================================================

# Flood masks
mask_cmap = ListedColormap([
    "#F1F1F1",  # Background
    "#2474A6"   # Flood
])

# Pixel classification errors
error_cmap = ListedColormap([
    "#F1F1F1",  # True negative
    "#23867C",  # True positive
    "#E89D13",  # False positive
    "#C83E63"   # False negative
])

error_norm = BoundaryNorm(
    [-0.5, 0.5, 1.5, 2.5, 3.5],
    error_cmap.N
)

# Direct prediction disagreement
disagreement_cmap = ListedColormap([
    "#F1F1F1",  # Agreement
    "#D95F02",  # Control-only flood
    "#7570B3"   # Light-only flood
])

disagreement_norm = BoundaryNorm(
    [-0.5, 0.5, 1.5, 2.5],
    disagreement_cmap.N
)

# ============================================================
# 8. COMPLETE FIGURE LEGEND
# ============================================================

legend_handles = [
    Patch(
        facecolor="#F1F1F1",
        edgecolor="#888888",
        label="Correct non-flood / prediction agreement"
    ),

    Patch(
        facecolor="#23867C",
        label="True positive flood"
    ),

    Patch(
        facecolor="#E89D13",
        label="False positive"
    ),

    Patch(
        facecolor="#C83E63",
        label="False negative"
    ),

    Patch(
        facecolor="#D95F02",
        label="Control-only flood"
    ),

    Patch(
        facecolor="#7570B3",
        label="Light-only flood"
    )
]

# ============================================================
# 9. MATPLOTLIB PUBLICATION SETTINGS
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 600
})

records = []

# ============================================================
# 10. GENERATE ALL SIX PAIRED ERROR FIGURES
# ============================================================

conditions = [
    (0, 0, "clean"),
    (4, 0, "shift4")
]

for sample_path in selected_files:

    print("\n" + "-" * 72)
    print("PROCESSING SAMPLE:", sample_path.name)

    # --------------------------------------------------------
    # Read original NPZ sample
    # --------------------------------------------------------

    with np.load(sample_path, allow_pickle=False) as data:

        required_keys = [
            "sar_pre",
            "sar_post",
            "ms_pre",
            "ms_post",
            "mask"
        ]

        for key in required_keys:
            assert key in data.files, (
                f"Missing {key} in {sample_path.name}"
            )

        sample = {
            key: torch.as_tensor(
                data[key],
                dtype=torch.float32,
                device=device
            ).unsqueeze(0)

            for key in required_keys
        }

    ground_truth = (
        sample["mask"][0, 0]
        .detach()
        .cpu()
        .numpy() >= 0.5
    )

    # --------------------------------------------------------
    # Evaluate clean and shifted optical inputs
    # --------------------------------------------------------

    for dx, dy, condition in conditions:

        optical_pre = translate_zero(
            sample["ms_pre"],
            dx,
            dy
        )

        optical_post = translate_zero(
            sample["ms_post"],
            dx,
            dy
        )

        predictions = {}

        # ----------------------------------------------------
        # Perform inference
        # ----------------------------------------------------

        with torch.inference_mode():

            for name, model in models.items():

                output = model(
                    sample["sar_pre"],
                    sample["sar_post"],
                    optical_pre,
                    optical_post
                )

                probabilities = (
                    output["logits"]
                    .sigmoid()[0, 0]
                    .detach()
                    .cpu()
                    .numpy()
                )

                predictions[name] = (
                    probabilities >= 0.5
                )

        control_prediction = predictions["Control"]
        light_prediction = predictions["Light"]

        # ----------------------------------------------------
        # Calculate spatial error maps
        # ----------------------------------------------------

        control_errors = make_error_map(
            control_prediction,
            ground_truth
        )

        light_errors = make_error_map(
            light_prediction,
            ground_truth
        )

        disagreement = make_disagreement_map(
            control_prediction,
            light_prediction
        )

        # ----------------------------------------------------
        # Calculate full-patch metrics
        # ----------------------------------------------------

        control_metrics = calculate_metrics(
            control_prediction,
            ground_truth
        )

        light_metrics = calculate_metrics(
            light_prediction,
            ground_truth
        )

        for name, metrics in [
            ("Control", control_metrics),
            ("Light", light_metrics)
        ]:

            records.append({
                "sample_id": sample_path.stem,
                "condition": condition,
                "dx": dx,
                "dy": dy,
                "model": name,
                **metrics
            })

        # ----------------------------------------------------
        # Create six-panel publication figure
        # ----------------------------------------------------

        fig, axes = plt.subplots(
            2,
            3,
            figsize=(13, 9),
            constrained_layout=True
        )

        panels = [
            (
                ground_truth.astype(np.uint8),
                "(a) Reference flood mask",
                mask_cmap,
                None
            ),

            (
                control_prediction.astype(np.uint8),
                "(b) Control prediction",
                mask_cmap,
                None
            ),

            (
                light_prediction.astype(np.uint8),
                "(c) Light-augmentation prediction",
                mask_cmap,
                None
            ),

            (
                control_errors,
                "(d) Control classification errors",
                error_cmap,
                error_norm
            ),

            (
                light_errors,
                "(e) Light-augmentation classification errors",
                error_cmap,
                error_norm
            ),

            (
                disagreement,
                "(f) Prediction disagreement",
                disagreement_cmap,
                disagreement_norm
            )
        ]

        for ax, (image, title, cmap, norm) in zip(
            axes.flat,
            panels
        ):

            if norm is None:

                ax.imshow(
                    image,
                    cmap=cmap,
                    vmin=0,
                    vmax=1,
                    interpolation="nearest"
                )

            else:

                ax.imshow(
                    image,
                    cmap=cmap,
                    norm=norm,
                    interpolation="nearest"
                )

            ax.set_title(
                title,
                fontsize=11,
                pad=8
            )

            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_visible(False)

        # ----------------------------------------------------
        # Correct title
        # ----------------------------------------------------

        fig.suptitle(
            f"OMBRIA Sample {sample_path.stem} | "
            f"Optical Displacement ({dx:+d}, {dy:+d}) Pixels",
            fontsize=13,
            fontweight="bold"
        )

        # ----------------------------------------------------
        # Correct and complete legend
        # ----------------------------------------------------

        fig.legend(
            handles=legend_handles,
            loc="outside lower center",
            ncol=3,
            frameon=False,
            fontsize=9
        )

        # ----------------------------------------------------
        # Save PNG and PDF
        # ----------------------------------------------------

        filename = (
            f"UA_FloodNet_PairedErrors_"
            f"{sample_path.stem}_{condition}"
        )

        png_path = OUT / f"{filename}.png"
        pdf_path = OUT / f"{filename}.pdf"

        fig.savefig(
            png_path,
            dpi=600,
            bbox_inches="tight",
            facecolor="white"
        )

        fig.savefig(
            pdf_path,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()
        plt.close(fig)

        print(
            f"Saved: {filename}\n"
            f"  Control IoU: {control_metrics['IoU']:.5f}\n"
            f"  Light IoU:   {light_metrics['IoU']:.5f}",
            flush=True
        )

# ============================================================
# 11. EXPORT QUALITATIVE SAMPLE METRICS
# ============================================================

results = pd.DataFrame(records)

assert len(results) == 12, (
    f"Expected 12 model/sample/condition records; "
    f"found {len(results)}"
)

csv_path = OUT / "step79_qualitative_sample_metrics.csv"

results.to_csv(
    csv_path,
    index=False
)

# ============================================================
# 12. SAVE FIGURE METADATA
# ============================================================

metadata = {
    "study": "UA-FloodNet",
    "dataset": "OMBRIA",
    "partition": "official held-out test",
    "event_independence": "not verified",
    "model_seed": 42,
    "selected_test_indices": selected_indices,
    "selected_test_samples": [
        path.name for path in selected_files
    ],
    "selection_method": (
        "Fixed positions in sorted test filenames; "
        "independent of prediction performance"
    ),
    "optical_shift_conditions": [
        {
            "dx": dx,
            "dy": dy,
            "condition": label
        }
        for dx, dy, label in conditions
    ],
    "threshold": 0.5,
    "figure_metrics": "full-patch metrics",
    "official_test_metrics": (
        "separate pooled evaluation with 20-pixel margin"
    ),
    "model_checkpoints": {
        name: str(path)
        for name, path in checkpoint_paths.items()
    }
}

metadata_path = OUT / "step79_figure_metadata.json"

metadata_path.write_text(
    json.dumps(metadata, indent=2)
)

# ============================================================
# 13. FINAL VERIFICATION
# ============================================================

png_files = sorted(
    OUT.glob("UA_FloodNet_PairedErrors_*.png")
)

pdf_files = sorted(
    OUT.glob("UA_FloodNet_PairedErrors_*.pdf")
)

assert len(png_files) == 6, (
    f"Expected 6 PNG files, found {len(png_files)}"
)

assert len(pdf_files) == 6, (
    f"Expected 6 PDF files, found {len(pdf_files)}"
)

print("\n" + "=" * 72)
print("STEP 79 COMPLETED SUCCESSFULLY")
print("=" * 72)

print("Generated PNG figures:", len(png_files))
print("Generated PDF figures:", len(pdf_files))
print("CSV metrics saved:", csv_path)
print("Figure metadata saved:", metadata_path)

print("\nOUTPUT DIRECTORY:")
print(OUT)

print("\nNEXT STAGE: STEP 80 — UA-FLOODNET ARCHITECTURE")

In [ ]:
# ============================================================
# STEP 80A — UA-FLOODNET ARCHITECTURE INSPECTION
# ============================================================

from pathlib import Path
import sys
import inspect
import json
import torch

ROOT = Path("/content/ua_drive_mount/MyDrive/UA_FloodNet")
CODE = Path("/content/UA_FloodNet/ablations_code")
OUT = ROOT / "v4/final_analysis/architecture"

OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))

from ablation_model import UAFloodNet
import ablation_model

print("=" * 75)
print("STEP 80A: VERIFIED UA-FLOODNET ARCHITECTURE")
print("=" * 75)

# 1. Source classes
print("\n1. DEFINED NETWORK CLASSES")

import ast

source_path = CODE / "ablation_model.py"
source = source_path.read_text()
tree = ast.parse(source)

for node in tree.body:
    if isinstance(node, ast.ClassDef):
        print(f"Class: {node.name} | Source line: {node.lineno}")

# 2. Full model definition
print("\n2. UAFloodNet IMPLEMENTATION")
print(inspect.getsource(UAFloodNet))

# 3. All model modules and their children
model = UAFloodNet(
    width=24,
    use_alignment=True,
    use_reliability=True
)

print("\n3. MODULE HIERARCHY")
print(model)

# 4. Parameter counts
total = sum(p.numel() for p in model.parameters())
trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("\n4. MODEL PARAMETERS")
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

# 5. Top-level module inventory
print("\n5. TOP-LEVEL MODULE INVENTORY")

module_inventory = []

for name, module in model.named_children():
    count = sum(p.numel() for p in module.parameters())

    record = {
        "module": name,
        "type": type(module).__name__,
        "parameters": count
    }

    module_inventory.append(record)

    print(
        f"{name:25s} "
        f"{type(module).__name__:25s} "
        f"{count:,} parameters"
    )

# 6. Verify outputs using synthetic inputs
print("\n6. FORWARD-PASS VERIFICATION")

model.eval()

with torch.inference_mode():
    sar_pre = torch.zeros(1, 1, 256, 256)
    sar_post = torch.zeros(1, 1, 256, 256)
    ms_pre = torch.zeros(1, 3, 256, 256)
    ms_post = torch.zeros(1, 3, 256, 256)

    output = model(
        sar_pre, sar_post,
        ms_pre, ms_post
    )

    for name, value in output.items():
        if torch.is_tensor(value):
            print(name, tuple(value.shape))
        else:
            print(name, type(value).__name__)

# 7. Save verified information
report = {
    "architecture": "UAFloodNet",
    "model_width": 24,
    "use_alignment": True,
    "use_reliability": True,
    "total_parameters": total,
    "trainable_parameters": trainable,
    "modules": module_inventory,
    "source": str(source_path)
}

(OUT / "step80_architecture_inventory.json").write_text(
    json.dumps(report, indent=2)
)

(OUT / "step80_model_source.txt").write_text(
    inspect.getsource(UAFloodNet)
)

(OUT / "step80_module_hierarchy.txt").write_text(
    str(model)
)

print("\n" + "=" * 75)
print("STEP 80A COMPLETED")
print("Architecture inspection saved to:", OUT)

In [ ]:
# ============================================================
# STEP 80B — VERIFY INTERNAL NETWORK OPERATIONS
# ============================================================

import sys
import inspect
from pathlib import Path

CODE = Path("/content/UA_FloodNet/ablations_code")
sys.path.insert(0, str(CODE))

from ablation_model import (
    Encoder,
    AlignFusion,
    UAFloodNet,
    training_loss
)

print("=" * 75)
print("STEP 80B: INTERNAL ARCHITECTURE VERIFICATION")
print("=" * 75)

components = {
    "ENCODER": Encoder,
    "ALIGNMENT AND RELIABILITY FUSION": AlignFusion,
    "TRAINING LOSS": training_loss
}

for name, component in components.items():
    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)
    print(inspect.getsource(component))

print("\nSTEP 80B COMPLETED")

In [ ]:

# ============================================================
# STEP 80C — VERIFIED UA-FLOODNET ARCHITECTURE FIGURE
# Output: SVG, PDF and 600-dpi PNG
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path("/content/ua_drive_mount/MyDrive/UA_FloodNet")
OUT = ROOT / "v4/final_analysis/architecture"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "savefig.dpi": 600,
})

COL = {
    "sar": "#DDEAF5",
    "opt": "#DDF1E6",
    "fusion": "#F0E5FA",
    "change": "#FFF0D6",
    "decoder": "#E5EFFA",
    "output": "#CBECDC",
    "aux": "#F2F3F5",
    "edge": "#46566C",
    "text": "#152438",
}

fig, ax = plt.subplots(figsize=(17, 12.5))
fig.patch.set_facecolor("white")
ax.set_xlim(0, 17)
ax.set_ylim(0, 12.5)
ax.axis("off")

def node(x, y, w, h, label, color, fs=9.5):
    patch = FancyBboxPatch(
        (x-w/2, y-h/2), w, h,
        boxstyle="round,pad=0.025,rounding_size=0.12",
        facecolor=color, edgecolor="#52657C",
        linewidth=1.2
    )
    ax.add_patch(patch)
    ax.text(
        x, y, label,
        ha="center", va="center",
        fontsize=fs, color=COL["text"],
        linespacing=1.35
    )

def arrow(x1, y1, x2, y2, dashed=False):
    ax.add_patch(FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="-|>", mutation_scale=12,
        linewidth=1.3, color=COL["edge"],
        linestyle="--" if dashed else "-"
    ))

ax.text(
    8.5, 12.13,
    "UA-FloodNet: Multitemporal SAR–Optical Flood Segmentation",
    ha="center", va="center",
    fontsize=18, fontweight="bold", color=COL["text"]
)

ax.text(
    8.5, 11.72,
    "Verified full model | Width = 24 | 1,809,577 trainable parameters",
    ha="center", fontsize=10.5, color="#52657C"
)

# ------------------------------------------------------------
# FOUR INPUTS
# ------------------------------------------------------------
inputs = [
    (1.5, "SAR before\n1 × 256 × 256", COL["sar"]),
    (5.5, "Optical before\n3 × 256 × 256", COL["opt"]),
    (11.5, "SAR after\n1 × 256 × 256", COL["sar"]),
    (15.5, "Optical after\n3 × 256 × 256", COL["opt"]),
]

for x, label, color in inputs:
    node(x, 10.85, 2.65, 0.75, label, color)

encoders = [
    (1.5, "SAR encoder\n(shared across dates)", COL["sar"]),
    (5.5, "Optical encoder\n(shared across dates)", COL["opt"]),
    (11.5, "SAR encoder\n(shared across dates)", COL["sar"]),
    (15.5, "Optical encoder\n(shared across dates)", COL["opt"]),
]

for x, label, color in encoders:
    node(x, 9.55, 2.85, 0.78, label, color)
    arrow(x, 10.46, x, 9.96)

ax.text(
    8.5, 9.55,
    "Three scales per encoder\n"
    "256² × 24  |  128² × 48  |  64² × 96",
    ha="center", va="center", fontsize=9.2,
    color="#52657C"
)

# ------------------------------------------------------------
# SAME-DATE MULTISCALE FUSION
# ------------------------------------------------------------
node(
    3.5, 8.0, 5.45, 1.02,
    "PRE-FLOOD: Three-scale AlignFusion\n"
    "Optical → SAR sampling + reliability weighting",
    COL["fusion"], 10
)
node(
    13.5, 8.0, 5.45, 1.02,
    "POST-FLOOD: Three-scale AlignFusion\n"
    "Optical → SAR sampling + reliability weighting",
    COL["fusion"], 10
)

for sx, tx in [(1.5, 2.3), (5.5, 4.7),
               (11.5, 12.3), (15.5, 14.7)]:
    arrow(sx, 9.13, tx, 8.53)

# ------------------------------------------------------------
# FUSED MULTISCALE FEATURES
# ------------------------------------------------------------
node(
    3.5, 6.72, 5.1, 0.73,
    "Pre-flood fused features F₀¹, F₀², F₀³",
    COL["fusion"]
)
node(
    13.5, 6.72, 5.1, 0.73,
    "Post-flood fused features F₁¹, F₁², F₁³",
    COL["fusion"]
)

arrow(3.5, 7.47, 3.5, 7.1)
arrow(13.5, 7.47, 13.5, 7.1)

node(
    8.5, 5.82, 8.0, 0.82,
    "Temporal change extraction at all three scales\n"
    "Dˡ = |F₀ˡ − F₁ˡ|; no cross-time warping",
    COL["change"], 10.4
)
arrow(3.5, 6.33, 6.9, 6.24)
arrow(13.5, 6.33, 10.1, 6.24)

# ------------------------------------------------------------
# COARSE-TO-FINE TEMPORAL DECODER
# ------------------------------------------------------------
node(
    8.5, 4.58, 9.6, 0.79,
    "Deep temporal block (64²; 288 → 96 channels)\n"
    "Concatenate F₀³, F₁³ and D³",
    COL["decoder"]
)
arrow(8.5, 5.4, 8.5, 4.99)

node(
    8.5, 3.52, 9.6, 0.76,
    "Upsample → decode2 (128²; 240 → 48 channels)\n"
    "Skip: F₀², F₁² and D²",
    COL["decoder"]
)
arrow(8.5, 4.16, 8.5, 3.92)

node(
    8.5, 2.48, 9.6, 0.76,
    "Upsample → decode1 (256²; 120 → 24 channels)\n"
    "Skip: F₀¹, F₁¹ and D¹",
    COL["decoder"]
)
arrow(8.5, 3.12, 8.5, 2.88)

node(
    8.5, 1.53, 8.0, 0.65,
    "1 × 1 convolution → flood logits (1 × 256 × 256)",
    COL["output"], 10
)
arrow(8.5, 2.09, 8.5, 1.88)

# ------------------------------------------------------------
# TRAINING-ONLY AUXILIARY BRANCH
# ------------------------------------------------------------
node(
    2.2, 2.75, 3.65, 1.1,
    "TRAINING-ONLY AUXILIARY HEADS\n"
    "64-D SAR/optical projections\n"
    "Cross-modal reconstructions",
    COL["aux"], 8.5
)

arrow(2.1, 6.3, 2.1, 3.35, dashed=True)

node(
    14.7, 2.65, 3.55, 1.25,
    "Training objective\n"
    "BCE + Dice\n"
    "+ 0.1 InfoNCE\n"
    "+ 0.1 Reconstruction\n"
    "+ 0.01 Flow smoothness",
    COL["aux"], 8.5
)
arrow(13.5, 7.47, 15.0, 3.33, dashed=True)

ax.text(
    8.5, 0.95,
    "Inference: sigmoid(logits) ≥ 0.5 gives the binary flood mask",
    ha="center", fontsize=10, color="#285A70"
)

ax.text(
    8.5, 0.44,
    "Dashed connections indicate training-related branches. "
    "Alignment is between modalities at corresponding dates, not between acquisition dates.",
    ha="center", fontsize=8.3, color="#52657C"
)

fig.savefig(
    OUT / "UA_FloodNet_Verified_Architecture.svg",
    bbox_inches="tight", facecolor="white"
)
fig.savefig(
    OUT / "UA_FloodNet_Verified_Architecture.pdf",
    bbox_inches="tight", facecolor="white"
)
fig.savefig(
    OUT / "UA_FloodNet_Verified_Architecture.png",
    dpi=600, bbox_inches="tight", facecolor="white"
)

plt.show()
plt.close(fig)

print("\nSTEP 80C COMPLETED")
print("Architecture files:", OUT)
for name in ("svg", "pdf", "png"):
    path = OUT / f"UA_FloodNet_Verified_Architecture.{name}"
    print(name.upper(), "FOUND" if path.exists() else "MISSING")



In [ ]:
%run /content/UA_FloodNet_Manuscript_Figures_Tables_FINAL.py

In [ ]:
# -*- coding: utf-8 -*-
"""
UA-FloodNet: Publication-Quality Manuscript Figures and Tables

Outputs
-------
1. Three-panel UA-FloodNet architecture
2. Seed-wise IoU degradation
3. Paired effects with 95% confidence intervals
4. Validation and official-test displacement profiles
5. Absolute clean versus shifted IoU
6. Manuscript tables
7. Existing qualitative prediction figures
8. Figure captions and reproducibility manifest

All numerical results come from previously saved experiments.
No models are retrained and no predictions are modified.
"""

from pathlib import Path
import os
import json
import shutil
import hashlib

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from matplotlib.patches import (
    Polygon,
    FancyArrowPatch,
    FancyBboxPatch,
    Patch
)

from matplotlib.lines import Line2D


# ============================================================
# SECTION 1 — PROJECT DIRECTORIES AND INPUT VERIFICATION
# ============================================================

MOUNTS = [
    Path("/content/ua_drive_mount/MyDrive/UA_FloodNet"),
    Path("/content/drive/MyDrive/UA_FloodNet")
]

ROOT = next(
    (p for p in MOUNTS if p.exists()),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Google Drive project not found. "
        "Mount Google Drive before running this script."
    )

BASE = ROOT / "v4"
FINAL = BASE / "final_analysis"

OUT = FINAL / "manuscript_package"
FIG = OUT / "figures"
TAB = OUT / "tables"

for folder in (OUT, FIG, TAB):
    folder.mkdir(parents=True, exist_ok=True)

VALL = (
    BASE
    / "robustness/multiseed/step71_all_shift_results.csv"
)

VALS = (
    BASE
    / "robustness/multiseed/step71_seed_summary.csv"
)

TESTL = (
    BASE
    / "official_test/step74_frozen_test_results.csv"
)

TESTS = (
    BASE
    / "official_test/step74_frozen_test_summary.csv"
)

STATS = (
    FINAL
    / "step75_validation_test_statistics.csv"
)

needed = [
    VALL,
    VALS,
    TESTL,
    TESTS,
    STATS
]

missing = [
    str(path)
    for path in needed
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required analysis files are missing:\n"
        + "\n".join(missing)
    )

val_long = pd.read_csv(VALL)
val_sum = pd.read_csv(VALS)

test_long = pd.read_csv(TESTL)
test_sum = pd.read_csv(TESTS)

statistics = pd.read_csv(STATS)

for label, df in [
    ("Validation shifts", val_long),
    ("Official test shifts", test_long)
]:
    assert len(df) == 104, (
        f"{label}: expected 104 records, found {len(df)}"
    )

    assert not df.duplicated(
        ["seed", "variant", "dx", "dy"]
    ).any()

    assert set(df.seed) == {7, 17, 37, 42}
    assert set(df.variant) == {"control", "light"}

for label, df in [
    ("Validation summary", val_sum),
    ("Official test summary", test_sum)
]:
    assert len(df) == 8, label

    assert not df.duplicated(
        ["seed", "variant"]
    ).any()

assert len(statistics) == 8

print("=" * 75)
print("UA-FLOODNET MANUSCRIPT FIGURE AND TABLE GENERATOR")
print("=" * 75)

print("Project root:", ROOT)
print("Output directory:", OUT)
print("Input analysis files verified.")


# ============================================================
# SECTION 2 — HIGHEST-QUALITY EXPORT CONFIGURATION
# ============================================================

# Change this to 1200 for ultra-high-resolution raster figures.
# SVG and PDF remain resolution-independent.
DPI = int(os.environ.get("UA_FLOODNET_DPI", "1200"))

if DPI not in (300, 600, 900, 1200):
    raise ValueError(
        "DPI must be 300, 600, 900, or 1200."
    )

# Safety limit to reduce Colab out-of-memory failures.
MAX_EDGE_PX = int(
    os.environ.get("UA_FLOODNET_MAX_EDGE_PX", "18000")
)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COL = {
    "navy": "#18324E",
    "blue": "#21B6E8",
    "sar": "#18B7E2",
    "opt": "#8F50E8",
    "enc": "#F47729",
    "fusion": "#9254E9",
    "dec": "#05BA75",
    "out": "#F6C537",
    "control": "#315D80",
    "light": "#2A9D8F",
    "line": "#318DEA",
    "grey": "#586777",
    "box": "#F7FAFD",
    "red": "#C84953",
}

outputs = []


def export(fig, stem, pdf=True):
    """
    Export SVG, PDF, PNG and lossless TIFF.
    Preserve vector quality independently of raster DPI.
    """

    svg = FIG / (stem + ".svg")

    fig.savefig(
        svg,
        format="svg",
        bbox_inches="tight",
        pad_inches=0.13,
        facecolor="white"
    )

    outputs.append(svg)

    if pdf:
        path = FIG / (stem + ".pdf")

        fig.savefig(
            path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=0.13,
            facecolor="white"
        )

        outputs.append(path)

    width, height = fig.get_size_inches()

    raster_dpi = min(
        DPI,
        int(MAX_EDGE_PX / max(width, height))
    )

    for ext, kwargs in [
        ("png", {}),
        (
            "tiff",
            {"pil_kwargs": {"compression": "tiff_lzw"}}
        )
    ]:
        path = (
            FIG
            / (stem + f"_{raster_dpi}dpi." + ext)
        )

        fig.savefig(
            path,
            dpi=raster_dpi,
            format=ext,
            bbox_inches="tight",
            pad_inches=0.13,
            facecolor="white",
            **kwargs
        )

        outputs.append(path)

    print(
        "Exported:",
        stem,
        "| Raster DPI:",
        raster_dpi,
        flush=True
    )

    plt.close(fig)


# ============================================================
# SECTION 3 — ARCHITECTURE DRAWING FUNCTIONS
# ============================================================

def box(
    ax,
    x,
    y,
    w,
    h,
    text,
    fc,
    fs=8.5,
    border="#42607A",
    bold=False
):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle=(
            "round,pad=0.02,rounding_size=0.12"
        ),
        facecolor=fc,
        edgecolor=border,
        linewidth=1.25
    )

    ax.add_patch(patch)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fs,
        color=COL["navy"],
        linespacing=1.25,
        weight="bold" if bold else "normal"
    )


def arrow(
    ax,
    a,
    b,
    color=None,
    style="-|>",
    lw=1.6,
    rad=0
):
    ax.add_patch(
        FancyArrowPatch(
            a,
            b,
            arrowstyle=style,
            mutation_scale=11,
            linewidth=lw,
            color=color or COL["line"],
            connectionstyle=f"arc3,rad={rad}"
        )
    )


def cube(
    ax,
    x,
    y,
    w,
    h,
    fill,
    label,
    sub="",
    dep=0.22,
    fs=8
):
    """
    Editable vector 3D-style cuboid.
    """

    import matplotlib.colors as mcolors

    color = np.array(
        mcolors.to_rgb(fill)
    )

    bright = tuple(
        np.minimum(color * 0.82 + 0.18, 1)
    )

    dark = tuple(color * 0.78)

    # Front
    ax.add_patch(
        Polygon(
            [
                (x, y),
                (x + w, y),
                (x + w, y + h),
                (x, y + h)
            ],
            facecolor=fill,
            edgecolor=COL["navy"],
            linewidth=1.05,
            zorder=3
        )
    )

    # Top
    ax.add_patch(
        Polygon(
            [
                (x, y + h),
                (x + dep, y + h + dep),
                (x + w + dep, y + h + dep),
                (x + w, y + h)
            ],
            facecolor=bright,
            edgecolor=COL["navy"],
            linewidth=1.05,
            zorder=3
        )
    )

    # Right side
    ax.add_patch(
        Polygon(
            [
                (x + w, y),
                (x + w + dep, y + dep),
                (x + w + dep, y + h + dep),
                (x + w, y + h)
            ],
            facecolor=dark,
            edgecolor=COL["navy"],
            linewidth=1.05,
            zorder=3
        )
    )

    ax.text(
        x + w / 2,
        y - 0.26,
        label,
        ha="center",
        va="top",
        fontsize=fs,
        weight="bold",
        color=COL["navy"]
    )

    if sub:
        ax.text(
            x + w / 2,
            y - 0.53,
            sub,
            ha="center",
            va="top",
            fontsize=fs - 1.4,
            color=COL["grey"]
        )


# ============================================================
# SECTION 4 — THREE-PANEL UA-FLOODNET ARCHITECTURE
# ============================================================

fig, axs = plt.subplots(
    3,
    1,
    figsize=(17.5, 12.0),
    gridspec_kw={
        "height_ratios": [1.38, 1.18, 0.96]
    }
)

fig.subplots_adjust(
    left=0.035,
    right=0.985,
    top=0.975,
    bottom=0.035,
    hspace=0.17
)

for ax in axs:
    ax.set_xlim(0, 17.0)
    ax.set_ylim(0, 4.75)
    ax.set_axis_off()

    rect = FancyBboxPatch(
        (0.02, 0.03),
        16.94,
        4.66,
        boxstyle=(
            "round,pad=0.02,rounding_size=0.18"
        ),
        facecolor="white",
        edgecolor="#4B9DF5",
        linewidth=1.3,
        linestyle="--"
    )

    ax.add_patch(rect)


# ------------------------------------------------------------
# PANEL A — MULTITEMPORAL SEGMENTATION ARCHITECTURE
# ------------------------------------------------------------

ax = axs[0]

ax.text(
    0.32,
    4.42,
    "(a) Four-input, three-scale temporal segmentation architecture",
    fontsize=14,
    weight="bold",
    color=COL["navy"]
)

cube(
    ax, 0.52, 2.55, 0.62, 1.10,
    COL["sar"],
    "SAR t0",
    "1 × 256²",
    fs=8.4
)

cube(
    ax, 1.57, 2.55, 0.62, 1.10,
    COL["opt"],
    "Optical t0",
    "3 × 256²",
    fs=8.3
)

cube(
    ax, 0.52, 0.79, 0.62, 1.10,
    COL["sar"],
    "SAR t1",
    "1 × 256²",
    fs=8.4
)

cube(
    ax, 1.57, 0.79, 0.62, 1.10,
    COL["opt"],
    "Optical t1",
    "3 × 256²",
    fs=8.3
)

box(
    ax,
    3.0, 2.56, 2.3, 1.0,
    "SAR + optical encoders\n24 → 48 → 96",
    "#E6F6FD",
    9.0,
    bold=True
)

box(
    ax,
    3.0, 0.80, 2.3, 1.0,
    "SAR + optical encoders\n24 → 48 → 96",
    "#F1E9FF",
    9.0,
    bold=True
)

for sx, sy, ty in [
    (1.36, 3.08, 3.23),
    (2.5, 3.08, 2.91),
    (1.36, 1.34, 1.48),
    (2.5, 1.34, 1.16)
]:
    arrow(
        ax,
        (sx, sy),
        (2.97, ty),
        lw=1.2
    )

ax.text(
    4.15,
    2.35,
    "Separate SAR/optical weights, reused over time",
    ha="center",
    fontsize=8.2,
    color=COL["grey"]
)

box(
    ax,
    5.95, 2.56, 2.42, 1.0,
    "Pre-flood AlignFusion\n3 scales × same-time pairs",
    "#ECDFFF",
    9.0,
    bold=True
)

box(
    ax,
    5.95, 0.80, 2.42, 1.0,
    "Post-flood AlignFusion\n3 scales × same-time pairs",
    "#ECDFFF",
    9.0,
    bold=True
)

arrow(
    ax,
    (5.33, 3.06),
    (5.93, 3.06)
)

arrow(
    ax,
    (5.33, 1.30),
    (5.93, 1.30)
)

box(
    ax,
    9.03, 1.70, 2.55, 1.28,
    "Temporal features\n"
    "F₀, F₁, |F₀ − F₁|\n"
    "at 3 scales",
    "#FFF0D9",
    9.3,
    bold=True
)

arrow(
    ax,
    (8.4, 3.06),
    (9.0, 2.59)
)

arrow(
    ax,
    (8.4, 1.3),
    (9.0, 2.08)
)

box(
    ax,
    12.03, 1.65, 2.15, 1.37,
    "Coarse-to-fine decoder\n"
    "64² → 128² → 256²\n"
    "channels 96 → 48 → 24",
    "#DCF7E9",
    8.9,
    bold=True
)

arrow(
    ax,
    (11.60, 2.34),
    (12.00, 2.34)
)

cube(
    ax,
    14.83, 2.01, 0.54, 0.89,
    COL["out"],
    "Flood logits",
    "1 × 256²",
    fs=8.8
)

arrow(
    ax,
    (14.2, 2.34),
    (14.78, 2.34)
)

ax.text(
    8.6,
    0.27,
    "Three fusers are reused for both dates; "
    "skip concatenations at all three spatial scales.",
    ha="center",
    fontsize=8.3,
    color=COL["grey"]
)


# ------------------------------------------------------------
# PANEL B — ALIGNMENT AND RELIABILITY FUSION
# ------------------------------------------------------------

ax = axs[1]

ax.text(
    0.32,
    4.42,
    "(b) AlignFusion: learned same-date optical alignment "
    "and reliability-weighted fusion",
    fontsize=13.2,
    weight="bold",
    color=COL["navy"]
)

cube(
    ax,
    0.63, 2.10, 0.77, 1.05,
    COL["sar"],
    "SAR features",
    "C × H × W",
    fs=8.7
)

cube(
    ax,
    0.63, 0.58, 0.77, 1.05,
    COL["opt"],
    "Optical features",
    "C × H × W",
    fs=8.7
)

box(
    ax,
    2.27, 2.24, 2.50, 0.88,
    "Concat(SAR, optical)\n"
    "flow_net → tanh × Δmax",
    "#E8F4FF",
    9
)

box(
    ax,
    2.27, 0.69, 2.50, 0.88,
    "grid_sample(optical, grid)\n"
    "border padding",
    "#E8F4FF",
    9
)

arrow(
    ax, (1.55, 2.6), (2.23, 2.60)
)

arrow(
    ax, (1.55, 1.16), (2.23, 1.16)
)

arrow(
    ax, (1.55, 1.42), (2.35, 2.23),
    lw=1.1
)

arrow(
    ax, (3.5, 2.20), (3.5, 1.60)
)

box(
    ax,
    5.25, 1.75, 2.46, 1.20,
    "SAR + aligned optical\n"
    "logvar_net → clamp(−6, 6)\n"
    "softmax(−logvar)",
    "#F1E7FF",
    8.9
)

arrow(
    ax, (4.79, 1.13), (5.23, 2.11)
)

arrow(
    ax, (4.80, 2.65), (5.23, 2.65)
)

box(
    ax,
    8.24, 1.75, 2.55, 1.20,
    "Weighted SAR and optical\n"
    "concat [wSAR·SAR, wMS·MS]\n"
    "C + C channels",
    "#E1FAF0",
    9
)

arrow(
    ax,
    (7.73, 2.34),
    (8.21, 2.34)
)

box(
    ax,
    11.32, 1.81, 2.24, 1.06,
    "mix: 2× Conv3×3\n"
    "BN + ReLU → C channels",
    "#DCF7E9",
    9
)

arrow(
    ax,
    (10.82, 2.34),
    (11.28, 2.34)
)

cube(
    ax,
    14.15, 1.91, 0.69, 0.88,
    COL["fusion"],
    "Fused features",
    "C × H × W",
    fs=8.9
)

arrow(
    ax,
    (13.60, 2.34),
    (14.08, 2.34)
)

ax.text(
    8.5,
    0.30,
    "Levels: C = 24, 48, 96; Δmax = 4, 2, 1 feature-grid pixels. "
    "Learned log variances are uncertainty proxies, "
    "not calibrated probabilities.",
    ha="center",
    fontsize=8.3,
    color=COL["red"]
)


# ------------------------------------------------------------
# PANEL C — CHANGE DECODER AND TRAINING OBJECTIVES
# ------------------------------------------------------------

ax = axs[2]

ax.text(
    0.32,
    4.42,
    "(c) Multiscale change decoding, training-only "
    "regularization and evaluation",
    fontsize=13.3,
    weight="bold",
    color=COL["navy"]
)

cube(
    ax,
    0.62, 2.22, 0.60, 0.93,
    COL["fusion"],
    "Deep fused",
    "64 × 64",
    fs=8.7
)

box(
    ax,
    2.15, 2.20, 2.58, 0.95,
    "Deep temporal block\n"
    "concat(F₀³, F₁³, D³)\n"
    "288 → 96",
    "#FFF0D9",
    8.9
)

arrow(
    ax,
    (1.50, 2.66),
    (2.12, 2.66)
)

box(
    ax,
    5.22, 2.20, 2.18, 0.95,
    "Upsample + decode2\n"
    "240 → 48, at 128²",
    "#DEF6E7",
    9
)

arrow(
    ax,
    (4.76, 2.66),
    (5.19, 2.66)
)

box(
    ax,
    7.90, 2.20, 2.18, 0.95,
    "Upsample + decode1\n"
    "120 → 24, at 256²",
    "#DEF6E7",
    9
)

arrow(
    ax,
    (7.43, 2.66),
    (7.87, 2.66)
)

cube(
    ax,
    10.67, 2.30, 0.54, 0.83,
    COL["out"],
    "Flood logits",
    "256 × 256",
    fs=8.6
)

arrow(
    ax,
    (10.11, 2.66),
    (10.61, 2.66)
)

box(
    ax,
    12.10, 2.17, 3.94, 1.0,
    "Evaluation: sigmoid ≥ 0.5\n"
    "Frozen checkpoints; clean and shifted IoU\n"
    "13 offsets, 4 seeds",
    "#FFF7DF",
    9.0
)

arrow(
    ax,
    (11.55, 2.67),
    (12.07, 2.67)
)

box(
    ax,
    2.04, 0.46, 6.20, 0.92,
    "Training-only auxiliary heads: "
    "64-D SAR/MS projections (InfoNCE)\n"
    "Deep-feature cross-modal 1×1 "
    "reconstructions (MSE)",
    "#F5EBFE",
    8.9
)

box(
    ax,
    9.02, 0.46, 6.86, 0.92,
    "Loss = BCE + Dice + 0.1 InfoNCE "
    "+ 0.1 reconstruction + 0.01 flow smoothness\n"
    "Optical augmentation: 25% chance, "
    "±2 pixels, same shift at both dates",
    "#E8F1FC",
    8.8
)

arrow(
    ax,
    (6.5, 2.18),
    (6.5, 1.41),
    style="-|>",
    lw=1.0
)

fig.suptitle(
    "UA-FloodNet | Verified implementation schematic",
    y=0.995,
    fontsize=17,
    weight="bold",
    color=COL["navy"]
)

export(
    fig,
    "Figure_Architecture_3Panel"
)

print("Architecture diagram completed.")


# ============================================================
# SECTION 5 — PREPARE QUANTITATIVE FIGURE DATA
# ============================================================

for df in (val_long, test_long):
    df["shift_magnitude"] = np.hypot(
        df.dx,
        df.dy
    )


# ============================================================
# SECTION 6 — SEED-WISE IOU DEGRADATION
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.7),
    sharey=True,
    layout="constrained"
)

datasets = [
    ("Candidate validation", val_sum),
    ("Official held-out test", test_sum)
]

for ax, (label, df) in zip(
    axes,
    datasets
):
    pivot = df.pivot(
        index="seed",
        columns="variant",
        values="mean_iou_loss"
    ).sort_index()

    x = np.arange(len(pivot))
    width = 0.34

    ax.bar(
        x - width / 2,
        pivot["control"],
        width,
        label="Control",
        color=COL["control"]
    )

    ax.bar(
        x + width / 2,
        pivot["light"],
        width,
        label="Mild augmentation",
        color=COL["light"]
    )

    ax.set_xticks(
        x,
        [str(i) for i in pivot.index]
    )

    ax.set_xlabel("Training seed")

    ax.set_title(
        label,
        weight="bold"
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

    ax.set_axisbelow(True)

axes[0].set_ylabel(
    "IoU degradation (clean − mean shifted)"
)

axes[0].legend(
    frameon=False,
    loc="upper left"
)

export(
    fig,
    "Figure_Seedwise_IoU_Degradation"
)


# ============================================================
# SECTION 7 — PAIRED EFFECTS WITH 95% CONFIDENCE INTERVALS
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.9),
    layout="constrained"
)

metrics = [
    (
        "mean_iou_loss",
        "IoU degradation (light − control)"
    ),
    (
        "mean_shifted_iou",
        "Mean shifted IoU (light − control)"
    )
]

for ax, (metric, label) in zip(
    axes,
    metrics
):
    subset = (
        statistics[
            statistics.metric == metric
        ]
        .set_index("partition")
        .loc[["validation", "official_test"]]
    )

    means = subset.mean_difference.to_numpy()
    lower = subset.ci95_lower.to_numpy()
    upper = subset.ci95_upper.to_numpy()

    ax.errorbar(
        [0, 1],
        means,
        yerr=np.vstack([
            means - lower,
            upper - means
        ]),
        fmt="o",
        color=COL["control"],
        capsize=7,
        linewidth=1.8
    )

    ax.axhline(
        0,
        color="#777777",
        linestyle="--",
        linewidth=1
    )

    ax.set_xticks(
        [0, 1],
        ["Validation", "Held-out test"]
    )

    ax.set_ylabel(label)

    ax.grid(
        axis="y",
        alpha=0.22
    )

    ax.set_axisbelow(True)

fig.suptitle(
    "Four-seed paired effects and exploratory "
    "95% t confidence intervals",
    weight="bold"
)

export(
    fig,
    "Figure_Paired_Effects_95CI"
)


# ============================================================
# SECTION 8 — MISREGISTRATION SHIFT PROFILES
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14.7, 4.9),
    sharey=True,
    layout="constrained"
)

shift_order = [
    (0, 0),
    (-1, 0),
    (1, 0),
    (-2, 0),
    (2, 0),
    (-4, 0),
    (4, 0),
    (-8, 0),
    (8, 0),
    (0, -2),
    (0, 2),
    (0, -4),
    (0, 4)
]

shift_labels = [
    f"({x:+d},{y:+d})"
    for x, y in shift_order
]

for ax, (name, df) in zip(
    axes,
    [
        ("Candidate validation", val_long),
        ("Official held-out test", test_long)
    ]
):
    for variant, color, label in [
        (
            "control",
            COL["control"],
            "Control"
        ),
        (
            "light",
            COL["light"],
            "Mild augmentation"
        )
    ]:
        group = (
            df[df.variant == variant]
            .groupby(["dx", "dy"])
            .IoU.mean()
        )

        values = [
            float(group.loc[(dx, dy)])
            for dx, dy in shift_order
        ]

        ax.plot(
            np.arange(13),
            values,
            "o-",
            markersize=4,
            linewidth=1.8,
            label=label,
            color=color
        )

    ax.set_xticks(
        np.arange(13),
        shift_labels,
        rotation=55,
        ha="right",
        fontsize=7.5
    )

    ax.set_title(
        name,
        weight="bold"
    )

    ax.set_xlabel(
        "Optical displacement (dx, dy), pixels"
    )

    ax.grid(alpha=0.2)

axes[0].set_ylabel(
    "Pooled flood IoU, mean across four seeds"
)

axes[0].legend(
    frameon=False
)

export(
    fig,
    "Figure_Shift_Profiles_Validation_Test"
)


# ============================================================
# SECTION 9 — ABSOLUTE CLEAN VS SHIFTED IOU
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.5, 4.7),
    layout="constrained"
)

for ax, (label, df) in zip(
    axes,
    [
        ("Candidate validation", val_sum),
        ("Official held-out test", test_sum)
    ]
):
    means = (
        df.groupby("variant")[
            [
                "clean_iou",
                "mean_shifted_iou"
            ]
        ]
        .mean()
        .loc[["control", "light"]]
    )

    x = np.arange(2)
    width = 0.33

    ax.bar(
        x - width / 2,
        means.clean_iou,
        width,
        label="Clean",
        color=COL["control"]
    )

    ax.bar(
        x + width / 2,
        means.mean_shifted_iou,
        width,
        label="Mean shifted",
        color=COL["light"]
    )

    ax.set_xticks(
        x,
        ["Control", "Mild augmentation"]
    )

    ax.set_title(
        label,
        weight="bold"
    )

    ax.set_ylabel(
        "Mean pooled flood IoU"
    )

    ax.set_ylim(0, 0.78)

    ax.legend(
        frameon=False
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

export(
    fig,
    "Figure_Absolute_IoU_Clean_vs_Shifted"
)


# ============================================================
# SECTION 10 — MANUSCRIPT TABLE FUNCTION
# ============================================================

def save_table(
    df,
    stem,
    caption,
    fmt=5
):
    """
    Save both machine-readable CSV and
    publication-quality visual table.
    """

    csv_path = TAB / (stem + ".csv")

    df.to_csv(
        csv_path,
        index=False
    )

    outputs.append(csv_path)

    display_df = df.copy()

    for column in display_df.columns:

        if pd.api.types.is_float_dtype(
            display_df[column]
        ):
            display_df[column] = (
                display_df[column]
                .map(
                    lambda value: f"{value:.{fmt}f}"
                )
            )

    figure_width = max(
        9,
        min(
            19,
            2.0 + len(display_df.columns) * 1.5
        )
    )

    figure_height = max(
        2.7,
        1.5 + len(display_df) * 0.42
    )

    fig, ax = plt.subplots(
        figsize=(
            figure_width,
            figure_height
        )
    )

    ax.axis("off")

    ax.set_title(
        caption,
        loc="left",
        weight="bold",
        fontsize=12,
        pad=15
    )

    table = ax.table(
        cellText=display_df.astype(str).values,
        colLabels=display_df.columns.tolist(),
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)

    table.set_fontsize(8)

    table.scale(
        1,
        1.45
    )

    for (row, column), cell in table.get_celld().items():

        cell.set_edgecolor(
            "#D1DAE5"
        )

        cell.set_linewidth(0.45)

        if row == 0:
            cell.set_facecolor(
                "#DDEBF5"
            )

            cell.get_text().set_weight(
                "bold"
            )

        elif row % 2 == 0:
            cell.set_facecolor(
                "#F4F7FA"
            )

    export(
        fig,
        stem
    )


# ============================================================
# SECTION 11 — TABLE 1: SEED-WISE MODEL PERFORMANCE
# ============================================================

table1 = pd.concat(
    [
        val_sum.assign(
            partition="Validation"
        ),
        test_sum.assign(
            partition="Official test"
        )
    ],
    ignore_index=True
)

table1 = table1[
    [
        "partition",
        "seed",
        "variant",
        "clean_iou",
        "mean_shifted_iou",
        "mean_iou_loss",
        "worst_shifted_iou",
        "clean_f1"
    ]
]

save_table(
    table1,
    "Table_Seedwise_Model_Performance",
    "Table 1. Seed-wise validation and official held-out performance"
)


# ============================================================
# SECTION 12 — TABLE 2: STATISTICAL SYNTHESIS
# ============================================================

table2 = statistics[
    [
        "partition",
        "metric",
        "control_mean",
        "light_mean",
        "mean_difference",
        "ci95_lower",
        "ci95_upper",
        "exploratory_p",
        "seeds_favoring_light"
    ]
]

save_table(
    table2,
    "Table_Statistical_Synthesis",
    "Table 2. Four-seed paired exploratory statistical synthesis",
    fmt=6
)


# ============================================================
# SECTION 13 — TABLE 3: SHIFT-CONDITION PERFORMANCE
# ============================================================

all_shift_results = pd.concat(
    [
        val_long.assign(
            partition="Validation"
        ),
        test_long.assign(
            partition="Official test"
        )
    ],
    ignore_index=True
)

table3 = (
    all_shift_results
    .groupby(
        [
            "partition",
            "dx",
            "dy",
            "variant"
        ],
        as_index=False
    )
    .IoU.mean()
)

save_table(
    table3,
    "Table_Shift_Condition_Means",
    "Table 3. Mean IoU for each predefined spatial displacement",
    fmt=5
)


# ============================================================
# SECTION 14 — PRESERVE ORIGINAL ANALYSIS DATA
# ============================================================

for source in needed:

    target = (
        TAB
        / ("source_" + source.name)
    )

    shutil.copy2(
        source,
        target
    )

    outputs.append(target)


# ============================================================
# SECTION 15 — INCLUDE EXISTING QUALITATIVE FIGURES
# ============================================================

QUAL = (
    FINAL
    / "figures/paired_errors"
)

qualitative_images = (
    sorted(
        QUAL.glob(
            "UA_FloodNet_PairedErrors_*.png"
        )
    )
    if QUAL.exists()
    else []
)

if len(qualitative_images) == 6:

    destination = (
        FIG
        / "qualitative_existing"
    )

    destination.mkdir(
        exist_ok=True
    )

    for source in sorted(
        QUAL.glob(
            "UA_FloodNet_PairedErrors_*"
        )
    ):

        if source.suffix.lower() in (
            ".png",
            ".pdf"
        ):

            target = (
                destination
                / source.name
            )

            shutil.copy2(
                source,
                target
            )

            outputs.append(target)

    metrics_path = (
        QUAL
        / "step79_qualitative_sample_metrics.csv"
    )

    if metrics_path.exists():

        target = (
            TAB
            / metrics_path.name
        )

        shutil.copy2(
            metrics_path,
            target
        )

        outputs.append(target)

    print(
        "Included six existing qualitative "
        "figure pairs without modification."
    )

else:
    print(
        "Six qualitative PNG figures were not found "
        "at the expected directory. "
        "No missing outputs were fabricated."
    )


# ============================================================
# SECTION 16 — MANUSCRIPT FIGURE AND TABLE CAPTIONS
# ============================================================

captions = {

    "Figure_Architecture_3Panel":
        (
            "Three-panel schematic of the implemented "
            "UA-FloodNet full variant: "
            "(a) time-shared modality encoders and "
            "multiscale temporal fusion; "
            "(b) same-date optical-to-SAR displacement "
            "sampling and uncertainty-proxy weighting; "
            "(c) coarse-to-fine decoder, training-only "
            "auxiliary objectives and evaluation."
        ),

    "Figure_Seedwise_IoU_Degradation":
        (
            "Seed-wise IoU deterioration under 12 "
            "nonzero synthetic misregistration conditions "
            "in candidate validation and the official "
            "held-out test. Lower degradation indicates "
            "greater relative shift tolerance."
        ),

    "Figure_Paired_Effects_95CI":
        (
            "Four-seed paired differences between mild "
            "augmentation and control. Error bars show "
            "exploratory 95% paired t confidence intervals. "
            "No independent flood-event inference is implied."
        ),

    "Figure_Shift_Profiles_Validation_Test":
        (
            "Pooled IoU across the 13 predefined optical "
            "displacement conditions, averaged over four "
            "checkpoint seeds. SAR and reference masks "
            "remain fixed."
        ),

    "Figure_Absolute_IoU_Clean_vs_Shifted":
        (
            "Absolute clean and shifted pooled flood IoU. "
            "Small relative robustness gains need not "
            "improve absolute delineation accuracy."
        ),

    "Table_Seedwise_Model_Performance":
        (
            "Full-image checkpoint selection was performed "
            "using the validation split. Numbers shown here "
            "use the prespecified central crop for robustness "
            "analysis."
        ),

    "Table_Statistical_Synthesis":
        (
            "Exploratory seed-level paired confidence "
            "intervals and unadjusted p-values based on "
            "four paired training seeds. "
            "Event independence is unverified."
        ),

    "Table_Shift_Condition_Means":
        (
            "Separate displacement directions are preserved. "
            "Shifted-condition averages should not be treated "
            "as statistically independent replicates."
        )
}

captions_json = (
    OUT
    / "figure_table_captions.json"
)

captions_json.write_text(
    json.dumps(
        captions,
        indent=2
    )
)

captions_txt = (
    OUT
    / "figure_table_captions.txt"
)

captions_txt.write_text(
    "\n\n".join(
        key + "\n" + value
        for key, value in captions.items()
    )
)


# ============================================================
# SECTION 17 — REPRODUCIBILITY MANIFEST
# ============================================================

manifest = {

    "study": "UA-FloodNet",

    "dpi_requested": DPI,

    "vector_master": [
        "SVG",
        "PDF"
    ],

    "four_seeds": [
        7,
        17,
        37,
        42
    ],

    "test_partition":
        "Official held-out OMBRIA test partition, 70 samples",

    "event_independence":
        "Not verified",

    "test_model_selection":
        False,

    "note":
        (
            "Existing selected qualitative examples are "
            "illustrative. All numerical figures use "
            "previously saved analysis CSV files."
        ),

    "sources": {
        str(path.relative_to(ROOT)):
            hashlib.sha256(
                path.read_bytes()
            ).hexdigest()
        for path in needed
    },

    "files": [
        str(path.relative_to(OUT))
        for path in outputs
    ]
}

manifest_path = (
    OUT
    / "reproducibility_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2
    )
)


# ============================================================
# SECTION 18 — FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 75)
print("UA-FLOODNET MANUSCRIPT PACKAGE COMPLETE")
print("=" * 75)

print(
    "Output directory:",
    OUT
)

print(
    "Generated vector and raster figure files:",
    len([
        path
        for path in outputs
        if path.parent == FIG
    ])
)

print(
    "Table CSV files:",
    len(
        list(
            TAB.glob("*.csv")
        )
    )
)

print(
    "Requested DPI:",
    DPI
)

print("\nPrimary architecture outputs:")

for extension in [
    ".svg",
    ".pdf"
]:
    path = (
        FIG
        / (
            "Figure_Architecture_3Panel"
            + extension
        )
    )

    print(
        path.name,
        "FOUND" if path.exists() else "MISSING"
    )

print("\nFINAL PACKAGE SAVED SUCCESSFULLY")

In [ ]:
import os

os.environ["UA_FLOODNET_DPI"] = "1200"

In [ ]:
%run /content/UA_FloodNet_Manuscript_Figures_Tables_COLAB_DISPLAY.py

In [ ]:
import os

os.environ["UA_FLOODNET_DPI"] = "1200"

In [ ]:
# ============================================================
# STEP 81A — SAFE GOOGLE DRIVE CONNECTION
# ============================================================

from google.colab import drive
from pathlib import Path

import os
import subprocess

print("=" * 65)
print("STEP 81A: GOOGLE DRIVE RECOVERY")
print("=" * 65)

mounts = [
    Path("/content/ua_drive_mount"),
    Path("/content/drive"),
]

ROOT = None

# 1. Check existing connections
for mount in mounts:
    candidate = mount / "MyDrive/UA_FloodNet"

    if candidate.is_dir():
        ROOT = candidate
        print("Existing project found:", ROOT)
        break

# 2. Connect only if the project is not already accessible
if ROOT is None:

    preferred = Path("/content/ua_drive_mount")

    if preferred.exists() and any(preferred.iterdir()):
        preferred = Path("/content/ua_drive_mount_new")

    preferred.mkdir(parents=True, exist_ok=True)

    if any(preferred.iterdir()):
        raise RuntimeError(
            f"Mount directory is not empty: {preferred}"
        )

    print("Connecting Google Drive...")
    drive.mount(str(preferred))

    ROOT = preferred / "MyDrive/UA_FloodNet"

# 3. Verify project
assert ROOT.is_dir(), (
    "UA_FloodNet project not found in the connected Google Drive. "
    "Check which Google account was selected."
)

print("\nPROJECT VERIFICATION")

required = {
    "Project": ROOT,
    "Final analysis": ROOT / "v4/final_analysis",
    "Control checkpoint": ROOT / "v4/full/control_seed42_50ep.pt",
    "Light checkpoint": ROOT / "v4/light/light_seed42_50ep.pt",
    "Official test results": (
        ROOT / "v4/official_test/step74_frozen_test_results.csv"
    ),
}

for name, path in required.items():
    print(
        f"{name}:",
        "FOUND" if path.exists() else "MISSING"
    )

print("\nActive project root:", ROOT)
print("STEP 81A COMPLETED")

In [ ]:
import os

os.environ["UA_FLOODNET_DPI"] = "600"

%run /content/UA_FloodNet_Step81_IEEE_Style_All_Figures_Tables_COLAB.py

In [ ]:
# ================================================================
# STEP 81B — COMPACT IEEE-STYLE UA-FLOODNET FIGURES
# ================================================================
#
# FIGURE DESIGN:
#   2 rows × 4 columns
#   Small square panels
#   Equal image dimensions
#   Minimal spacing
#   Panel labels (a)–(h)
#   No large titles inside the figures
#
# ROW 1:
#   (a) SAR before flooding
#   (b) SAR after flooding
#   (c) Optical before flooding
#   (d) Optical after flooding
#
# ROW 2:
#   (e) Ground-truth flood mask
#   (f) Control prediction
#   (g) Mild-augmentation prediction
#   (h) Mild-augmentation classification errors
#
# OUTPUT:
#   6 compact scientific figures
#   SVG, PDF, PNG, TIFF
#   Publication captions
#   Per-sample metric CSV
#   Reproducibility metadata
#
# DATA:
#   Actual OMBRIA official held-out test images
#   Frozen UA-FloodNet seed-42 checkpoints
#
# ================================================================


# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

from pathlib import Path

import os
import sys
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch

import matplotlib.pyplot as plt

from matplotlib.colors import (
    ListedColormap,
    BoundaryNorm
)

from matplotlib.patches import Patch

from IPython.display import (
    display,
    Image as IPyImage
)


# ================================================================
# 2. PUBLICATION SETTINGS
# ================================================================

# Requested output resolution.
# SVG and PDF remain resolution-independent.
DPI = 1200

# Compact IEEE-style figure dimensions, in inches.
FIGURE_WIDTH = 7.2
FIGURE_HEIGHT = 4.05

# Colab display preview.
PREVIEW_DPI = 150

# Panel layout.
ROWS = 2
COLUMNS = 4

# Model decision threshold.
THRESHOLD = 0.5

# Fixed optical displacement experiment.
CONDITIONS = [
    (0, 0, "clean"),
    (4, 0, "shift4")
]

# Original fixed sample selection.
SAMPLE_IDS = [
    "0013",
    "0408",
    "0777"
]

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 8,

    "axes.labelsize": 8,

    "axes.titlesize": 8,

    "pdf.fonttype": 42,

    "ps.fonttype": 42,

    "svg.fonttype": "none",

    "figure.facecolor": "white",

    "savefig.facecolor": "white"

})


# ================================================================
# 3. LOCATE GOOGLE DRIVE
# ================================================================

print("=" * 72)
print("STEP 81B: COMPACT IEEE-STYLE UA-FLOODNET FIGURES")
print("=" * 72)

MOUNTS = [

    Path(
        "/content/ua_drive_mount/MyDrive/UA_FloodNet"
    ),

    Path(
        "/content/ua_drive_mount_new/MyDrive/UA_FloodNet"
    ),

    Path(
        "/content/drive/MyDrive/UA_FloodNet"
    )

]

ROOT = next(
    (p for p in MOUNTS if p.exists()),
    None
)

if ROOT is None:

    from google.colab import drive

    # Avoid reusing a nonempty mount directory.
    mount = Path("/content/ua_floodnet_drive")

    if not mount.exists():
        mount.mkdir(parents=True)

    if any(mount.iterdir()):
        raise RuntimeError(
            "The alternative mount directory is not empty. "
            "Inspect it before attempting a new mount."
        )

    drive.mount(str(mount))

    ROOT = mount / "MyDrive/UA_FloodNet"

assert ROOT.exists(), (
    "UA_FloodNet was not found in the connected Google Drive."
)

print("Project root:", ROOT)


# ================================================================
# 4. OUTPUT DIRECTORIES
# ================================================================

OUT = (
    ROOT
    / "v4"
    / "final_analysis"
    / "manuscript_package_step81"
    / "compact_figures"
)

PNG_DIR = OUT / "PNG"
PDF_DIR = OUT / "PDF"
SVG_DIR = OUT / "SVG"
TIFF_DIR = OUT / "TIFF"
PREVIEW_DIR = OUT / "Previews"

for directory in [
    OUT,
    PNG_DIR,
    PDF_DIR,
    SVG_DIR,
    TIFF_DIR,
    PREVIEW_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Output:", OUT)


# ================================================================
# 5. RESTORE PYTHON SOURCE CODE
# ================================================================

CODE = Path(
    "/content/UA_FloodNet/ablations_code"
)

CODE.mkdir(
    parents=True,
    exist_ok=True
)

for filename in [
    "ablation_model.py",
    "data.py"
]:

    destination = CODE / filename
    source = ROOT / "v4/code" / filename

    if not destination.exists():

        assert source.exists(), (
            f"Missing source file: {source}"
        )

        shutil.copy2(
            source,
            destination
        )

    print("Source available:", filename)

sys.path.insert(
    0,
    str(CODE)
)

from ablation_model import UAFloodNet


# ================================================================
# 6. VERIFY OR RESTORE OMBRIA TEST DATA
# ================================================================

DATA = Path(
    "/content/UA_FloodNet/dataset"
)

TEST = DATA / "test"

test_files = sorted(
    TEST.glob("*.npz")
)

if len(test_files) != 70:

    print("\nRestoring frozen OMBRIA dataset...")

    archive = (
        ROOT
        / "dataset_backup"
        / "OMBRIA_NPZ_candidate_seed42.zip"
    )

    assert archive.exists(), (
        f"Missing dataset archive: {archive}"
    )

    DATA.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(archive, "r") as z:

        assert z.testzip() is None, (
            "Dataset ZIP integrity test failed."
        )

        data_root = DATA.resolve()

        for member in z.infolist():

            target = (
                DATA / member.filename
            ).resolve()

            if not target.is_relative_to(data_root):
                raise RuntimeError(
                    f"Unsafe ZIP path: {member.filename}"
                )

        z.extractall(DATA)

    test_files = sorted(
        TEST.glob("*.npz")
    )

assert len(test_files) == 70, (
    f"Expected 70 test samples; found {len(test_files)}"
)

print("Official test samples:", len(test_files))


# ================================================================
# 7. INITIALIZE COMPUTING DEVICE
# ================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Computing device:", device)


# ================================================================
# 8. LOAD FROZEN MODELS
# ================================================================

checkpoint_paths = {

    "Control": (
        ROOT
        / "v4/full/control_seed42_50ep.pt"
    ),

    "Light": (
        ROOT
        / "v4/light/light_seed42_50ep.pt"
    )

}

models = {}

for name, path in checkpoint_paths.items():

    assert path.exists(), (
        f"Missing checkpoint: {path}"
    )

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    assert int(checkpoint["seed"]) == 42

    model = UAFloodNet(

        width=checkpoint["width"],

        use_alignment=checkpoint["use_alignment"],

        use_reliability=checkpoint["use_reliability"]

    ).to(device)

    model.load_state_dict(
        checkpoint["model"],
        strict=True
    )

    model.eval()

    models[name] = model

    print(
        f"{name} loaded | "
        f"seed={checkpoint['seed']} | "
        f"epoch={checkpoint['epoch']}"
    )


# ================================================================
# 9. IMAGE AND METRIC FUNCTIONS
# ================================================================

def shift_zero(x, dx=0, dy=0):

    """
    Translate optical tensor spatially with zero padding.
    Positive dx = rightward shift.
    Positive dy = downward shift.
    """

    _, _, height, width = x.shape

    output = torch.zeros_like(x)

    y0 = max(0, dy)
    y1 = min(height, height + dy)

    x0 = max(0, dx)
    x1 = min(width, width + dx)

    if y1 > y0 and x1 > x0:

        output[
            :, :, y0:y1, x0:x1
        ] = x[
            :,
            :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]

    return output


def tensor_to_rgb(tensor):

    """
    Convert saved 3-channel OMBRIA optical composite
    to a displayable H × W × 3 array.

    No unverified spectral-band identities are assumed.
    """

    image = (
        tensor
        .detach()
        .cpu()
        .numpy()
    )

    if image.ndim == 3:
        image = np.transpose(
            image,
            (1, 2, 0)
        )

    return np.clip(
        image,
        0,
        1
    )


def make_error_map(pred, gt):

    """
    0 = True negative
    1 = True positive
    2 = False positive
    3 = False negative
    """

    error = np.zeros(
        gt.shape,
        dtype=np.uint8
    )

    error[pred & gt] = 1

    error[pred & ~gt] = 2

    error[~pred & gt] = 3

    return error


def calculate_metrics(pred, gt):

    tp = int(
        np.logical_and(pred, gt).sum()
    )

    fp = int(
        np.logical_and(pred, ~gt).sum()
    )

    fn = int(
        np.logical_and(~pred, gt).sum()
    )

    iou = tp / max(
        tp + fp + fn,
        1
    )

    f1 = (
        2 * tp
    ) / max(
        2 * tp + fp + fn,
        1
    )

    return {
        "IoU": iou,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn
    }


# ================================================================
# 10. SCIENTIFIC COLOUR PALETTES
# ================================================================

MASK_CMAP = ListedColormap([

    "#080808",  # Background

    "#FFFFFF"   # Flood

])

ERROR_CMAP = ListedColormap([

    "#101010",  # True negative

    "#FFFFFF",  # True positive

    "#F6A000",  # False positive

    "#D93459"   # False negative

])

ERROR_NORM = BoundaryNorm(

    [-0.5, 0.5, 1.5, 2.5, 3.5],

    ERROR_CMAP.N

)

legend_handles = [

    Patch(
        facecolor="#101010",
        label="TN"
    ),

    Patch(
        facecolor="#FFFFFF",
        edgecolor="#555555",
        label="TP"
    ),

    Patch(
        facecolor="#F6A000",
        label="FP"
    ),

    Patch(
        facecolor="#D93459",
        label="FN"
    )

]


# ================================================================
# 11. COMPACT FIGURE DRAWING FUNCTION
# ================================================================

def draw_compact_figure(
    panels,
    figure_name,
    sample_id,
    dx,
    dy
):

    """
    Eight panels arranged as 2 × 4.

    Export quality is controlled separately from
    the physical manuscript display dimensions.
    """

    fig, axes = plt.subplots(

        ROWS,
        COLUMNS,

        figsize=(
            FIGURE_WIDTH,
            FIGURE_HEIGHT
        )

    )

    # Tight spacing, similar to the reference paper.
    fig.subplots_adjust(

        left=0.015,
        right=0.985,

        top=0.99,
        bottom=0.145,

        wspace=0.055,
        hspace=0.24

    )

    labels = [
        "(a)", "(b)", "(c)", "(d)",
        "(e)", "(f)", "(g)", "(h)"
    ]

    for index, ax in enumerate(
        axes.flat
    ):

        image, kind = panels[index]

        if kind == "sar":

            ax.imshow(
                image,
                cmap="gray",
                vmin=0,
                vmax=1,
                interpolation="nearest"
            )

        elif kind == "optical":

            ax.imshow(
                image,
                interpolation="nearest"
            )

        elif kind == "mask":

            ax.imshow(
                image.astype(np.uint8),
                cmap=MASK_CMAP,
                vmin=0,
                vmax=1,
                interpolation="nearest"
            )

        elif kind == "error":

            ax.imshow(
                image,
                cmap=ERROR_CMAP,
                norm=ERROR_NORM,
                interpolation="nearest"
            )

        ax.set_aspect("equal")

        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_visible(False)

        # Small panel letter directly below the image.
        ax.set_xlabel(

            labels[index],

            fontsize=8.5,

            labelpad=2,

            fontweight="normal"

        )

    # Small legend: no large decorative title.
    fig.legend(

        handles=legend_handles,

        loc="lower center",

        bbox_to_anchor=(0.5, 0.005),

        ncol=4,

        frameon=False,

        fontsize=7.5,

        handlelength=1.1,

        columnspacing=1.2

    )

    # --------------------------------------------------------
    # Export high-quality files
    # --------------------------------------------------------

    fig.savefig(

        SVG_DIR / f"{figure_name}.svg",

        format="svg",

        bbox_inches="tight",

        pad_inches=0.025,

        facecolor="white"

    )

    fig.savefig(

        PDF_DIR / f"{figure_name}.pdf",

        format="pdf",

        bbox_inches="tight",

        pad_inches=0.025,

        facecolor="white"

    )

    fig.savefig(

        PNG_DIR / f"{figure_name}_{DPI}dpi.png",

        format="png",

        dpi=DPI,

        bbox_inches="tight",

        pad_inches=0.025,

        facecolor="white"

    )

    fig.savefig(

        TIFF_DIR / f"{figure_name}_{DPI}dpi.tiff",

        format="tiff",

        dpi=DPI,

        pil_kwargs={
            "compression": "tiff_lzw"
        },

        bbox_inches="tight",

        pad_inches=0.025,

        facecolor="white"

    )

    # --------------------------------------------------------
    # Create a lightweight Colab preview.
    # --------------------------------------------------------

    preview_path = (
        PREVIEW_DIR
        / f"{figure_name}_preview.png"
    )

    fig.savefig(

        preview_path,

        format="png",

        dpi=PREVIEW_DPI,

        bbox_inches="tight",

        pad_inches=0.025,

        facecolor="white"

    )

    plt.close(fig)

    print(
        f"\nSample {sample_id} | "
        f"Shift ({dx:+d}, {dy:+d})"
    )

    display(
        IPyImage(
            filename=str(preview_path)
        )
    )

    print("Saved:", figure_name)


# ================================================================
# 12. GENERATE ALL COMPACT FIGURES
# ================================================================

print("\n" + "=" * 72)
print("GENERATING COMPACT OMBRIA FIGURES")
print("=" * 72)

records = []
captions = {}

for sample_id in SAMPLE_IDS:

    sample_path = (
        TEST
        / f"{sample_id}.npz"
    )

    assert sample_path.exists(), (
        f"Missing test sample: {sample_path}"
    )

    with np.load(
        sample_path,
        allow_pickle=False
    ) as data:

        sample = {

            key: torch.as_tensor(

                data[key],

                dtype=torch.float32,

                device=device

            ).unsqueeze(0)

            for key in [
                "sar_pre",
                "sar_post",
                "ms_pre",
                "ms_post",
                "mask"
            ]

        }

    gt = (
        sample["mask"][0, 0]
        .cpu()
        .numpy() >= 0.5
    )

    sar_before = (
        sample["sar_pre"][0, 0]
        .cpu()
        .numpy()
    )

    sar_after = (
        sample["sar_post"][0, 0]
        .cpu()
        .numpy()
    )

    for dx, dy, condition in CONDITIONS:

        ms_before = shift_zero(
            sample["ms_pre"],
            dx,
            dy
        )

        ms_after = shift_zero(
            sample["ms_post"],
            dx,
            dy
        )

        predictions = {}

        with torch.inference_mode():

            for name, model in models.items():

                output = model(

                    sample["sar_pre"],

                    sample["sar_post"],

                    ms_before,

                    ms_after

                )

                probabilities = (
                    output["logits"]
                    .sigmoid()[0, 0]
                    .cpu()
                    .numpy()
                )

                predictions[name] = (
                    probabilities >= THRESHOLD
                )

        control_pred = predictions["Control"]

        light_pred = predictions["Light"]

        light_errors = make_error_map(
            light_pred,
            gt
        )

        # ----------------------------------------------------
        # Eight compact panels
        # ----------------------------------------------------

        panels = [

            (
                sar_before,
                "sar"
            ),

            (
                sar_after,
                "sar"
            ),

            (
                tensor_to_rgb(
                    ms_before[0]
                ),
                "optical"
            ),

            (
                tensor_to_rgb(
                    ms_after[0]
                ),
                "optical"
            ),

            (
                gt,
                "mask"
            ),

            (
                control_pred,
                "mask"
            ),

            (
                light_pred,
                "mask"
            ),

            (
                light_errors,
                "error"
            )

        ]

        figure_name = (
            f"UA_FloodNet_Compact_"
            f"{sample_id}_{condition}"
        )

        draw_compact_figure(

            panels=panels,

            figure_name=figure_name,

            sample_id=sample_id,

            dx=dx,

            dy=dy

        )

        # ----------------------------------------------------
        # Save source-derived metrics
        # ----------------------------------------------------

        for model_name, prediction in [

            ("Control", control_pred),

            ("Light", light_pred)

        ]:

            metrics = calculate_metrics(
                prediction,
                gt
            )

            records.append({

                "sample_id": sample_id,

                "condition": condition,

                "dx": dx,

                "dy": dy,

                "model": model_name,

                **metrics

            })

        # ----------------------------------------------------
        # Scientific figure caption
        # ----------------------------------------------------

        captions[figure_name] = (

            f"Compact multimodal flood-detection comparison "
            f"for OMBRIA official test sample {sample_id}. "
            f"Panels (a–d) show pre-flood SAR, post-flood SAR, "
            f"pre-flood optical and post-flood optical composites; "
            f"(e) the reference flood mask; "
            f"(f) the control prediction; "
            f"(g) the mild-augmentation prediction; "
            f"and (h) the classification-error map for "
            f"mild augmentation. "
            f"The applied optical displacement is "
            f"({dx:+d}, {dy:+d}) pixels. "
            f"SAR inputs and reference masks remain unchanged. "
            f"Error-map legend: TN, true negative; "
            f"TP, true positive; FP, false positive; "
            f"FN, false negative."
        )


# ================================================================
# 13. EXPORT QUALITATIVE METRICS
# ================================================================

results = pd.DataFrame(records)

assert len(results) == 12, (
    f"Expected 12 records, found {len(results)}"
)

csv_path = (
    OUT
    / "compact_qualitative_metrics.csv"
)

results.to_csv(
    csv_path,
    index=False
)

print("\nQUALITATIVE PERFORMANCE")

print(
    results[
        [
            "sample_id",
            "condition",
            "model",
            "IoU",
            "F1"
        ]
    ].round(5).to_string(index=False)
)


# ================================================================
# 14. EXPORT CAPTIONS
# ================================================================

caption_path = (
    OUT
    / "compact_figure_captions.json"
)

caption_path.write_text(
    json.dumps(
        captions,
        indent=2
    )
)

text_path = (
    OUT
    / "compact_figure_captions.txt"
)

text_path.write_text(

    "\n\n".join(

        name + "\n" + caption

        for name, caption in captions.items()

    )

)


# ================================================================
# 15. EXPORT REPRODUCIBILITY METADATA
# ================================================================

metadata = {

    "study": "UA-FloodNet",

    "dataset": "OMBRIA",

    "partition": "official held-out test",

    "event_independence": "not verified",

    "selected_samples": SAMPLE_IDS,

    "sample_selection":
        "Fixed previously selected test samples",

    "model_seed": 42,

    "threshold": THRESHOLD,

    "optical_shift_conditions": [
        [0, 0],
        [4, 0]
    ],

    "SAR_shifted": False,

    "mask_shifted": False,

    "figure_grid": "2 rows x 4 columns",

    "figure_dimensions_inches": [
        FIGURE_WIDTH,
        FIGURE_HEIGHT
    ],

    "raster_dpi": DPI,

    "formats": [
        "SVG",
        "PDF",
        "PNG",
        "TIFF"
    ],

    "metrics_scope":
        "Full-patch illustrative performance, not pooled "
        "20-pixel-interior official evaluation",

    "checkpoints": {
        name: str(path)
        for name, path in checkpoint_paths.items()
    }

}

metadata_path = (
    OUT
    / "compact_figure_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2
    )
)


# ================================================================
# 16. VERIFY ALL OUTPUTS
# ================================================================

png_files = sorted(
    PNG_DIR.glob("UA_FloodNet_Compact_*.png")
)

pdf_files = sorted(
    PDF_DIR.glob("UA_FloodNet_Compact_*.pdf")
)

svg_files = sorted(
    SVG_DIR.glob("UA_FloodNet_Compact_*.svg")
)

tiff_files = sorted(
    TIFF_DIR.glob("UA_FloodNet_Compact_*.tiff")
)

assert len(png_files) == 6
assert len(pdf_files) == 6
assert len(svg_files) == 6
assert len(tiff_files) == 6

print("\n" + "=" * 72)
print("STEP 81B COMPLETED SUCCESSFULLY")
print("=" * 72)

print("Compact figures:", 6)
print("PNG:", len(png_files))
print("PDF:", len(pdf_files))
print("SVG:", len(svg_files))
print("TIFF:", len(tiff_files))

print("Requested raster DPI:", DPI)

print("\nOutput directory:")
print(OUT)

print("\nNEXT: REVIEW COMPACT FIGURE LAYOUT")

In [ ]:

# ============================================================
# FINAL COMPACT IEEE-STYLE UA-FLOODNET FIGURES
# Six figures | No legends inside images
# Individual clean/shifted captions | 1200 dpi
# ============================================================

from pathlib import Path
import sys
import shutil
import zipfile
import json
import os

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import display, Image as IPyImage

# ------------------------------------------------------------
# 1. FIGURE SETTINGS
# ------------------------------------------------------------

DPI = int(os.getenv("UA_FLOODNET_DPI", "1200"))
PREVIEW_DPI = 160

FIGURE_WIDTH = 7.2
FIGURE_HEIGHT = 4.05

THRESHOLD = 0.5

SAMPLES = ("0013", "0408", "0777")

CONDITIONS = (
    (0, 0, "clean"),
    (4, 0, "shift4")
)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "savefig.facecolor": "white"
})

# ------------------------------------------------------------
# 2. LOCATE GOOGLE DRIVE
# ------------------------------------------------------------

roots = [
    Path("/content/ua_drive_mount/MyDrive/UA_FloodNet"),
    Path("/content/ua_drive_mount_new/MyDrive/UA_FloodNet"),
    Path("/content/drive/MyDrive/UA_FloodNet"),
    Path("/content/ua_floodnet_drive/MyDrive/UA_FloodNet")
]

ROOT = next(
    (p for p in roots if p.is_dir()),
    None
)

if ROOT is None:
    from google.colab import drive

    for candidate in (
        "/content/ua_drive_mount",
        "/content/ua_drive_mount_new",
        "/content/ua_floodnet_drive"
    ):
        mount = Path(candidate)

        if not mount.exists() or not any(mount.iterdir()):
            mount.mkdir(parents=True, exist_ok=True)
            drive.mount(str(mount))

            possible_root = mount / "MyDrive/UA_FloodNet"

            if possible_root.is_dir():
                ROOT = possible_root
                break

    if ROOT is None:
        raise FileNotFoundError(
            "UA_FloodNet folder not found. "
            "Check the connected Google Drive account."
        )

# ------------------------------------------------------------
# 3. OUTPUT DIRECTORIES
# ------------------------------------------------------------

OUT = (
    ROOT
    / "v4/final_analysis/manuscript_package_step81"
    / "compact_figures_final"
)

folders = {
    extension: OUT / extension.upper()
    for extension in ("png", "pdf", "svg", "tiff")
}

PREVIEWS = OUT / "Previews"

for folder in (*folders.values(), PREVIEWS):
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Output:", OUT)

# ------------------------------------------------------------
# 4. RESTORE SOURCE CODE
# ------------------------------------------------------------

CODE = Path("/content/UA_FloodNet/ablations_code")
CODE.mkdir(parents=True, exist_ok=True)

for filename in ("ablation_model.py", "data.py"):
    destination = CODE / filename

    if not destination.is_file():
        source = ROOT / "v4/code" / filename

        if not source.is_file():
            raise FileNotFoundError(source)

        shutil.copy2(source, destination)

sys.path.insert(0, str(CODE))

from ablation_model import UAFloodNet

# ------------------------------------------------------------
# 5. VERIFY OR RESTORE DATASET
# ------------------------------------------------------------

DATA = Path("/content/UA_FloodNet/dataset")
TEST = DATA / "test"

if len(list(TEST.glob("*.npz"))) != 70:

    archive = (
        ROOT
        / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
    )

    if not archive.exists():
        raise FileNotFoundError(archive)

    DATA.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive) as z:

        if z.testzip() is not None:
            raise RuntimeError("Dataset ZIP integrity failed.")

        extraction_root = DATA.resolve()

        for member in z.infolist():
            target = (DATA / member.filename).resolve()

            if not target.is_relative_to(extraction_root):
                raise ValueError("Unsafe archive entry.")

        z.extractall(DATA)

if len(list(TEST.glob("*.npz"))) != 70:
    raise RuntimeError(
        "Official test partition must contain 70 samples."
    )

for sample_id in SAMPLES:
    path = TEST / f"{sample_id}.npz"

    if not path.is_file():
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# 6. LOAD FROZEN CHECKPOINTS
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_paths = {
    "Control": ROOT / "v4/full/control_seed42_50ep.pt",
    "Light": ROOT / "v4/light/light_seed42_50ep.pt"
}

models = {}

for name, path in checkpoint_paths.items():

    if not path.is_file():
        raise FileNotFoundError(path)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    if int(checkpoint["seed"]) != 42:
        raise ValueError(f"Unexpected checkpoint seed: {name}")

    model = UAFloodNet(
        width=int(checkpoint["width"]),
        use_alignment=bool(checkpoint["use_alignment"]),
        use_reliability=bool(checkpoint["use_reliability"])
    ).to(device)

    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval()

    models[name] = model

    print(
        f"{name}: seed {checkpoint['seed']}, "
        f"best epoch {checkpoint['epoch']}"
    )

print("Device:", device)

# ------------------------------------------------------------
# 7. HELPER FUNCTIONS
# ------------------------------------------------------------

def shift_zero(x, dx=0, dy=0):

    _, _, height, width = x.shape

    result = torch.zeros_like(x)

    y0 = max(dy, 0)
    y1 = min(height, height + dy)

    x0 = max(dx, 0)
    x1 = min(width, width + dx)

    if y1 > y0 and x1 > x0:

        result[:, :, y0:y1, x0:x1] = x[
            :,
            :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]

    return result


def to_rgb(tensor):

    image = tensor.detach().cpu().numpy()

    return np.clip(
        image.transpose(1, 2, 0),
        0,
        1
    )


def make_error_map(pred, gt):

    result = np.zeros(gt.shape, dtype=np.uint8)

    result[pred & gt] = 1
    result[pred & ~gt] = 2
    result[~pred & gt] = 3

    return result


def calculate_metrics(pred, gt):

    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())

    return {
        "IoU": tp / max(1, tp + fp + fn),
        "F1": 2 * tp / max(1, 2 * tp + fp + fn),
        "TP": tp,
        "FP": fp,
        "FN": fn
    }

# ------------------------------------------------------------
# 8. MASK AND ERROR COLOURS
# ------------------------------------------------------------

mask_cmap = ListedColormap([
    "#080808",
    "#FFFFFF"
])

error_cmap = ListedColormap([
    "#101010",  # True negative
    "#FFFFFF",  # True positive
    "#F6A000",  # False positive
    "#D93459"   # False negative
])

error_norm = BoundaryNorm(
    [-0.5, 0.5, 1.5, 2.5, 3.5],
    error_cmap.N
)

letters = tuple("abcdefgh")

records = []
captions = {}

# ------------------------------------------------------------
# 9. GENERATE SIX COMPACT FIGURES
# ------------------------------------------------------------

for sample_id in SAMPLES:

    sample_path = TEST / f"{sample_id}.npz"

    with np.load(
        sample_path,
        allow_pickle=False
    ) as data:

        keys = (
            "sar_pre",
            "sar_post",
            "ms_pre",
            "ms_post",
            "mask"
        )

        batch = {
            key: torch.from_numpy(
                data[key].copy()
            ).float().unsqueeze(0).to(device)
            for key in keys
        }

    gt = (
        batch["mask"][0, 0]
        .cpu().numpy() >= 0.5
    )

    sar_pre = batch["sar_pre"][0, 0].cpu().numpy()
    sar_post = batch["sar_post"][0, 0].cpu().numpy()

    for dx, dy, condition in CONDITIONS:

        ms_pre = shift_zero(batch["ms_pre"], dx, dy)
        ms_post = shift_zero(batch["ms_post"], dx, dy)

        predictions = {}

        with torch.inference_mode():

            for name, model in models.items():

                logits = model(
                    batch["sar_pre"],
                    batch["sar_post"],
                    ms_pre,
                    ms_post
                )["logits"]

                predictions[name] = (
                    logits.sigmoid()[0, 0]
                    .cpu().numpy() >= THRESHOLD
                )

        for name, pred in predictions.items():

            records.append({
                "sample_id": sample_id,
                "condition": condition,
                "dx": dx,
                "dy": dy,
                "model": name,
                **calculate_metrics(pred, gt)
            })

        panels = [
            (sar_pre, "sar"),
            (sar_post, "sar"),
            (to_rgb(ms_pre[0]), "rgb"),
            (to_rgb(ms_post[0]), "rgb"),
            (gt, "mask"),
            (predictions["Control"], "mask"),
            (predictions["Light"], "mask"),
            (
                make_error_map(predictions["Light"], gt),
                "error"
            )
        ]

        # Compact layout; no figure title or legend.
        fig, axes = plt.subplots(
            2,
            4,
            figsize=(FIGURE_WIDTH, FIGURE_HEIGHT)
        )

        fig.subplots_adjust(
            left=0.01,
            right=0.99,
            top=0.985,
            bottom=0.075,
            wspace=0.055,
            hspace=0.25
        )

        for index, (ax, (image, kind)) in enumerate(
            zip(axes.flat, panels)
        ):

            if kind == "sar":
                ax.imshow(
                    image,
                    cmap="gray",
                    vmin=0,
                    vmax=1,
                    interpolation="nearest"
                )

            elif kind == "rgb":
                ax.imshow(
                    image,
                    interpolation="nearest"
                )

            elif kind == "mask":
                ax.imshow(
                    image.astype(np.uint8),
                    cmap=mask_cmap,
                    vmin=0,
                    vmax=1,
                    interpolation="nearest"
                )

            else:
                ax.imshow(
                    image,
                    cmap=error_cmap,
                    norm=error_norm,
                    interpolation="nearest"
                )

            ax.set_aspect("equal")
            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_visible(False)

            ax.set_xlabel(
                f"({letters[index]})",
                fontsize=8.5,
                labelpad=2
            )

        filename = (
            f"UA_FloodNet_Compact_{sample_id}_{condition}"
        )

        # Vector SVG and PDF
        for extension in ("svg", "pdf"):

            fig.savefig(
                folders[extension] / f"{filename}.{extension}",
                format=extension,
                bbox_inches="tight",
                pad_inches=0.015,
                facecolor="white"
            )

        # High-resolution PNG
        fig.savefig(
            folders["png"] / f"{filename}_{DPI}dpi.png",
            dpi=DPI,
            bbox_inches="tight",
            pad_inches=0.015,
            facecolor="white"
        )

        # Lossless TIFF
        fig.savefig(
            folders["tiff"] / f"{filename}_{DPI}dpi.tiff",
            format="tiff",
            dpi=DPI,
            pil_kwargs={"compression": "tiff_lzw"},
            bbox_inches="tight",
            pad_inches=0.015,
            facecolor="white"
        )

        # Lightweight notebook preview
        preview_path = (
            PREVIEWS / f"{filename}_preview.png"
        )

        fig.savefig(
            preview_path,
            dpi=PREVIEW_DPI,
            bbox_inches="tight",
            pad_inches=0.015,
            facecolor="white"
        )

        plt.close(fig)

        print(
            f"\nSample {sample_id}: "
            f"{condition} — {filename}"
        )

        display(IPyImage(filename=str(preview_path)))

        # ----------------------------------------------------
        # 10. CONDITION-SPECIFIC MANUSCRIPT CAPTION
        # ----------------------------------------------------

        if condition == "clean":

            condition_text = (
                "Clean condition: no synthetic displacement "
                "was applied to either optical acquisition."
            )

        else:

            condition_text = (
                "Shifted condition: the pre-event and "
                "post-event optical composites were both "
                "translated four pixels horizontally to "
                "the right using zero padding. "
                "SAR observations and the reference mask "
                "remained unchanged."
            )

        captions[filename] = (
            f"Compact qualitative comparison of UA-FloodNet "
            f"predictions for official OMBRIA test sample "
            f"{sample_id}. "
            f"(a) Pre-event Sentinel-1 SAR; "
            f"(b) post-event Sentinel-1 SAR; "
            f"(c) pre-event Sentinel-2 three-channel composite; "
            f"(d) post-event Sentinel-2 three-channel composite; "
            f"(e) reference flood mask; "
            f"(f) unaugmented control prediction; "
            f"(g) mild-augmentation prediction; "
            f"and (h) mild-augmentation classification errors. "
            f"{condition_text} "
            f"Binary masks display flood in white and "
            f"non-flood in black. "
            f"For panel (h), black denotes true negatives "
            f"(TN), white denotes true positives (TP), "
            f"orange denotes false positives (FP), "
            f"and pink denotes false negatives (FN). "
            f"Both models use training seed 42 and a "
            f"probability threshold of 0.5. "
            f"Patch-level examples are illustrative and "
            f"do not replace pooled official-test results."
        )

# ------------------------------------------------------------
# 11. SAVE METRICS AND CAPTIONS
# ------------------------------------------------------------

results = pd.DataFrame(records)

assert len(results) == 12

results.to_csv(
    OUT / "compact_qualitative_metrics.csv",
    index=False
)

(OUT / "compact_figure_captions.json").write_text(
    json.dumps(captions, indent=2),
    encoding="utf-8"
)

caption_text = "\n\n".join(
    name + "\n" + caption
    for name, caption in captions.items()
)

(OUT / "compact_figure_captions.txt").write_text(
    caption_text,
    encoding="utf-8"
)

metadata = {
    "dataset": "OMBRIA",
    "partition": "official held-out test",
    "sample_ids": list(SAMPLES),
    "selection": "fixed previously chosen examples",
    "seed": 42,
    "conditions": [
        {"dx": dx, "dy": dy, "name": condition}
        for dx, dy, condition in CONDITIONS
    ],
    "threshold": THRESHOLD,
    "layout": "2 rows by 4 columns",
    "legend": "captions only",
    "figure_width_inches": FIGURE_WIDTH,
    "figure_height_inches": FIGURE_HEIGHT,
    "raster_dpi": DPI,
    "event_independence": "not verified",
    "metrics": "full-patch illustrative metrics"
}

(OUT / "compact_figure_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 12. VERIFY FINAL FILES
# ------------------------------------------------------------

for extension in ("svg", "pdf", "png", "tiff"):

    count = len(
        list(
            folders[extension].glob(
                "UA_FloodNet_Compact_*"
            )
        )
    )

    assert count == 6, (
        f"Expected 6 {extension} figures; found {count}"
    )

print("\n" + "=" * 72)
print("FINAL COMPACT FIGURES COMPLETED")
print("=" * 72)

print("Figures:", 6)
print("SVG:", 6)
print("PDF:", 6)
print("PNG:", 6)
print("TIFF:", 6)

print("\nCaptions:")
print(OUT / "compact_figure_captions.txt")

print("\nMetrics:")
print(OUT / "compact_qualitative_metrics.csv")

print("\nOutput directory:")
print(OUT)

print("\nFINAL SAMPLE METRICS")
print(
    results[
        ["sample_id", "condition", "model", "IoU", "F1"]
    ].round(5).to_string(index=False)
)


In [ ]:
# ============================================================
# STEP 82A — SAFE GOOGLE DRIVE REMOUNT
# ============================================================

from google.colab import drive
from pathlib import Path
import os

print("=" * 65)
print("UA-FLOODNET GOOGLE DRIVE RECOVERY")
print("=" * 65)

mount_locations = [
    Path("/content/ua_drive_mount"),
    Path("/content/drive"),
    Path("/content/ua_drive_mount_new"),
    Path("/content/ua_floodnet_drive")
]

ROOT = None

# Check whether an existing mount is accessible
for mount in mount_locations:
    candidate = mount / "MyDrive/UA_FloodNet"

    if candidate.is_dir():
        ROOT = candidate
        print("Existing Drive connection found:", mount)
        break

# Otherwise, create a fresh mount
if ROOT is None:

    mount = Path("/content/ua_drive_mount_82")

    if mount.exists() and any(mount.iterdir()):
        raise RuntimeError(
            "The proposed mount directory is not empty. "
            "Inspect it before mounting."
        )

    mount.mkdir(parents=True, exist_ok=True)

    print("\nMounting Google Drive...")
    drive.mount(str(mount))

    ROOT = mount / "MyDrive/UA_FloodNet"

# Verify project
assert ROOT.is_dir(), (
    "UA_FloodNet project folder not found. "
    "Check the Google account selected during authorization."
)

print("\nPROJECT VERIFICATION")
print("Project found:", ROOT.exists())
print("V4 experiments:", (ROOT / "v4").exists())
print("Final analysis:", (ROOT / "v4/final_analysis").exists())
print("Source code:", (ROOT / "v4/code/ablation_model.py").exists())

print("\nACTIVE PROJECT ROOT:")
print(ROOT)

print("\nDRIVE CONNECTION SUCCESSFUL")

In [ ]:

# ================================================================
# STEP 82 — COMPLETE UA-FLOODNET ARCHITECTURE GENERATOR
# SAFE GOOGLE DRIVE MOUNT + THREE SCIENTIFIC DIAGRAMS
# ================================================================

from pathlib import Path
import os
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import to_rgb
from IPython.display import display, Image as IPyImage
from google.colab import drive

print("=" * 72)
print("STEP 82: UA-FLOODNET ARCHITECTURE FIGURES")
print("=" * 72)

# ---------------------------------------------------------------
# 1. SAFELY LOCATE OR MOUNT GOOGLE DRIVE
# ---------------------------------------------------------------

mounts = [
    Path("/content/ua_drive_mount_82"),
    Path("/content/ua_drive_mount"),
    Path("/content/ua_drive_mount_new"),
    Path("/content/ua_floodnet_drive"),
    Path("/content/drive")
]

ROOT = None

for mount in mounts:
    candidate = mount / "MyDrive/UA_FloodNet"
    if candidate.is_dir():
        ROOT = candidate
        print("Existing project:", ROOT)
        break

if ROOT is None:
    for i in range(82, 100):
        mount = Path(f"/content/ua_drive_mount_{i}")
        if mount.exists() and any(mount.iterdir()):
            continue

        mount.mkdir(parents=True, exist_ok=True)
        print("Mounting Google Drive at:", mount)
        drive.mount(str(mount))
        candidate = mount / "MyDrive/UA_FloodNet"

        if candidate.is_dir():
            ROOT = candidate
            break

        raise FileNotFoundError(
            "UA_FloodNet project not found in the mounted "
            "Google account."
        )

assert ROOT is not None and ROOT.is_dir()

# ---------------------------------------------------------------
# 2. OUTPUT DIRECTORY AND EXPORT CONFIGURATION
# ---------------------------------------------------------------

OUT = (
    ROOT / "v4/final_analysis/"
    "manuscript_package_step81/architecture_final"
)
OUT.mkdir(parents=True, exist_ok=True)

DPI = int(os.getenv("UA_FLOODNET_DPI", "1200"))
PREVIEW_DPI = 125

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "savefig.facecolor": "white"
})

COLORS = {
    "sar": "#18B6E5",
    "opt": "#9753E8",
    "encoder": "#F77B35",
    "fusion": "#A15AE6",
    "change": "#FFD66A",
    "decoder": "#0BC681",
    "output": "#FFDD44",
    "neutral": "#E8EDF4",
    "arrow": "#258FE8",
    "edge": "#23384F",
    "muted": "#55697E"
}

print("Project:", ROOT)
print("Output:", OUT)
print("Requested DPI:", DPI)

# ---------------------------------------------------------------
# 3. VECTOR DRAWING HELPERS
# ---------------------------------------------------------------

def scaled_color(color, factor):
    values = to_rgb(color)
    return tuple(min(1, max(0, v * factor)) for v in values)


def cube(ax, x, y, w, h, color, title,
         subtitle="", depth=0.17, fontsize=8.3):

    front = Polygon(
        [(x, y), (x+w, y),
         (x+w, y+h), (x, y+h)],
        facecolor=color,
        edgecolor=COLORS["edge"],
        linewidth=1.0,
        zorder=3
    )
    top = Polygon(
        [(x, y+h),
         (x+depth, y+h+depth),
         (x+w+depth, y+h+depth),
         (x+w, y+h)],
        facecolor=scaled_color(color, 1.13),
        edgecolor=COLORS["edge"],
        linewidth=1.0,
        zorder=3
    )
    side = Polygon(
        [(x+w, y),
         (x+w+depth, y+depth),
         (x+w+depth, y+h+depth),
         (x+w, y+h)],
        facecolor=scaled_color(color, 0.76),
        edgecolor=COLORS["edge"],
        linewidth=1.0,
        zorder=3
    )

    for patch in (front, top, side):
        ax.add_patch(patch)

    ax.text(
        x+w/2, y-0.20, title,
        ha="center", va="top",
        fontsize=fontsize,
        fontweight="bold",
        color=COLORS["edge"]
    )

    if subtitle:
        ax.text(
            x+w/2, y-0.44, subtitle,
            ha="center", va="top",
            fontsize=fontsize-1,
            color=COLORS["muted"]
        )


def node(ax, x, y, w, h, text, color,
         fontsize=9, bold=False):

    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.025,rounding_size=0.10",
        facecolor=color,
        edgecolor="#647C95",
        linewidth=1.1,
        zorder=2
    )
    ax.add_patch(patch)

    ax.text(
        x+w/2, y+h/2, text,
        ha="center", va="center",
        fontsize=fontsize,
        fontweight="bold" if bold else "normal",
        linespacing=1.3,
        color=COLORS["edge"],
        zorder=4
    )


def link(ax, start, end, curved=0,
         dashed=False, linewidth=1.45):

    ax.add_patch(FancyArrowPatch(
        start, end,
        arrowstyle="-|>",
        mutation_scale=11,
        linewidth=linewidth,
        linestyle="--" if dashed else "-",
        color=COLORS["arrow"],
        connectionstyle=f"arc3,rad={curved}",
        zorder=1
    ))


def new_canvas(width=15, height=6):
    fig, ax = plt.subplots(figsize=(width, height))
    ax.set_xlim(0, 17)
    ax.set_ylim(0, 9)
    ax.axis("off")
    fig.subplots_adjust(
        left=0.015, right=0.985,
        bottom=0.04, top=0.98
    )
    return fig, ax


def export_figure(fig, filename):

    svg = OUT / f"{filename}.svg"
    pdf = OUT / f"{filename}.pdf"
    png = OUT / f"{filename}_{DPI}dpi.png"
    preview = OUT / f"{filename}_preview.png"

    fig.savefig(
        svg, format="svg",
        bbox_inches="tight",
        pad_inches=0.08
    )

    fig.savefig(
        pdf, format="pdf",
        bbox_inches="tight",
        pad_inches=0.08
    )

    fig.savefig(
        png, dpi=DPI,
        bbox_inches="tight",
        pad_inches=0.08
    )

    fig.savefig(
        preview, dpi=PREVIEW_DPI,
        bbox_inches="tight",
        pad_inches=0.08
    )

    plt.close(fig)

    print("\nGenerated:", filename)
    display(IPyImage(filename=str(preview)))


# ================================================================
# 4. FIGURE 1 — FULL UA-FLOODNET ARCHITECTURE
# ================================================================

fig, ax = new_canvas(16, 7)

ax.text(
    8.5, 8.60,
    "UA-FloodNet: Multitemporal SAR–Optical Architecture",
    ha="center", fontsize=15, fontweight="bold",
    color=COLORS["edge"]
)

inputs = [
    (0.45, 6.50, "SAR t0", "1 × 256²", "sar"),
    (0.45, 4.65, "Optical t0", "3 × 256²", "opt"),
    (0.45, 2.80, "SAR t1", "1 × 256²", "sar"),
    (0.45, 0.95, "Optical t1", "3 × 256²", "opt")
]

for x, y, label, sub, key in inputs:
    cube(
        ax, x, y, 0.75, 0.85,
        COLORS[key], label, sub
    )

encoders = [
    (2.25, 6.35, "SAR encoder t0\n24 → 48 → 96", "#FFDDCB"),
    (2.25, 4.50, "Optical encoder t0\n24 → 48 → 96", "#EBDEFF"),
    (2.25, 2.65, "SAR encoder t1\n24 → 48 → 96", "#FFDDCB"),
    (2.25, 0.80, "Optical encoder t1\n24 → 48 → 96", "#EBDEFF")
]

for x, y, label, color in encoders:
    node(ax, x, y, 2.75, 0.96, label, color, 9)

for sy, ty in [
    (6.93, 6.83),
    (5.08, 4.98),
    (3.23, 3.13),
    (1.38, 1.28)
]:
    link(ax, (1.43, sy), (2.20, ty))

node(
    ax, 5.70, 5.25, 2.70, 1.20,
    "Pre-flood AlignFusion\n"
    "3 scales; optical → SAR",
    "#E6D4FF", 9.7, True
)

node(
    ax, 5.70, 2.05, 2.70, 1.20,
    "Post-flood AlignFusion\n"
    "3 scales; optical → SAR",
    "#E6D4FF", 9.7, True
)

for start, end in [
    ((5.05, 6.83), (5.65, 6.00)),
    ((5.05, 4.98), (5.65, 5.65)),
    ((5.05, 3.13), (5.65, 2.80)),
    ((5.05, 1.28), (5.65, 2.35))
]:
    link(ax, start, end)

node(
    ax, 9.00, 3.70, 2.62, 1.45,
    "Three-scale temporal\n"
    "feature comparison\n"
    "F0, F1, |F1 − F0|",
    "#FFF0CB", 9.5, True
)

link(ax, (8.45, 5.85), (8.95, 4.72))
link(ax, (8.45, 2.65), (8.95, 4.05))

node(
    ax, 12.15, 3.70, 2.35, 1.45,
    "Multiscale decoder\n"
    "64² → 128² → 256²\n"
    "96 → 48 → 24",
    "#C7F5DD", 9.3, True
)

link(ax, (11.67, 4.42), (12.10, 4.42))

cube(
    ax, 15.30, 4.10, 0.62, 0.90,
    COLORS["output"],
    "Flood logits", "1 × 256²"
)

link(ax, (14.55, 4.42), (15.25, 4.42))

node(
    ax, 8.95, 0.70, 5.10, 1.05,
    "Training-only auxiliary branches\n"
    "InfoNCE, cross-reconstruction, flow smoothness",
    COLORS["neutral"], 9.2
)

link(
    ax, (7.10, 2.00), (8.90, 1.25),
    dashed=True, linewidth=1.1
)

ax.text(
    8.5, 0.18,
    "SAR and optical encoders use separate weights, "
    "each shared across acquisition dates.",
    ha="center", fontsize=8.7,
    color=COLORS["muted"]
)

export_figure(
    fig,
    "Figure_1_UA_FloodNet_Architecture"
)


# ================================================================
# 5. FIGURE 2 — ALIGNMENT AND RELIABILITY FUSION
# ================================================================

fig, ax = new_canvas(15.5, 6)

ax.text(
    8.5, 8.55,
    "Alignment and Reliability-Weighted Feature Fusion",
    ha="center", fontsize=15, fontweight="bold",
    color=COLORS["edge"]
)

cube(
    ax, 0.55, 6.35, 0.90, 1.05,
    COLORS["sar"],
    "SAR features", "C × H × W"
)

cube(
    ax, 0.55, 3.35, 0.90, 1.05,
    COLORS["opt"],
    "Optical features", "C × H × W"
)

node(
    ax, 2.45, 5.95, 2.55, 1.35,
    "SAR–optical concat\n"
    "flow_net\n"
    "Δmax × tanh",
    "#D9EFFD", 9.5
)

link(ax, (1.66, 6.88), (2.40, 6.64))
link(ax, (1.65, 3.87), (2.40, 6.15))

node(
    ax, 5.55, 5.55, 2.62, 1.55,
    "Optical feature sampling\n"
    "normalized grid\n"
    "grid_sample\n"
    "border padding",
    "#E7D9FD", 9.3
)

link(ax, (5.06, 6.60), (5.50, 6.40))
link(ax, (1.70, 3.68), (5.50, 5.90), curved=-0.08)

node(
    ax, 8.65, 5.55, 2.58, 1.55,
    "logvar_net\n"
    "clamp(−6, 6)\n"
    "softmax(−logvars)\n"
    "sensor weights",
    "#FFF0CE", 9.3
)

link(ax, (8.22, 6.33), (8.60, 6.33))

node(
    ax, 11.70, 5.55, 2.45, 1.55,
    "Weighted SAR + MS\n"
    "feature concatenation\n"
    "convolutional mix",
    "#CFF4DF", 9.5
)

link(ax, (11.28, 6.33), (11.65, 6.33))

cube(
    ax, 15.10, 5.80, 0.65, 0.92,
    COLORS["fusion"],
    "Fused feature", "C × H × W"
)

link(ax, (14.20, 6.33), (15.04, 6.33))

node(
    ax, 4.25, 2.20, 8.55, 1.15,
    "One AlignFusion module is applied at each feature "
    "scale and reused across both dates.",
    COLORS["neutral"], 10
)

ax.text(
    8.5, 1.25,
    "Flow is a feature-grid sampling offset, not a physical "
    "surface-displacement measurement.",
    ha="center", fontsize=9.0,
    color=COLORS["muted"]
)

ax.text(
    8.5, 0.72,
    "Reliability weights are learned log-variance proxies; "
    "their uncertainty calibration has not been established.",
    ha="center", fontsize=9.0,
    color=COLORS["muted"]
)

export_figure(
    fig,
    "Figure_2_AlignFusion_Module"
)


# ================================================================
# 6. FIGURE 3 — TEMPORAL DECODER AND TRAINING LOSS
# ================================================================

fig, ax = new_canvas(15.5, 6)

ax.text(
    8.5, 8.55,
    "Multiscale Temporal Decoder and Auxiliary Objectives",
    ha="center", fontsize=15, fontweight="bold",
    color=COLORS["edge"]
)

features = [
    (0.65, 6.25, "Deep features", "96 channels"),
    (0.65, 4.08, "Middle features", "48 channels"),
    (0.65, 1.91, "Fine features", "24 channels")
]

for x, y, label, sub in features:
    cube(
        ax, x, y, 0.80, 1.05,
        COLORS["fusion"], label, sub
    )

node(
    ax, 2.55, 6.10, 2.75, 1.20,
    "Deep temporal concat\n"
    "F0³, F1³, |ΔF³|\n"
    "288 channels",
    "#FFF0D3", 9.3
)

node(
    ax, 6.05, 6.10, 2.78, 1.20,
    "Deep temporal block\n"
    "64² × 96",
    "#CEF4DE", 9.5
)

link(ax, (1.75, 6.79), (2.50, 6.70))
link(ax, (5.35, 6.70), (6.00, 6.70))

node(
    ax, 6.05, 4.03, 2.78, 1.20,
    "Upsample + decode2\n"
    "128² × 48",
    "#CEF4DE", 9.5
)

link(ax, (7.45, 6.05), (7.45, 5.28))
link(ax, (1.75, 4.60), (6.00, 4.62), curved=0.12)

node(
    ax, 6.05, 1.96, 2.78, 1.20,
    "Upsample + decode1\n"
    "256² × 24",
    "#CEF4DE", 9.5
)

link(ax, (7.45, 3.99), (7.45, 3.21))
link(ax, (1.75, 2.48), (6.00, 2.55), curved=0.12)

cube(
    ax, 10.20, 2.14, 0.75, 1.00,
    COLORS["output"],
    "Flood logits", "1 × 256²"
)

link(ax, (8.88, 2.55), (10.15, 2.65))

node(
    ax, 12.0, 2.10, 3.10, 1.23,
    "Sigmoid probability\n"
    "threshold = 0.5\n"
    "binary flood map",
    "#FFEDD0", 9.5
)

link(ax, (11.18, 2.65), (11.95, 2.65))

node(
    ax, 10.2, 5.63, 5.40, 1.55,
    "TRAINING LOSS\n"
    "BCE + Dice + 0.1 InfoNCE\n"
    "+ 0.1 reconstruction MSE\n"
    "+ 0.01 flow smoothness",
    COLORS["neutral"], 10, True
)

link(
    ax, (10.75, 5.58), (10.75, 3.22),
    dashed=True, linewidth=1.1
)

ax.text(
    8.5, 0.82,
    "The auxiliary objectives are training regularizers. "
    "Only the flood segmentation output is required "
    "during inference.",
    ha="center", fontsize=9,
    color=COLORS["muted"]
)

export_figure(
    fig,
    "Figure_3_Temporal_Decoder"
)


# ================================================================
# 7. SAVE MANUSCRIPT FIGURE CAPTIONS
# ================================================================

captions = {
    "Figure 1": (
        "UA-FloodNet architecture integrating multitemporal "
        "Sentinel-1 SAR and three-channel Sentinel-2 optical "
        "composites. The model employs modality-specific "
        "encoders with temporal weight sharing, three-scale "
        "same-date optical-to-SAR fusion, temporal feature "
        "comparison and a hierarchical segmentation decoder."
    ),
    "Figure 2": (
        "Detailed alignment and reliability-fusion mechanism. "
        "A learned optical-to-SAR sampling field is applied "
        "through grid_sample, while softmax-normalized "
        "log-variance proxies determine modality-specific "
        "feature weights prior to convolutional fusion."
    ),
    "Figure 3": (
        "Multiscale temporal decoder and training objectives. "
        "Pre-event and post-event fused features and their "
        "absolute differences are combined through coarse-to-fine "
        "decoding. Training incorporates segmentation, "
        "contrastive, reconstruction and flow-smoothness losses."
    )
}

(OUT / "architecture_figure_captions.json").write_text(
    json.dumps(captions, indent=2),
    encoding="utf-8"
)

(OUT / "architecture_figure_captions.txt").write_text(
    "\n\n".join(
        f"{key}. {value}"
        for key, value in captions.items()
    ),
    encoding="utf-8"
)


# ================================================================
# 8. FINAL VERIFICATION
# ================================================================

names = [
    "Figure_1_UA_FloodNet_Architecture",
    "Figure_2_AlignFusion_Module",
    "Figure_3_Temporal_Decoder"
]

print("\n" + "=" * 72)
print("STEP 82 COMPLETED")
print("=" * 72)

for name in names:
    print(
        name,
        "| SVG:", (OUT / f"{name}.svg").exists(),
        "| PDF:", (OUT / f"{name}.pdf").exists(),
        "| PNG:", (OUT / f"{name}_{DPI}dpi.png").exists()
    )

print("\nAll architecture figures saved to:")
print(OUT)


In [ ]:

# ================================================================
# STEP 83 — FINAL QUANTITATIVE FIGURES FOR UA-FLOODNET
# ================================================================
# Figure 7: Misregistration sensitivity, 4 training seeds
# Figure 8: Absolute IoU and paired statistical uncertainty
#
# INPUTS:
#   Step 71 — candidate-validation evaluation
#   Step 74 — official held-out-test evaluation
#
# OUTPUTS:
#   Figure 7: SVG, PDF, PNG, TIFF
#   Figure 8: SVG, PDF, PNG, TIFF
#   Recomputed statistical summary CSV
#   Figure captions and reproducibility metadata
#
# Data are read from the user's saved experimental CSVs.
# No model training, predictions or test-set tuning.
# ================================================================

from pathlib import Path
import os
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as student_t
from IPython.display import display, Image as IPyImage
from google.colab import drive

# ------------------------------------------------
# 1. Locate or mount Google Drive
# ------------------------------------------------

mounts = [
    Path("/content/ua_drive_mount_82"),
    Path("/content/ua_drive_mount"),
    Path("/content/ua_drive_mount_new"),
    Path("/content/ua_floodnet_drive"),
    Path("/content/drive")
]

ROOT = None

for mount in mounts:
    candidate = mount / "MyDrive/UA_FloodNet"
    if candidate.is_dir():
        ROOT = candidate
        break

if ROOT is None:
    mount = Path("/content/ua_step83_drive")
    if mount.exists() and any(mount.iterdir()):
        raise RuntimeError(
            "Mount path contains files but the project "
            "was not found. Inspect the mount."
        )
    mount.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mount))
    ROOT = mount / "MyDrive/UA_FloodNet"

assert ROOT.is_dir(), (
    "UA_FloodNet was not found. Check the Google "
    "account selected during Drive authorization."
)

BASE = ROOT / "v4"
OUT = (
    BASE / "final_analysis"
    / "manuscript_package_step81"
    / "quantitative_figures_final"
)

OUT.mkdir(parents=True, exist_ok=True)

DPI = int(os.getenv("UA_FLOODNET_DPI", "1200"))
PREVIEW_DPI = 170

print("=" * 70)
print("STEP 83 — UA-FLOODNET QUANTITATIVE FIGURES")
print("=" * 70)
print("Project:", ROOT)
print("Output:", OUT)

# ------------------------------------------------
# 2. Locate the saved experiment results
# ------------------------------------------------

def locate_csv(folder, exact_name, required, prefix):
    """
    Find CSVs by filename and verify their data schema.
    Never substitute simulated or invented results.
    """
    if not folder.is_dir():
        raise FileNotFoundError(folder)

    candidate_paths = []
    exact = folder / exact_name

    if exact.is_file():
        candidate_paths.append(exact)

    candidate_paths.extend(
        p for p in sorted(folder.glob(f"{prefix}*.csv"))
        if p not in candidate_paths
    )

    valid = []

    for path in candidate_paths:
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        df.columns = [
            str(col).strip() for col in df.columns
        ]

        if required.issubset(set(df.columns)):
            valid.append((path, df))

    if not valid:
        print("\nAvailable files in:", folder)
        for path in sorted(folder.glob("*.csv")):
            print(" ", path.name)

        raise FileNotFoundError(
            f"No {prefix} CSV has the required fields "
            f"{sorted(required)}."
        )

    # Prefer exact expected name when valid.
    return valid[0]


val_dir = BASE / "robustness/multiseed"
test_dir = BASE / "official_test"

summary_fields = {
    "seed", "variant", "clean_iou",
    "mean_shifted_iou", "mean_iou_loss"
}

val_summary_path, val_summary = locate_csv(
    val_dir,
    "step71_seed_summary.csv",
    summary_fields,
    "step71"
)

test_summary_path, test_summary = locate_csv(
    test_dir,
    "step74_frozen_test_summary.csv",
    summary_fields,
    "step74"
)

# Long results may use IoU or iou.
def locate_shift_results(folder, exact_name, prefix):
    candidates = []

    exact = folder / exact_name
    if exact.is_file():
        candidates.append(exact)

    candidates += [
        p for p in sorted(folder.glob(f"{prefix}*.csv"))
        if p not in candidates
    ]

    for path in candidates:
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        df.columns = [
            str(c).strip() for c in df.columns
        ]

        rename = {
            col: "iou"
            for col in df.columns
            if col.lower() == "iou"
        }

        df = df.rename(columns=rename)

        needed = {
            "seed", "variant", "dx", "dy", "iou"
        }

        if needed.issubset(df.columns):
            return path, df

    raise FileNotFoundError(
        f"Displacement-level {prefix} CSV not found "
        f"with seed, variant, dx, dy and IoU fields."
    )


val_long_path, val_long = locate_shift_results(
    val_dir,
    "step71_all_shift_results.csv",
    "step71"
)

test_long_path, test_long = locate_shift_results(
    test_dir,
    "step74_frozen_test_results.csv",
    "step74"
)

print("\nVerified input files:")

for name, path in [
    ("Validation summary", val_summary_path),
    ("Test summary", test_summary_path),
    ("Validation shifts", val_long_path),
    ("Test shifts", test_long_path)
]:
    print(name, ":", path.name)

# ------------------------------------------------
# 3. Strict scientific validation
# ------------------------------------------------

SEEDS = [7, 17, 37, 42]
VARIANTS = ["control", "light"]

SHIFTS = [
    (0, 0),
    (-1, 0), (1, 0),
    (-2, 0), (2, 0),
    (-4, 0), (4, 0),
    (-8, 0), (8, 0),
    (0, -2), (0, 2),
    (0, -4), (0, 4)
]

def validate_summary(df, name):
    expected = {
        (s, v)
        for s in SEEDS
        for v in VARIANTS
    }

    actual = set(
        zip(df.seed.astype(int), df.variant)
    )

    assert actual == expected, (
        f"{name} seed–variant combinations are incorrect."
    )

    assert len(df) == 8
    assert not df.duplicated(["seed", "variant"]).any()

    for field in [
        "clean_iou",
        "mean_shifted_iou",
        "mean_iou_loss"
    ]:
        assert df[field].between(0, 1).all(), (
            f"{name}: invalid values for {field}"
        )

    error = (
        df.clean_iou
        - df.mean_shifted_iou
        - df.mean_iou_loss
    ).abs().max()

    assert error < 2e-4, (
        f"{name} degradation values are inconsistent."
    )


def validate_long(df, name):
    expected = {
        (s, v, dx, dy)
        for s in SEEDS
        for v in VARIANTS
        for dx, dy in SHIFTS
    }

    actual = set(
        zip(
            df.seed.astype(int),
            df.variant,
            df.dx.astype(int),
            df.dy.astype(int)
        )
    )

    assert actual == expected, (
        f"{name} displacement combinations do not match "
        "the predefined 13-condition experiment."
    )

    assert len(df) == 104
    assert not df.duplicated(
        ["seed", "variant", "dx", "dy"]
    ).any()

    assert df.iou.between(0, 1).all()


validate_summary(val_summary, "Validation")
validate_summary(test_summary, "Official test")

validate_long(val_long, "Validation")
validate_long(test_long, "Official test")

# Match summary clean scores against displacement-level
# clean scores; crop-based analysis may differ from
# checkpoint-selection metrics.
for summary, long_df, name in [
    (val_summary, val_long, "Validation"),
    (test_summary, test_long, "Official test")
]:
    clean = long_df[
        (long_df.dx == 0) &
        (long_df.dy == 0)
    ][["seed", "variant", "iou"]]

    merged = summary.merge(
        clean,
        on=["seed", "variant"],
        validate="one_to_one"
    )

    assert np.allclose(
        merged.clean_iou,
        merged.iou,
        atol=2e-4
    ), (
        f"{name}: clean IoU mismatch between result files."
    )

print("\nDATA AUDIT PASSED")
print("Training seeds: 4")
print("Model variants: 2")
print("Displacement conditions: 13")
print("Validation records: 104")
print("Official test records: 104")

# ------------------------------------------------
# 4. Publication-quality appearance
# ------------------------------------------------

CONTROL = "#1673B8"
LIGHT = "#10B88C"
DARK = "#18324A"
GRID = "#DFE8EF"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.edgecolor": "#637B8E",
    "axes.linewidth": 0.75,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "savefig.facecolor": "white"
})

def panel(ax, letter):
    ax.text(
        -0.10, 1.045,
        f"({letter})",
        transform=ax.transAxes,
        weight="bold",
        fontsize=11,
        color=DARK
    )

def finish_axes(ax):
    ax.grid(
        axis="y",
        color=GRID,
        linewidth=0.7
    )
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def export(fig, name):
    """Save vector masters and 1200-dpi raster exports."""
    paths = {
        "svg": OUT / f"{name}.svg",
        "pdf": OUT / f"{name}.pdf",
        "png": OUT / f"{name}_{DPI}dpi.png",
        "tiff": OUT / f"{name}_{DPI}dpi.tiff",
    }

    for fmt in ("svg", "pdf"):
        fig.savefig(
            paths[fmt],
            format=fmt,
            bbox_inches="tight",
            pad_inches=0.08
        )

    fig.savefig(
        paths["png"],
        format="png",
        dpi=DPI,
        bbox_inches="tight",
        pad_inches=0.08
    )

    fig.savefig(
        paths["tiff"],
        format="tiff",
        dpi=DPI,
        pil_kwargs={"compression": "tiff_lzw"},
        bbox_inches="tight",
        pad_inches=0.08
    )

    preview = OUT / f"{name}_preview.png"

    fig.savefig(
        preview,
        dpi=PREVIEW_DPI,
        bbox_inches="tight",
        pad_inches=0.08
    )

    plt.close(fig)

    print("\nGenerated:", name)
    display(IPyImage(filename=str(preview)))

    for fmt, path in paths.items():
        assert path.is_file() and path.stat().st_size > 0
        print(fmt.upper(), ":", path.name)


# ================================================================
# FIGURE 7 — MISREGISTRATION SENSITIVITY
# ================================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(10.8, 7.2),
    gridspec_kw={
        "height_ratios": [1.17, 0.9]
    }
)

fig.subplots_adjust(
    left=0.09,
    right=0.98,
    top=0.97,
    bottom=0.10,
    wspace=0.23,
    hspace=0.40
)

def draw_profile(ax, df, title):
    x = np.arange(len(SHIFTS))

    for variant, color, marker, label in [
        ("control", CONTROL, "o", "Control"),
        ("light", LIGHT, "s", "Light augmentation")
    ]:
        subset = df[df.variant == variant]

        means = (
            subset.groupby(["dx", "dy"])
            .iou.mean()
        )

        values = [
            means.loc[(dx, dy)]
            for dx, dy in SHIFTS
        ]

        ax.plot(
            x,
            values,
            marker=marker,
            markersize=3.4,
            linewidth=1.6,
            color=color,
            label=label
        )

    ax.set_title(title, fontsize=10, weight="bold")
    ax.set_ylabel("Flood IoU")
    ax.set_xticks(x)

    ax.set_xticklabels(
        [f"{dx:+d},{dy:+d}" for dx, dy in SHIFTS],
        rotation=65,
        ha="right",
        fontsize=7
    )

    ax.set_xlabel("Optical displacement (dx, dy), pixels")
    ax.legend(loc="best", frameon=False, fontsize=8)
    finish_axes(ax)


def draw_degradation(ax, df, title):
    pivot = (
        df.pivot(
            index="seed",
            columns="variant",
            values="mean_iou_loss"
        )
        .loc[SEEDS]
    )

    x = np.arange(len(SEEDS))
    bar_width = 0.34

    ax.bar(
        x - bar_width / 2,
        pivot["control"],
        bar_width,
        color=CONTROL,
        label="Control"
    )

    ax.bar(
        x + bar_width / 2,
        pivot["light"],
        bar_width,
        color=LIGHT,
        label="Light augmentation"
    )

    ax.set_xticks(x, SEEDS)
    ax.set_xlabel("Training seed")
    ax.set_ylabel("Clean IoU − mean shifted IoU")
    ax.set_title(title, fontsize=10, weight="bold")
    ax.legend(frameon=False, fontsize=8)
    finish_axes(ax)


draw_profile(
    axes[0, 0],
    val_long,
    "Candidate validation"
)

draw_profile(
    axes[0, 1],
    test_long,
    "Official held-out test"
)

draw_degradation(
    axes[1, 0],
    val_summary,
    "Validation: seed-wise degradation"
)

draw_degradation(
    axes[1, 1],
    test_summary,
    "Held-out test: seed-wise degradation"
)

for ax, letter in zip(
    axes.flat,
    "abcd"
):
    panel(ax, letter)

export(
    fig,
    "Figure_7_Misregistration_Sensitivity"
)


# ================================================================
# FIGURE 8 — ACCURACY AND STATISTICAL UNCERTAINTY
# ================================================================

# Use the four matched training seeds as experimental
# replicates. The CI and p-values are exploratory because
# the training seeds do not establish event independence.

def paired_statistics(df, partition):
    pivot = df.pivot(
        index="seed",
        columns="variant",
        values=[
            "clean_iou",
            "mean_shifted_iou",
            "mean_iou_loss"
        ]
    ).loc[SEEDS]

    rows = []

    for metric in [
        "clean_iou",
        "mean_shifted_iou",
        "mean_iou_loss"
    ]:
        control = pivot[(metric, "control")].to_numpy()
        light = pivot[(metric, "light")].to_numpy()

        differences = light - control
        n = len(differences)

        mean_diff = float(differences.mean())
        sd = float(differences.std(ddof=1))
        se = sd / np.sqrt(n)

        critical = student_t.ppf(
            0.975,
            df=n-1
        )

        ci_lower = mean_diff - critical * se
        ci_upper = mean_diff + critical * se

        if se == 0:
            pvalue = 1.0 if mean_diff == 0 else 0.0
        else:
            statistic = mean_diff / se
            pvalue = float(
                2 * student_t.sf(abs(statistic), n-1)
            )

        rows.append({
            "partition": partition,
            "metric": metric,
            "n_seeds": n,
            "control_mean": float(control.mean()),
            "light_mean": float(light.mean()),
            "mean_difference": mean_diff,
            "sd_difference": sd,
            "ci95_lower": ci_lower,
            "ci95_upper": ci_upper,
            "exploratory_p": pvalue
        })

    return pd.DataFrame(rows)


stats = pd.concat(
    [
        paired_statistics(val_summary, "validation"),
        paired_statistics(test_summary, "official_test")
    ],
    ignore_index=True
)

stats.to_csv(
    OUT / "step83_verified_paired_statistics.csv",
    index=False
)

fig, axes = plt.subplots(
    2, 2,
    figsize=(10.4, 7.1)
)

fig.subplots_adjust(
    left=0.11,
    right=0.97,
    top=0.97,
    bottom=0.12,
    hspace=0.36,
    wspace=0.30
)

def draw_absolute_iou(ax, df, title):
    group = (
        df.groupby("variant")
        [["clean_iou", "mean_shifted_iou"]]
        .mean()
        .loc[VARIANTS]
    )

    x = np.arange(2)
    w = 0.34

    bars1 = ax.bar(
        x - w / 2,
        group.clean_iou,
        w,
        color=CONTROL,
        label="Clean"
    )

    bars2 = ax.bar(
        x + w / 2,
        group.mean_shifted_iou,
        w,
        color=LIGHT,
        label="Mean shifted"
    )

    ax.set_xticks(
        x,
        ["Control", "Light augmentation"]
    )

    ax.set_ylim(0.60, 0.73)
    ax.set_ylabel("Mean flood IoU")
    ax.set_title(title, fontsize=10, weight="bold")
    ax.legend(frameon=False, fontsize=8)

    for bar in [*bars1, *bars2]:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f"{bar.get_height():.4f}",
            ha="center",
            fontsize=7.3
        )

    finish_axes(ax)


def draw_confidence_intervals(ax, df, title):
    labels = [
        "Clean IoU",
        "Mean shifted IoU",
        "IoU degradation"
    ]

    metric_names = [
        "clean_iou",
        "mean_shifted_iou",
        "mean_iou_loss"
    ]

    table = (
        df.set_index("metric")
        .loc[metric_names]
    )

    means = table.mean_difference.to_numpy()
    lower = table.ci95_lower.to_numpy()
    upper = table.ci95_upper.to_numpy()

    x = np.arange(3)

    ax.errorbar(
        x,
        means,
        yerr=np.vstack([
            means - lower,
            upper - means
        ]),
        fmt="o",
        markersize=5,
        capsize=5,
        color=CONTROL,
        ecolor=CONTROL,
        linewidth=1.7
    )

    ax.axhline(
        0,
        linestyle="--",
        color="#687989",
        linewidth=0.9
    )

    ax.set_xticks(
        x,
        labels,
        rotation=20,
        ha="right"
    )

    ax.set_ylabel("Light − control")
    ax.set_title(title, fontsize=10, weight="bold")
    finish_axes(ax)


draw_absolute_iou(
    axes[0, 0],
    val_summary,
    "Candidate validation"
)

draw_absolute_iou(
    axes[0, 1],
    test_summary,
    "Official held-out test"
)

draw_confidence_intervals(
    axes[1, 0],
    stats[stats.partition == "validation"],
    "Validation: paired effects (95% CI)"
)

draw_confidence_intervals(
    axes[1, 1],
    stats[stats.partition == "official_test"],
    "Held-out test: paired effects (95% CI)"
)

for ax, letter in zip(
    axes.flat,
    "abcd"
):
    panel(ax, letter)

export(
    fig,
    "Figure_8_Accuracy_Robustness_Uncertainty"
)


# ================================================================
# 5. MANUSCRIPT CAPTIONS
# ================================================================

captions = {

    "Figure 7": (
        "Sensitivity of UA-FloodNet to synthetic SAR–optical "
        "misregistration. Panels (a) and (b) show mean pooled "
        "flood intersection-over-union (IoU) across 13 optical "
        "displacement conditions for candidate validation "
        "and the official held-out test, respectively. "
        "Panels (c) and (d) compare IoU degradation "
        "(clean minus mean shifted IoU) between the "
        "unaugmented control and light-augmentation models "
        "for four matched training seeds (7, 17, 37, 42). "
        "The optical pre-event and post-event inputs receive "
        "the same synthetic displacement; SAR inputs and "
        "reference masks remain unchanged."
    ),

    "Figure 8": (
        "Absolute flood segmentation performance and "
        "paired statistical uncertainty. Panels (a) and (b) "
        "show four-seed mean clean IoU and mean IoU over "
        "the 12 nonzero displacement conditions for "
        "candidate validation and the official held-out "
        "test, respectively. Panels (c) and (d) present "
        "the corresponding paired differences "
        "(light augmentation minus control), with "
        "exploratory 95% Student's t confidence intervals "
        "computed across four matched training seeds. "
        "Intervals quantify variability across model seeds "
        "and do not establish geographic or flood-event "
        "independence."
    )
}

(OUT / "step83_figure_captions.json").write_text(
    json.dumps(captions, indent=2),
    encoding="utf-8"
)

(OUT / "step83_figure_captions.txt").write_text(
    "\n\n".join(
        f"{name}. {caption}"
        for name, caption in captions.items()
    ),
    encoding="utf-8"
)

# ------------------------------------------------
# 6. REPRODUCIBILITY RECORD
# ------------------------------------------------

import hashlib

sources = [
    val_summary_path,
    test_summary_path,
    val_long_path,
    test_long_path
]

manifest = {
    "project": "UA-FloodNet",
    "step": 83,
    "figure_numbers": [7, 8],
    "training_seeds": SEEDS,
    "n_training_seeds": 4,
    "n_displacement_conditions": 13,
    "n_nonzero_conditions": 12,
    "test_samples": 70,
    "test_independence": "event independence not verified",
    "statistical_scope": "exploratory paired seed-level t intervals",
    "raster_dpi": DPI,
    "source_files": {
        str(path.relative_to(ROOT)): hashlib.sha256(
            path.read_bytes()
        ).hexdigest()
        for path in sources
    }
}

(OUT / "step83_reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------
# 7. REPORT RESULTS AND VERIFY EXPORTS
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 83 — EXPERIMENTAL RESULT SUMMARY")
print("=" * 70)

for name, df in [
    ("Validation", val_summary),
    ("Official test", test_summary)
]:
    print("\n", name)

    print(
        df.groupby("variant")
        [["clean_iou", "mean_shifted_iou", "mean_iou_loss"]]
        .mean()
        .round(6)
        .to_string()
    )

print("\nPAIRED STATISTICS")
print(
    stats[
        [
            "partition",
            "metric",
            "mean_difference",
            "ci95_lower",
            "ci95_upper",
            "exploratory_p"
        ]
    ].round(6).to_string(index=False)
)

for name in [
    "Figure_7_Misregistration_Sensitivity",
    "Figure_8_Accuracy_Robustness_Uncertainty"
]:
    for fmt in ("svg", "pdf"):
        assert (OUT / f"{name}.{fmt}").exists()

    for fmt in ("png", "tiff"):
        assert (OUT / f"{name}_{DPI}dpi.{fmt}").exists()

print("\nSTEP 83 COMPLETED")
print("Figures generated: 2")
print("Output directory:", OUT)
print("Caption file: step83_figure_captions.txt")
print("Statistical file: step83_verified_paired_statistics.csv")


In [ ]:

# ============================================================
# STEP 84 — UA-FLOODNET FINAL MANUSCRIPT TABLES
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from IPython.display import display
from google.colab import drive

# Word and Excel outputs
import subprocess, sys, importlib.util

for package, module in [
    ("python-docx", "docx"),
    ("openpyxl", "openpyxl")
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip",
            "install", "-q", package
        ])

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

print("=" * 75)
print("STEP 84 — FINAL MANUSCRIPT TABLE GENERATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. FIND PROJECT
# ------------------------------------------------------------

mount_paths = [
    "/content/ua_drive_mount_82",
    "/content/ua_drive_mount",
    "/content/ua_drive_mount_new",
    "/content/ua_floodnet_drive",
    "/content/ua_step83_drive",
    "/content/drive"
]

ROOT = None

for path in mount_paths:
    candidate = Path(path) / "MyDrive/UA_FloodNet"
    if candidate.is_dir():
        ROOT = candidate
        break

if ROOT is None:
    mount = Path("/content/ua_step84_drive")
    if mount.exists() and any(mount.iterdir()):
        raise RuntimeError("Mount directory is not empty.")
    mount.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mount))
    ROOT = mount / "MyDrive/UA_FloodNet"

assert ROOT.is_dir(), "UA_FloodNet project not found."

BASE = ROOT / "v4"

OUT = (
    BASE / "final_analysis"
    / "manuscript_package_step81"
    / "tables_final"
)

OUT.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Output:", OUT)

# ------------------------------------------------------------
# 2. LOCATE ACTUAL EXPERIMENT TABLES
# ------------------------------------------------------------

REQUIRED = {
    "seed", "variant", "clean_iou",
    "mean_shifted_iou", "mean_iou_loss",
    "worst_shifted_iou"
}

def find_summary(directory, prefix):
    candidates = sorted(directory.glob(f"{prefix}*.csv"))

    for path in candidates:
        try:
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()

            if REQUIRED.issubset(df.columns):
                pairs = set(zip(
                    df.seed.astype(int),
                    df.variant.astype(str).str.lower()
                ))

                expected = {
                    (seed, variant)
                    for seed in (7, 17, 37, 42)
                    for variant in ("control", "light")
                }

                if pairs == expected and len(df) == 8:
                    return path, df
        except Exception:
            continue

    raise FileNotFoundError(
        f"Matching validated summary CSV not found in {directory}"
    )

VAL_PATH, VAL = find_summary(
    BASE / "robustness/multiseed", "step71"
)

TEST_PATH, TEST = find_summary(
    BASE / "official_test", "step74"
)

print("\nValidation:", VAL_PATH.name)
print("Official test:", TEST_PATH.name)

for frame in (VAL, TEST):
    frame["variant"] = frame.variant.str.lower()
    frame["seed"] = frame.seed.astype(int)
    frame.sort_values(["seed", "variant"], inplace=True)

    if frame.duplicated(["seed", "variant"]).any():
        raise AssertionError("Duplicate seed/model results.")

    assert np.allclose(
        frame.clean_iou - frame.mean_shifted_iou,
        frame.mean_iou_loss,
        atol=0.00002
    ), "IoU degradation does not match clean minus shifted."

# ------------------------------------------------------------
# 3. COLLECT TABLES
# ------------------------------------------------------------

tables = {}
captions = {}

# TABLE 1 — DATASET

manifest = (
    ROOT / "dataset_backup/candidate_split_seed42.csv"
)

assert manifest.is_file(), "Frozen split manifest missing."

split_df = pd.read_csv(manifest)

assert "split" in split_df.columns

counts = split_df.split.value_counts()

assert all(
    int(counts.get(s, 0)) == n
    for s, n in [
        ("train", 499),
        ("val", 125),
        ("test", 70)
    ]
)

tables["Table_1"] = pd.DataFrame([
    ["Training", 499, "Source training partition"],
    ["Candidate validation", 125, "Seed-42 patch-level split"],
    ["Official held-out test", 70, "Original source test partition"],
    ["Total", 694, "All paired OMBRIA patches"]
], columns=[
    "Partition", "Paired patches", "Specification"
])

captions["Table_1"] = (
    "OMBRIA dataset partitioning. Each paired patch contains "
    "pre- and post-event Sentinel-1 SAR, three-channel optical "
    "composites and a binary reference mask. Image dimensions "
    "are 256 × 256 pixels. Flood-event independence has not "
    "been verified."
)

# TABLE 2 — CONFIGURATION

tables["Table_2"] = pd.DataFrame([
    ["Architecture", "UA-FloodNet full", "Both configurations"],
    ["Encoder base width", "24", "Both configurations"],
    ["Feature channels", "24, 48, 96", "Three encoder scales"],
    ["Input dimensions", "256 × 256", "All paired samples"],
    ["Maximum training epochs", "50", "All runs"],
    ["Batch size", "4", "All runs"],
    ["Initial learning rate", "0.001", "Both configurations"],
    ["Optimizer", "AdamW", "Both configurations"],
    ["Model seeds", "7, 17, 37, 42", "Paired experiments"],
    ["Control shift probability", "0.00", "No augmentation"],
    ["Light shift probability", "0.25", "Train-time augmentation"],
    ["Light maximum shift", "2 pixels", "Horizontal and vertical"],
    ["Segmentation threshold", "0.50", "Evaluation"],
    ["Contrastive loss weight", "0.10", "Training"],
    ["Reconstruction loss weight", "0.10", "Training"],
    ["Flow smoothness weight", "0.01", "Training"]
], columns=[
    "Parameter", "Value", "Application"
])

captions["Table_2"] = (
    "UA-FloodNet architecture and training specifications. "
    "Both experimental configurations use the same full "
    "architecture; light augmentation applies random optical "
    "translations during training. Values summarize the "
    "previously audited training configuration."
)

# TABLES 3 AND 4 — SEED-WISE RESULTS

def performance_table(df):
    result = df[[
        "seed", "variant", "clean_iou",
        "mean_shifted_iou", "mean_iou_loss",
        "worst_shifted_iou"
    ]].copy()

    result.columns = [
        "Seed", "Configuration", "Clean IoU",
        "Shifted IoU", "IoU degradation",
        "Worst-shift IoU"
    ]

    result["Configuration"] = (
        result["Configuration"]
        .map({"control": "Control", "light": "Light"})
    )

    return result.sort_values(
        ["Seed", "Configuration"]
    ).reset_index(drop=True)

tables["Table_3"] = performance_table(VAL)
tables["Table_4"] = performance_table(TEST)

captions["Table_3"] = (
    "Four-seed performance on the candidate validation "
    "partition. Shifted IoU is averaged over twelve nonzero "
    "simulated optical displacement conditions."
)

captions["Table_4"] = (
    "Four-seed performance on the official OMBRIA held-out "
    "test partition. Clean, shifted and worst-displacement "
    "IoU values are evaluated using matched configurations."
)

# TABLE 5 — PAIRED STATISTICAL INFERENCE

def infer_paired(df, partition):
    pivot = df.pivot(
        index="seed",
        columns="variant",
        values=[
            "clean_iou", "mean_shifted_iou",
            "mean_iou_loss", "worst_shifted_iou"
        ]
    ).loc[[7,17,37,42]]

    metric_map = [
        ("clean_iou", "Clean IoU"),
        ("mean_shifted_iou", "Mean shifted IoU"),
        ("mean_iou_loss", "IoU degradation"),
        ("worst_shifted_iou", "Worst-shifted IoU")
    ]

    records = []

    for field, label in metric_map:
        difference = (
            pivot[(field, "light")] -
            pivot[(field, "control")]
        ).to_numpy(dtype=float)

        n = len(difference)
        mean = difference.mean()
        sd = difference.std(ddof=1)
        sem = sd / np.sqrt(n)
        critical = student_t.ppf(.975, n-1)

        lower = mean - critical * sem
        upper = mean + critical * sem

        p_value = (
            float(
                2 * student_t.sf(abs(mean/sem), n-1)
            )
            if sem > 0 else np.nan
        )

        records.append([
            partition, label, mean, lower, upper, p_value
        ])

    return records

stat_rows = (
    infer_paired(VAL, "Validation") +
    infer_paired(TEST, "Official test")
)

tables["Table_5"] = pd.DataFrame(
    stat_rows,
    columns=[
        "Partition", "Metric", "Light − control",
        "95% CI lower", "95% CI upper",
        "Exploratory p"
    ]
)

captions["Table_5"] = (
    "Matched four-seed comparisons between light augmentation "
    "and control. Differences are computed as light minus "
    "control. Two-sided exploratory Student's t confidence "
    "intervals and p-values use four paired training seeds. "
    "Inference is limited by the small number of seeds."
)

# TABLE 6 — DISPLACEMENT SENSITIVITY
# This table requires the original long-format test data.

def locate_long(directory, prefix):
    for path in sorted(directory.glob(f"{prefix}*.csv")):
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        df.columns = df.columns.str.strip()

        if "IoU" in df.columns and "iou" not in df.columns:
            df = df.rename(columns={"IoU": "iou"})

        if {"seed","variant","dx","dy","iou"}.issubset(df.columns):
            return path, df

    raise FileNotFoundError(
        "Missing displacement-level CSV. "
        "Cannot construct Table 6 without actual measurements."
    )

LONG_PATH, LONG = locate_long(
    BASE / "official_test", "step74"
)

LONG["variant"] = LONG.variant.astype(str).str.lower()

expected = {
    (seed, variant, dx, dy)
    for seed in (7,17,37,42)
    for variant in ("control","light")
    for dx,dy in [
        (0,0),(-1,0),(1,0),(-2,0),(2,0),
        (-4,0),(4,0),(-8,0),(8,0),
        (0,-2),(0,2),(0,-4),(0,4)
    ]
}

observed = set(zip(
    LONG.seed.astype(int),
    LONG.variant,
    LONG.dx.astype(int),
    LONG.dy.astype(int)
))

assert observed == expected, (
    "Official test displacement conditions do not match."
)

shift_means = LONG.groupby(
    ["dx","dy","variant"]
).iou.mean().unstack("variant").reset_index()

tables["Table_6"] = pd.DataFrame({
    "dx (pixels)": shift_means.dx,
    "dy (pixels)": shift_means.dy,
    "Control IoU": shift_means.control,
    "Light IoU": shift_means.light,
    "Light − control": (
        shift_means.light - shift_means.control
    )
})

captions["Table_6"] = (
    "Official held-out test IoU by synthetic optical "
    "displacement, averaged over four training seeds. "
    "The same displacement is applied to both optical "
    "acquisitions; SAR inputs and reference masks remain fixed."
)

# ------------------------------------------------------------
# 4. DISPLAY AND SAVE CSV
# ------------------------------------------------------------

for key, df in tables.items():

    print("\n" + "=" * 75)
    print(key.replace("_", " "))
    print(captions[key])
    print("=" * 75)

    shown = df.copy()

    for col in shown.select_dtypes(include="number").columns:
        if col not in ("Seed", "dx (pixels)", "dy (pixels)",
                       "Paired patches"):
            shown[col] = shown[col].map(
                lambda value: f"{value:.5f}"
            )

    display(shown)

    df.to_csv(
        OUT / f"{key}.csv",
        index=False
    )

# ------------------------------------------------------------
# 5. WORD DOCUMENT — JOURNAL TABLE STYLE
# ------------------------------------------------------------

doc = Document()

section = doc.sections[0]
section.top_margin = Inches(.8)
section.bottom_margin = Inches(.8)
section.left_margin = Inches(.75)
section.right_margin = Inches(.75)

normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(9)

def set_cell_border(cell, **borders):
    tcPr = cell._tc.get_or_add_tcPr()
    for edge, params in borders.items():
        tag = "w:" + edge
        element = tcPr.first_child_found_in(tag)

        if element is None:
            element = OxmlElement(tag)
            tcPr.append(element)

        for attr, value in params.items():
            element.set(qn("w:" + attr), str(value))

for key, df in tables.items():

    heading = doc.add_paragraph()
    heading.add_run(
        f"{key.replace('_', ' ')}. {captions[key]}"
    ).bold = True

    table = doc.add_table(
        rows=1,
        cols=len(df.columns)
    )

    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = True

    for j, field in enumerate(df.columns):
        table.rows[0].cells[j].text = str(field)

    for record in df.itertuples(index=False, name=None):
        cells = table.add_row().cells

        for j, value in enumerate(record):
            if isinstance(value, (float, np.floating)):
                cells[j].text = f"{value:.5f}"
            else:
                cells[j].text = str(value)

    for row_index, row in enumerate(table.rows):
        for cell in row.cells:
            # Remove all borders then add horizontal rules.
            set_cell_border(
                cell,
                top={"val":"nil"},
                bottom={"val":"nil"},
                left={"val":"nil"},
                right={"val":"nil"}
            )

            for paragraph in cell.paragraphs:
                paragraph.paragraph_format.space_after = Pt(0)

                for run in paragraph.runs:
                    run.font.name = "Times New Roman"
                    run.font.size = Pt(8)

                    if row_index == 0:
                        run.bold = True

            if row_index == 0:
                set_cell_border(
                    cell,
                    top={"val":"single","sz":"10"},
                    bottom={"val":"single","sz":"6"}
                )

            if row_index == len(table.rows)-1:
                set_cell_border(
                    cell,
                    bottom={"val":"single","sz":"10"}
                )

    doc.add_paragraph()

word_path = OUT / "UA_FloodNet_Final_Tables.docx"
doc.save(word_path)

# ------------------------------------------------------------
# 6. EXCEL WORKBOOK
# ------------------------------------------------------------

excel_path = OUT / "UA_FloodNet_Final_Tables.xlsx"

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:
    for key, df in tables.items():
        df.to_excel(
            writer,
            sheet_name=key,
            index=False
        )

# ------------------------------------------------------------
# 7. CAPTIONS AND AUDIT METADATA
# ------------------------------------------------------------

(OUT / "table_captions.json").write_text(
    json.dumps(captions, indent=2),
    encoding="utf-8"
)

metadata = {
    "study": "UA-FloodNet",
    "step": 84,
    "tables": list(tables.keys()),
    "training_seeds": [7,17,37,42],
    "validation_patches": 125,
    "official_test_patches": 70,
    "validation_summary_source": str(VAL_PATH),
    "official_test_summary_source": str(TEST_PATH),
    "official_test_shift_source": str(LONG_PATH),
    "event_independence": "not verified"
}

(OUT / "table_source_manifest.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. FINAL VERIFICATION
# ------------------------------------------------------------

assert len(tables) == 6
assert word_path.exists()
assert excel_path.exists()

for key in tables:
    assert (OUT / f"{key}.csv").exists()

print("\n" + "=" * 75)
print("STEP 84 COMPLETED")
print("=" * 75)

print("Manuscript tables:", len(tables))
print("Word:", word_path)
print("Excel:", excel_path)
print("Individual CSV tables: 6")
print("Output directory:", OUT)
